In [37]:

# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 0 — Frozen Dependencies & Environment Validation
# ============================================================
#
# PURPOSE
# -------
# CELL นี้เป็นประตูด่านแรกของ Pipeline
#
# หน้าที่:
#   1) ติดตั้ง dependency หลักด้วย version ที่ผ่าน Discovery แล้ว
#   2) ตรวจว่า pip install สำเร็จจริง
#   3) ตรวจว่า package ที่ต้องใช้มีอยู่จริง
#   4) ตรวจว่า version ที่ติดตั้งจริงตรงกับ version ที่กำหนด
#   5) ตรวจ dependency conflict ก่อน/หลัง install
#   6) ตรวจว่า module ที่โหลดอยู่ใน RAM ไม่ขัดกับ version บน disk
#
# ถ้าด่านสำคัญใดไม่ผ่าน:
#   -> STOP
#   -> ห้ามไป CELL 1
#
# IMPORTANT
# ---------
# Version ด้านล่างมาจาก Discovery ที่รันจริงใน Colab
# ไม่ใช่ version ที่คาดเดา
# ============================================================


# ------------------------------------------------------------
# Colab/Jupyter warning guard
#
# Prevent jupyter_client's Python 3.12 utcnow deprecation
# from recursively flooding cell output and making execution
# appear to run forever. This does not suppress pip stderr or
# any pipeline audit failure.
# ------------------------------------------------------------

import warnings as _warnings

_warnings.filterwarnings(
    "ignore",
    message=(
        r"datetime\.datetime\.utcnow\(\) is deprecated.*"
    ),
    category=DeprecationWarning,

)


# ------------------------------------------------------------
# 1) Frozen dependency versions
# ------------------------------------------------------------
#
# เรา pin package ที่มีผลโดยตรงต่อ:
#
# - การอ่าน DBN
# - Databento schema
# - Market calendar
# - NYSE schedule
#
# Dependency อื่น เช่น pandas / numpy / pyarrow
# จะถูกบันทึก version ใน CELL 1
# แต่ยังไม่ hard-pin ใน Colab V1
# ------------------------------------------------------------

PINNED = {
    "databento": "0.83.0",
    "databento-dbn": "0.65.0",
    "pandas-market-calendars": "5.4.0",
    "exchange-calendars": "4.13.2",
}


# ------------------------------------------------------------
# 2) Packages ที่ต้องรายงาน version
# ------------------------------------------------------------
#
# PINNED packages
# +
# dependency สำคัญที่อาจมีผลกับการคำนวณ/ไฟล์/calendar
# ------------------------------------------------------------

TRACKED_PACKAGES = [
    "databento",
    "databento-dbn",
    "pandas-market-calendars",
    "exchange-calendars",
    "pandas",
    "numpy",
    "pyarrow",
    "zstandard",
    "toolz",
]


# ------------------------------------------------------------
# 3) Modules ที่ต้องระวังเรื่อง stale runtime
# ------------------------------------------------------------
#
# ตัวอย่าง:
#
# pip บอกว่า pandas บน disk = version ใหม่
# แต่ Python runtime ยังถือ pandas version เก่าอยู่ใน RAM
#
# ถ้าเกิดแบบนั้น manifest จะโกหกว่าเราใช้ version ใหม่
# ทั้งที่ calculation จริงใช้ของเก่า
# ------------------------------------------------------------

RUNTIME_SENSITIVE = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "databento": "databento",
    "pandas-market-calendars": "pandas_market_calendars",
    "exchange-calendars": "exchange_calendars",
}


# ------------------------------------------------------------
# 4) Imports
# ------------------------------------------------------------

import sys
import re
import subprocess
import importlib

from importlib.metadata import (
    version,
    PackageNotFoundError,
)


# ------------------------------------------------------------
# 5) Helper — package version on disk
# ------------------------------------------------------------

def pkg_version(name):
    """
    อ่าน version ที่ติดตั้งอยู่จริงจาก package metadata
    โดยไม่พึ่ง __version__ ของ module
    """

    try:
        return version(name)

    except PackageNotFoundError:
        return "NOT INSTALLED"


# ------------------------------------------------------------
# 6) Helper — normalize package name
# ------------------------------------------------------------
#
# Python package names อาจเขียนได้หลายรูป:
#
# pandas_market_calendars
# pandas-market-calendars
#
# normalize เพื่อใช้ตอนตรวจข้อความ conflict
# ------------------------------------------------------------

def normalize_name(name):

    return re.sub(
        r"[-_.]+",
        "-",
        str(name).lower(),
    )


# ------------------------------------------------------------
# 7) Helper — pip check
# ------------------------------------------------------------
#
# pip check ตรวจว่า package ที่ติดตั้งอยู่
# มี dependency requirement ขัดกันหรือไม่
#
# คืนค่าเป็น set ของข้อความ conflict
# ------------------------------------------------------------

def pip_check_conflicts():

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "check",
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode == 0:
        return set()

    text = (
        result.stdout
        + "\n"
        + result.stderr
    )

    return {
        line.strip()
        for line in text.splitlines()
        if line.strip()
    }


# ------------------------------------------------------------
# 8) ตรวจ conflict ก่อน CELL 0 เปลี่ยน environment
# ------------------------------------------------------------
#
# จุดประสงค์:
#
# ถ้า Colab มี conflict อยู่ก่อนแล้ว
# เราต้องแยกออกจาก conflict ที่ CELL 0 สร้าง
# ------------------------------------------------------------

_conflicts_before = pip_check_conflicts()


print("=== PRE-INSTALL ENVIRONMENT ===")

print(
    "Python:",
    sys.version.split()[0]
)

print(
    "Existing dependency conflicts:",
    len(_conflicts_before)
)


if _conflicts_before:

    print(
        "\nConflicts ที่มีอยู่ก่อน CELL 0:"
    )

    for _line in sorted(
        _conflicts_before
    ):

        print(
            " ",
            _line
        )


# ------------------------------------------------------------
# 9) Build exact install specifications
# ------------------------------------------------------------

_specs = [
    f"{name}=={wanted_version}"
    for name, wanted_version
    in PINNED.items()
]


print(
    "\n=== FROZEN PACKAGE REQUEST ==="
)

for _spec in _specs:

    print(
        " ",
        _spec
    )


# ------------------------------------------------------------
# 10) Install pinned packages
# ------------------------------------------------------------
#
# ไม่ใช้:
#
#   !pip
#
# เพราะเราต้องการ return code จริง
#
# ไม่ใช้:
#
#   -q
#
# เพราะ resolver warning/error ต้องมองเห็นได้
# ------------------------------------------------------------

_install = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-input",
        *_specs,
    ],
    capture_output=True,
    text=True,
)


# ------------------------------------------------------------
# 11) แสดง pip output
# ------------------------------------------------------------

if _install.stdout.strip():

    print(
        "\n=== PIP OUTPUT ==="
    )

    print(
        _install.stdout[-5000:]
    )


if _install.stderr.strip():

    print(
        "\n=== PIP WARNINGS / STDERR ==="
    )

    print(
        _install.stderr[-5000:]
    )


# ------------------------------------------------------------
# 12) Installation safety gate
# ------------------------------------------------------------

if _install.returncode != 0:

    raise RuntimeError(
        "\nCELL 0 FAILED: pip install ไม่สำเร็จ\n\n"
        "ห้ามไป CELL 1\n\n"
        "สาเหตุที่เป็นไปได้:\n"
        "- Colab เปลี่ยน Python version\n"
        "- pinned wheel ไม่มีสำหรับ Python รุ่นใหม่\n"
        "- dependency resolver เปลี่ยน\n"
        "- package ถูกถอนจาก repository\n\n"
        "ห้ามแก้เลข version เพื่อให้ผ่านทันที\n"
        "ต้องทำ Dependency Discovery/Audit ใหม่ก่อน"
    )


print(
    "\nPIP INSTALL RETURN CODE: PASS"
)


# ------------------------------------------------------------
# 13) Refresh import caches
# ------------------------------------------------------------

importlib.invalidate_caches()


# ------------------------------------------------------------
# 14) ตรวจ package presence + version จริง
# ------------------------------------------------------------

print(
    "\n=== INSTALLED VERSIONS ==="
)

print(
    f"{'python':30s}"
    f"{sys.version.split()[0]}"
)


for _name in TRACKED_PACKAGES:

    print(
        f"{_name:30s}"
        f"{pkg_version(_name)}"
    )


# ------------------------------------------------------------
# 15) Package presence gate
# ------------------------------------------------------------

for _name in PINNED:

    _installed = pkg_version(
        _name
    )

    if _installed == "NOT INSTALLED":

        raise RuntimeError(
            f"\nCELL 0 FAILED\n\n"
            f"Package ไม่ได้ถูกติดตั้ง: {_name}\n\n"
            "pip อาจรายงานการทำงานไม่ตรงกับ "
            "environment ปัจจุบัน\n\n"
            "ห้ามไป CELL 1"
        )


print(
    "\nPACKAGE PRESENCE CHECK: PASS"

)


# ------------------------------------------------------------
# 16) Exact version validation
# ------------------------------------------------------------
#
# สิ่งที่เราขอ:
#
#   databento==0.83.0
#
# ต้องเท่ากับสิ่งที่อยู่บน disk จริง
#
# ไม่เชื่อเพียงข้อความจาก pip
# ------------------------------------------------------------

for _name, _wanted in (
    PINNED.items()
):

    _installed = pkg_version(
        _name
    )

    if _installed != _wanted:

        raise RuntimeError(
            f"\nVERSION PIN NOT APPLIED: {_name}\n\n"
            f"ต้องการ : {_wanted}\n"
            f"ได้จริง : {_installed}\n\n"
            "ห้ามไป CELL 1\n\n"
            "ลอง Runtime > Restart session "
            "แล้วรัน CELL 0 ใหม่"
        )


print(
    "VERSION PIN VALIDATION: PASS"
)


# ------------------------------------------------------------
# 17) Dependency conflict หลัง install
# ------------------------------------------------------------

_conflicts_after = (
    pip_check_conflicts()
)

_new_conflicts = sorted(
    _conflicts_after
    -
    _conflicts_before
)


# ------------------------------------------------------------
# 18) แยก conflict:
#
# A) กระทบ package ที่เรา pin โดยตรง
#       -> STOP
#
# B) package อื่นใน Colab ecosystem
#       -> WARNING + บันทึก
#
# ตัวอย่างที่เราเคยพบ:
#
# ibis-framework ต้องการ toolz<1
# แต่ exchange-calendars ใช้ toolz>=1
#
# เราไม่ได้ใช้ ibis-framework ใน MES Pipeline
# จึงบันทึกเป็น environment warning
# ไม่ตีความว่า environment "ไม่มี conflict"
# ------------------------------------------------------------

_pinned_norm = {
    normalize_name(name)
    for name in PINNED
}


_critical_new_conflicts = []

_noncritical_new_conflicts = []


for _line in _new_conflicts:

    _line_norm = normalize_name(
        _line
    )

    _is_critical = any(
        package_name
        in _line_norm

        for package_name
        in _pinned_norm
    )

    if _is_critical:

        _critical_new_conflicts.append(
            _line
        )

    else:

        _noncritical_new_conflicts.append(
            _line
        )


# ------------------------------------------------------------
# 19) Non-critical conflicts
# ------------------------------------------------------------

if _noncritical_new_conflicts:

    print(
        "\n=== ENVIRONMENT WARNING ==="
    )

    print(
        "CELL 0 สร้าง conflict ใหม่ใน package "
        "ที่ Pipeline ไม่ได้ใช้โดยตรง:"
    )

    for _line in (
        _noncritical_new_conflicts
    ):

        print(
            " ",
            _line
        )

    print(
        "\nPipeline ยังเดินต่อได้ "
        "แต่ conflict นี้ต้องถูกเก็บใน Environment Audit"
    )


# ------------------------------------------------------------
# 20) Critical conflicts
# ------------------------------------------------------------

if _critical_new_conflicts:

    raise RuntimeError(
        "\nCELL 0 FAILED\n\n"
        "พบ dependency conflict ใหม่ "
        "ที่พัวพันกับ package หลักของ Pipeline:\n\n"
        + "\n".join(
            _critical_new_conflicts
        )
        + "\n\nห้ามไป CELL 1"
    )


print(
    "\nCRITICAL DEPENDENCY CONFLICT CHECK: PASS"
)


# ------------------------------------------------------------
# 21) รายงาน Global pip-check state
# ------------------------------------------------------------
#
# สำคัญ:
#
# PASS ด้านบนไม่ได้แปลว่า Colab ไม่มี conflict
#
# มันแปลว่า:
# "ไม่มี conflict ใหม่ที่กระทบ pinned pipeline packages"
# ------------------------------------------------------------

print(
    "\n=== GLOBAL PIP CHECK STATE ==="
)

print(
    "Conflicts before install:",
    len(_conflicts_before)
)

print(
    "Conflicts after install :",
    len(_conflicts_after)
)

print(
    "New conflicts           :",
    len(_new_conflicts)
)


if _conflicts_after:

    print(
        "\nCurrent global conflicts:"
    )

    for _line in sorted(
        _conflicts_after
    ):

        print(
            " ",
            _line
        )


# ------------------------------------------------------------
# 22) Runtime vs disk version check
# ------------------------------------------------------------
#
# importlib.metadata อ่าน version บน disk
#
# แต่ module ที่ถูก import ไปแล้วอาจยังค้าง version เก่า
# อยู่ใน RAM
#
# ถ้ามี mismatch:
#
# STOP
#
# เพราะ CELL 1 จะบันทึก Environment ผิด
# ------------------------------------------------------------

_stale_modules = []


for (
    _dist_name,
    _module_name,
) in RUNTIME_SENSITIVE.items():

    _module = sys.modules.get(
        _module_name
    )

    # module ยังไม่เคย import
    # จึงไม่มี stale runtime problem
    if _module is None:
        continue

    _loaded_version = getattr(
        _module,
        "__version__",
        None,
    )

    _disk_version = pkg_version(
        _dist_name
    )

    if (
        _loaded_version
        and
        _disk_version
        != "NOT INSTALLED"
        and
        str(_loaded_version)
        != str(_disk_version)
    ):

        _stale_modules.append(
            f"{_dist_name}: "
            f"loaded={_loaded_version}, "
            f"disk={_disk_version}"
        )


if _stale_modules:

    raise RuntimeError(
        "\nCELL 0 FAILED: STALE RUNTIME\n\n"
        "Python กำลังใช้ module คนละ version "
        "กับที่ติดตั้งอยู่บน disk:\n\n"
        + "\n".join(
            _stale_modules
        )
        + "\n\nให้เลือก:\n"
        "Runtime > Restart session\n"
        "แล้วรัน CELL 0 ใหม่\n\n"
        "ห้ามไป CELL 1 ก่อน"
    )


print(
    "RUNTIME / DISK VERSION MATCH: PASS"
)


# ------------------------------------------------------------
# 23) Final environment summary
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 55
)

print(
    "CELL 0 FINAL VALIDATION"
)

print(
    "=" * 55
)


print(
    "\nFrozen packages:"
)

for _name, _wanted in (
    PINNED.items()
):

    print(
        f"  {_name:30s}"
        f"{_wanted}"
    )


print(
    "\nFinal gates:"
)

print(
    "  pip install                 : PASS"
)

print(
    "  package presence            : PASS"
)

print(
    "  exact pinned versions       : PASS"
)

print(
    "  critical dependency conflict: PASS"
)

print(
    "  runtime/disk version match  : PASS"
)


print(
    "\nCELL 0 FINAL: PASS"
)

=== PRE-INSTALL ENVIRONMENT ===
Python: 3.12.13
Existing dependency conflicts: 2

Conflicts ที่มีอยู่ก่อน CELL 0:
  ibis-framework 9.5.0 has requirement toolz<1,>=0.11, but you have toolz 1.1.0.
  ipython 7.34.0 requires jedi, which is not installed.

=== FROZEN PACKAGE REQUEST ===
  databento==0.83.0
  databento-dbn==0.65.0
  pandas-market-calendars==5.4.0
  exchange-calendars==4.13.2

=== PIP OUTPUT ===


PIP INSTALL RETURN CODE: PASS

=== INSTALLED VERSIONS ===
python                        3.12.13
databento                     0.83.0
databento-dbn                 0.65.0
pandas-market-calendars       5.4.0
exchange-calendars            4.13.2
pandas                        2.2.2
numpy                         2.0.2
pyarrow                       18.1.0
zstandard                     0.25.0
toolz                         1.1.0

PACKAGE PRESENCE CHECK: PASS
VERSION PIN VALIDATION: PASS

CRITICAL DEPENDENCY CONFLICT CHECK: PASS

=== GLOBAL PIP CHECK STATE ===
Conflicts before install: 2
Con

In [38]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 1 — Raw Source Identity & Provenance
# ============================================================
#
# PURPOSE
# -------
# CELL นี้ทำหน้าที่สร้าง "บัตรประจำตัว" ของข้อมูลต้นทาง
# ก่อนที่เราจะ decode ข้อมูลตลาดจริงใน CELL 2
#
# CELL นี้จะ:
#   1) ตรวจว่า dependency จาก CELL 0 พร้อมใช้งานจริง
#   2) Mount Google Drive
#   3) ระบุ DBN เป็น Raw Source of Truth
#   4) คำนวณ SHA-256 + file size + mtime ของ DBN
#   5) Fingerprint parquet เก่า ถ้ามี
#   6) อ่าน DBN metadata header
#   7) สรุป continuous-contract mappings แบบกระชับ
#   8) บันทึก Python / package environment
#   9) บันทึก pip freeze + pip check
#  10) ตรวจ frozen baseline ถ้ามีอยู่แล้ว
#  11) เขียน runtime_source_audit.json
#
# IMPORTANT
# ---------
# - ไม่มีการดึง Market Data ใหม่จาก Databento
# - ไม่มี Databento API call
# - DBN ใน Google Drive คือ Source of Truth
# - Parquet เก่าใช้เป็น cross-check เท่านั้น
# - CELL 2 จะเป็นคน decode DBN เต็มไฟล์และ freeze baseline
# ============================================================


# ------------------------------------------------------------
# 1) Standard-library imports
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

import sys
import subprocess
import hashlib
import json


# ------------------------------------------------------------
# 2) Dependency gate
#
# CELL 0 ต้องทำให้ databento import ได้จริง
# ถ้าไม่ได้ ให้หยุดด้วยข้อความที่อ่านง่าย
# ------------------------------------------------------------

try:
    import databento as db

except ModuleNotFoundError as e:
    raise RuntimeError(
        "CELL 1 STOPPED — ไม่พบ package 'databento'\n\n"
        "ให้ทำตามลำดับนี้:\n"
        "1) Run CELL 0\n"
        "2) รอจนเห็น 'CELL 0 FINAL: PASS'\n"
        "3) ห้าม Restart session\n"
        "4) แล้ว Run CELL 1 ใหม่"
    ) from e


try:
    from google.colab import drive

except ModuleNotFoundError as e:
    raise RuntimeError(
        "CELL 1 ต้องรันใน Google Colab "
        "หรือ environment ที่เข้าถึง Google Drive ได้"
    ) from e


# ------------------------------------------------------------
# 3) Mount Google Drive
# ------------------------------------------------------------

drive.mount(
    "/content/drive"
)


# ------------------------------------------------------------
# 4) Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)

DATA_DIR = (
    PROJECT_DIR /
    "Data"
)


# ------------------------------------------------------------
# RAW SOURCE OF TRUTH
# ------------------------------------------------------------

MES_DBN_PATH = (
    DATA_DIR /
    "MES_2019_2026_1m.dbn.zst"
)


# ------------------------------------------------------------
# Derived artifact จาก Notebook เก่า
#
# ไม่ใช่ Source of Truth
# CELL 2 จะใช้เทียบกับ DBN
# ------------------------------------------------------------

OLD_PARQUET_PATH = (
    DATA_DIR /
    "MES_2019_2026_1m.parquet"
)


# ------------------------------------------------------------
# Clean Pipeline output directory
# ------------------------------------------------------------

CLEAN_OUTPUT_DIR = (
    DATA_DIR /
    "MES_Clean_Pipeline_V1"
)

CLEAN_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 5) Audit artifact paths
# ------------------------------------------------------------

RUNTIME_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "runtime_source_audit.json"
)

PIP_FREEZE_PATH = (
    CLEAN_OUTPUT_DIR /
    "pip_freeze_snapshot.txt"
)

PIP_CHECK_PATH = (
    CLEAN_OUTPUT_DIR /
    "pip_check_snapshot.txt"
)

BASELINE_PATH = (
    CLEAN_OUTPUT_DIR /
    "raw_source_baseline.json"
)


# ------------------------------------------------------------
# 6) Historical request reference
#
# ค่านี้มาจาก Research Log เดิม
# ใช้เป็น reference เท่านั้น ไม่ใช่ proof
# ------------------------------------------------------------

LEGACY_REQUEST_REFERENCE = {
    "dataset": "GLBX.MDP3",
    "symbol": "MES.v.0",
    "stype_in": "continuous",
    "schema": "ohlcv-1m",
    "start": "2019-04-15",
    "end_exclusive": "2026-08-01",
    "continuous_rule": "volume",
    "continuous_rank": 0,
    "back_adjusted": False,
}


# ------------------------------------------------------------
# 7) Source existence gate
#
# DBN ไม่มี = ทำต่อไม่ได้
# Parquet ไม่มี = ยังทำต่อได้ แต่ CELL 2 จะ skip comparison
# ------------------------------------------------------------

if not MES_DBN_PATH.exists():
    raise FileNotFoundError(
        "RAW DBN SOURCE NOT FOUND\n\n"
        f"{MES_DBN_PATH}\n\n"
        "Pipeline stopped."
    )


HAS_OLD_PARQUET = (
    OLD_PARQUET_PATH.exists()
)


# ------------------------------------------------------------
# 8) Helper — installed package version
# ------------------------------------------------------------

def pkg_version(name):
    try:
        return version(name)

    except PackageNotFoundError:
        return "NOT INSTALLED"


# ------------------------------------------------------------
# 9) Helper — SHA-256
#
# อ่านเป็น chunk เพื่อไม่เอาไฟล์ทั้งหมดใส่ RAM
# ------------------------------------------------------------

def sha256_file(
    path,
    chunk_size=8 * 1024 * 1024,
):
    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:
            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ------------------------------------------------------------
# 10) Helper — file identity
#
# mtime_utc = filesystem modification time
# ไม่เรียกว่า download time
# ------------------------------------------------------------

def file_identity(path):
    stat = path.stat()

    return {
        "path":
            str(path),

        "size_bytes":
            int(stat.st_size),

        "mtime_utc":
            datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc,
            ).isoformat(),

        "sha256":
            sha256_file(path),
    }


# ------------------------------------------------------------
# 11) Fingerprint DBN
# ------------------------------------------------------------

print(
    "Computing DBN SHA-256..."
)

raw_identity = (
    file_identity(
        MES_DBN_PATH
    )
)


# ------------------------------------------------------------
# 12) Fingerprint old parquet — OPTIONAL
# ------------------------------------------------------------

if HAS_OLD_PARQUET:

    print(
        "Computing old Parquet SHA-256..."
    )

    parquet_identity = (
        file_identity(
            OLD_PARQUET_PATH
        )
    )

else:

    parquet_identity = None


# ------------------------------------------------------------
# 13) Parse DBN header / metadata
#
# ถ้า header อ่านไม่ได้ CELL นี้ต้องหยุด
# ------------------------------------------------------------

try:

    dbn_store = (
        db.DBNStore.from_file(
            MES_DBN_PATH
        )
    )

    dbn_metadata = (
        dbn_store.metadata
    )

except Exception as e:

    raise RuntimeError(
        "DBN HEADER / METADATA PARSE FAILED\n\n"
        f"{type(e).__name__}: {e}\n\n"
        "ยังไม่อนุญาตให้ไป CELL 2"
    ) from e


# ------------------------------------------------------------
# 14) Helper — JSON-safe conversion
# ------------------------------------------------------------

def json_safe(value):

    if value is None:
        return None

    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value

    if isinstance(
        value,
        dict,
    ):
        return {
            str(k):
                json_safe(v)

            for k, v
            in value.items()
        }

    if isinstance(
        value,
        (
            list,
            tuple,
            set,
        ),
    ):
        return [
            json_safe(v)
            for v in value
        ]

    # Enum / datetime / DBN-specific objects
    return str(value)


# ------------------------------------------------------------
# 15) Helper — summarize bulky metadata
#
# จุดสำคัญ:
# Databento mappings ของไฟล์นี้เป็น dict
#
# ตัวอย่างแนวคิด:
#
# {
#     "MES.v.0": [
#         contract interval 1,
#         contract interval 2,
#         ...
#     ]
# }
#
# ดังนั้น:
#
# symbol_key_count
#   = จำนวน continuous symbol keys
#
# total_mapping_intervals
#   = จำนวนช่วง contract จริงทั้งหมด
# ------------------------------------------------------------

def summarize_bulky_metadata(value):

    # ========================================================
    # CASE A — dict
    # ========================================================

    if isinstance(
        value,
        dict,
    ):

        symbol_key_count = len(value)

        if symbol_key_count == 0:

            return {
                "type":
                    "dict",

                "symbol_key_count":
                    0,

                "total_mapping_intervals":
                    0,

                "first_symbol_key":
                    None,

                "last_symbol_key":
                    None,

                "first_interval":
                    None,

                "last_interval":
                    None,
            }


        keys = list(
            value.keys()
        )

        first_key = (
            keys[0]
        )

        last_key = (
            keys[-1]
        )


        # ----------------------------------------------------
        # Count mapping intervals
        # ----------------------------------------------------

        total_mapping_intervals = 0

        for mapping_value in (
            value.values()
        ):

            if isinstance(
                mapping_value,
                (
                    list,
                    tuple,
                ),
            ):

                total_mapping_intervals += (
                    len(mapping_value)
                )

            else:

                total_mapping_intervals += 1


        # ----------------------------------------------------
        # First interval
        # ----------------------------------------------------

        first_value = (
            value[first_key]
        )

        if (
            isinstance(
                first_value,
                (
                    list,
                    tuple,
                ),
            )
            and
            len(first_value) > 0
        ):

            first_interval = (
                json_safe(
                    first_value[0]
                )
            )

        else:

            first_interval = (
                json_safe(
                    first_value
                )
            )


        # ----------------------------------------------------
        # Last interval
        # ----------------------------------------------------

        last_value = (
            value[last_key]
        )

        if (
            isinstance(
                last_value,
                (
                    list,
                    tuple,
                ),
            )
            and
            len(last_value) > 0
        ):

            last_interval = (
                json_safe(
                    last_value[-1]
                )
            )

        else:

            last_interval = (
                json_safe(
                    last_value
                )
            )


        return {
            "type":
                "dict",

            "symbol_key_count":
                symbol_key_count,

            "total_mapping_intervals":
                total_mapping_intervals,

            "first_symbol_key":
                json_safe(
                    first_key
                ),

            "last_symbol_key":
                json_safe(
                    last_key
                ),

            "first_interval":
                first_interval,

            "last_interval":
                last_interval,
        }


    # ========================================================
    # CASE B — list / tuple
    # ========================================================

    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        count = len(value)

        return {
            "type":
                type(value).__name__,

            "count":
                count,

            "first":
                (
                    json_safe(
                        value[0]
                    )
                    if count
                    else None
                ),

            "last":
                (
                    json_safe(
                        value[-1]
                    )
                    if count
                    else None
                ),
        }


    # ========================================================
    # CASE C — unknown structure
    # ========================================================

    try:

        return {
            "type":
                type(value).__name__,

            "count":
                len(value),

            "representation":
                str(value)[:1000],
        }

    except Exception as e:

        return {
            "type":
                str(
                    type(value)
                ),

            "summary_error":
                str(e),
        }


# ------------------------------------------------------------
# 16) Extract DBN metadata
#
# mappings ไม่ dump ทั้งก้อน
# ------------------------------------------------------------

BULKY_FIELDS = {
    "mappings",
}

metadata_public = {}

metadata_bulky_summary = {}


for name in dir(
    dbn_metadata
):

    if name.startswith("_"):
        continue


    try:

        value = getattr(
            dbn_metadata,
            name,
        )

    except Exception:
        continue


    if callable(value):
        continue


    if name in BULKY_FIELDS:

        metadata_bulky_summary[
            name
        ] = summarize_bulky_metadata(
            value
        )

        continue


    metadata_public[
        name
    ] = json_safe(
        value
    )


# ------------------------------------------------------------
# 17) Environment snapshot
# ------------------------------------------------------------

ENV_TRACKED_PACKAGES = [
    "databento",
    "databento-dbn",
    "pandas-market-calendars",
    "exchange-calendars",
    "pandas",
    "numpy",
    "pyarrow",
    "zstandard",
    "toolz",
]


ENVIRONMENT = {
    "python":
        sys.version,

    "python_short":
        sys.version.split()[0],

    "packages": {
        name:
            pkg_version(name)

        for name
        in ENV_TRACKED_PACKAGES
    },
}


# ------------------------------------------------------------
# 18) pip freeze snapshot
#
# เป็น forensic record
# ไม่ใช่ requirements.txt
# ------------------------------------------------------------

try:

    freeze_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "freeze",
        ],
        capture_output=True,
        text=True,
        timeout=180,
        check=True,
    )


    freeze_text = (
        freeze_result.stdout
    )


    PIP_FREEZE_PATH.write_text(
        freeze_text,
        encoding="utf-8",
    )


    ENVIRONMENT[
        "pip_freeze_file"
    ] = str(
        PIP_FREEZE_PATH
    )


    ENVIRONMENT[
        "pip_freeze_lines"
    ] = len(
        freeze_text.splitlines()
    )


    ENVIRONMENT[
        "pip_freeze_sha256"
    ] = hashlib.sha256(
        freeze_text.encode(
            "utf-8"
        )
    ).hexdigest()


except Exception as e:

    ENVIRONMENT[
        "pip_freeze_file"
    ] = None

    ENVIRONMENT[
        "pip_freeze_lines"
    ] = None

    ENVIRONMENT[
        "pip_freeze_sha256"
    ] = None

    ENVIRONMENT[
        "pip_freeze_error"
    ] = (
        f"{type(e).__name__}: {e}"
    )


# ------------------------------------------------------------
# 19) pip check snapshot
#
# Global conflicts อาจมีจาก Colab อยู่ก่อนแล้ว
# CELL 0 เป็นคนตัดสินว่า critical หรือไม่
# CELL 1 แค่บันทึกหลักฐาน
# ------------------------------------------------------------

pip_check_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "check",
    ],
    capture_output=True,
    text=True,
)


pip_check_text = (
    (
        pip_check_result.stdout
        +
        "\n"
        +
        pip_check_result.stderr
    )
    .strip()
)


if not pip_check_text:

    pip_check_text = (
        "No broken requirements found."
    )


PIP_CHECK_PATH.write_text(
    pip_check_text + "\n",
    encoding="utf-8",
)


ENVIRONMENT[
    "pip_check_returncode"
] = (
    pip_check_result.returncode
)


ENVIRONMENT[
    "pip_check_file"
] = str(
    PIP_CHECK_PATH
)


ENVIRONMENT[
    "pip_check_sha256"
] = hashlib.sha256(
    pip_check_text.encode(
        "utf-8"
    )
).hexdigest()


# ------------------------------------------------------------
# 20) Baseline check
#
# ครั้งแรก baseline ยังไม่มี
# CELL 2 จะสร้างเมื่อ:
#
# DBN full decode
# + integrity audit
# + DBN↔Parquet comparison
#
# ผ่านแล้วเท่านั้น
# ------------------------------------------------------------

if BASELINE_PATH.exists():

    try:

        with open(
            BASELINE_PATH,
            "r",
            encoding="utf-8",
        ) as f:

            baseline = (
                json.load(f)
            )


        baseline_raw = (
            baseline[
                "raw_file"
            ]
        )


        baseline_hash = (
            baseline_raw[
                "sha256"
            ]
        )


        baseline_size = (
            baseline_raw[
                "size_bytes"
            ]
        )


    except Exception as e:

        raise RuntimeError(
            "BASELINE FILE EXISTS BUT CANNOT BE READ\n\n"
            f"{type(e).__name__}: {e}"
        ) from e


    if (
        baseline_hash
        !=
        raw_identity[
            "sha256"
        ]
    ):

        raise RuntimeError(
            "RAW DBN HASH CHANGED\n\n"
            f"Baseline : {baseline_hash}\n"
            f"Current  : {raw_identity['sha256']}\n\n"
            "Pipeline stopped."
        )


    if (
        int(baseline_size)
        !=
        int(
            raw_identity[
                "size_bytes"
            ]
        )
    ):

        raise RuntimeError(
            "RAW DBN FILE SIZE CHANGED\n\n"
            f"Baseline : {baseline_size}\n"
            f"Current  : {raw_identity['size_bytes']}\n\n"
            "Pipeline stopped."
        )


    baseline_status = (
        "PASS — raw DBN matches frozen baseline"
    )


else:

    baseline_status = (
        "NOT FROZEN YET — CELL 2 AUDIT REQUIRED"
    )


# ------------------------------------------------------------
# 21) Runtime audit manifest
# ------------------------------------------------------------

runtime_audit = {

    "manifest_written_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "raw_file":
        raw_identity,

    "old_parquet_available":
        HAS_OLD_PARQUET,

    "old_parquet_reference":
        parquet_identity,

    "legacy_request_reference":
        LEGACY_REQUEST_REFERENCE,

    "dbn_metadata":
        metadata_public,

    "dbn_metadata_bulky_summary":
        metadata_bulky_summary,

    "environment":
        ENVIRONMENT,

    "baseline_status":
        baseline_status,
}


with open(
    RUNTIME_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        runtime_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 22) OUTPUT — Raw Source Identity
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 65
)

print(
    "RAW SOURCE IDENTITY"
)

print(
    "=" * 65
)


print(
    "DBN file       :",
    MES_DBN_PATH
)

print(
    "Size bytes     :",
    f"{raw_identity['size_bytes']:,}"
)

print(
    "SHA256         :",
    raw_identity[
        "sha256"
    ]
)

print(
    "File mtime UTC :",
    raw_identity[
        "mtime_utc"
    ]
)


# ------------------------------------------------------------
# 23) OUTPUT — Old Parquet
# ------------------------------------------------------------

print(
    "\n=== OLD PARQUET CROSS-CHECK ARTIFACT ==="
)


if HAS_OLD_PARQUET:

    print(
        "Status         : FOUND"
    )

    print(
        "Size bytes     :",
        f"{parquet_identity['size_bytes']:,}"
    )

    print(
        "SHA256         :",
        parquet_identity[
            "sha256"
        ]
    )


else:

    print(
        "Status         : NOT FOUND"
    )

    print(
        "CELL 2 DBN↔Parquet comparison "
        "จะถูกระบุเป็น SKIPPED"
    )


# ------------------------------------------------------------
# 24) OUTPUT — DBN Metadata
# ------------------------------------------------------------

print(
    "\n=== DBN METADATA ==="
)


print(
    "Metadata fields:"
)

print(
    sorted(
        metadata_public.keys()
    )
)


print(
    "\nBulky metadata summary:"
)

if metadata_bulky_summary:

    for (
        name,
        summary,
    ) in (
        metadata_bulky_summary.items()
    ):

        print(
            f"{name}:",
            summary
        )

else:

    print(
        "None"
    )


# ------------------------------------------------------------
# 25) OUTPUT — Environment
# ------------------------------------------------------------

print(
    "\n=== ENVIRONMENT ==="
)


print(
    "Python:",
    ENVIRONMENT[
        "python_short"
    ]
)


for (
    package_name,
    package_ver,
) in (
    ENVIRONMENT[
        "packages"
    ].items()
):

    print(
        f"  {package_name:30s}"
        f"{package_ver}"
    )


print(
    "\npip freeze SHA256:",
    ENVIRONMENT.get(
        "pip_freeze_sha256"
    )
)


print(
    "pip check return code:",
    ENVIRONMENT[
        "pip_check_returncode"
    ]
)


# ------------------------------------------------------------
# 26) OUTPUT — Baseline / Artifacts
# ------------------------------------------------------------

print(
    "\n=== BASELINE ==="
)

print(
    baseline_status
)


print(
    "\n=== AUDIT ARTIFACTS ==="
)

print(
    "Runtime manifest :",
    RUNTIME_AUDIT_PATH
)

print(
    "pip freeze       :",
    PIP_FREEZE_PATH
)

print(
    "pip check        :",
    PIP_CHECK_PATH
)


# ------------------------------------------------------------
# 27) Final gate
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 65
)

print(
    "CELL 1 LOCAL VALIDATION: PASS"
)

print(
    "=" * 65
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Computing DBN SHA-256...
Computing old Parquet SHA-256...

RAW SOURCE IDENTITY
DBN file       : /content/drive/MyDrive/Quant_Lab/Data/MES_2019_2026_1m.dbn.zst
Size bytes     : 40,078,487
SHA256         : 49f243a443abd199607bb51ce8d6c82928e2ba2a0ebb4a11ede10e7e0a0a46d0
File mtime UTC : 2026-08-08T16:54:21+00:00

=== OLD PARQUET CROSS-CHECK ARTIFACT ===
Status         : FOUND
Size bytes     : 43,567,185
SHA256         : 6bf5cb7331bce0ebc4edccee0a454afa9390983d004217067cc0346e02710e43

=== DBN METADATA ===
Metadata fields:
['dataset', 'end', 'limit', 'not_found', 'partial', 'schema', 'start', 'stype_in', 'stype_out', 'symbol_cstr_len', 'symbols', 'ts_out', 'version']

Bulky metadata summary:
mappings: {'type': 'dict', 'symbol_key_count': 1, 'total_mapping_intervals': 30, 'first_symbol_key': 'MES.v.0', 'last_symbol_key': 'MES.v.0', 'first_interval': {'start_date'

In [39]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 2 — Raw Decode, Integrity & DBN↔Parquet Cross-check
# ============================================================
#
# PURPOSE
# -------
# 1) Decode DBN ทั้งไฟล์
# 2) ตรวจโครงสร้างข้อมูล 1-minute
# 3) ตรวจ timestamp / duplicates / OHLCV / instrument_id
# 4) เทียบ DBN กับ parquet เก่าทีละ observation
# 5) บันทึก Raw Integrity Audit
# 6) Freeze raw_source_baseline.json อัตโนมัติเมื่อทุก Gate ผ่าน
#
# IMPORTANT
# ---------
# ไม่มีการดาวน์โหลด Market Data ใหม่
# ทุกอย่างอ่านจากไฟล์ใน Google Drive
# ============================================================
# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
import databento as db
# ------------------------------------------------------------
# 2) Required paths
#
# ตั้งเองอีกครั้งเพื่อไม่พึ่งตัวแปรลอยจาก Runtime
# ------------------------------------------------------------
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)
DATA_DIR = (
    PROJECT_DIR /
    "Data"
)
MES_DBN_PATH = (
    DATA_DIR /
    "MES_2019_2026_1m.dbn.zst"
)
OLD_PARQUET_PATH = (
    DATA_DIR /
    "MES_2019_2026_1m.parquet"
)
CLEAN_OUTPUT_DIR = (
    DATA_DIR /
    "MES_Clean_Pipeline_V1"
)
RUNTIME_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "runtime_source_audit.json"
)
CELL2_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell2_raw_integrity_audit.json"
)
BASELINE_PATH = (
    CLEAN_OUTPUT_DIR /
    "raw_source_baseline.json"
)
# ------------------------------------------------------------
# 3) Dependency / source gates
# ------------------------------------------------------------
if not MES_DBN_PATH.exists():
    raise FileNotFoundError(
        "CELL 2 STOPPED — RAW DBN ไม่พบ\n\n"
        f"{MES_DBN_PATH}"
    )
if not RUNTIME_AUDIT_PATH.exists():
    raise RuntimeError(
        "CELL 2 STOPPED — ไม่พบ runtime_source_audit.json\n\n"
        "ให้ Run CELL 1 ให้ผ่านก่อน"
    )
HAS_OLD_PARQUET = (
    OLD_PARQUET_PATH.exists()
)
# ------------------------------------------------------------
# 4) Load CELL 1 audit
# ------------------------------------------------------------
with open(
    RUNTIME_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:
    runtime_audit = (
        json.load(f)
    )
raw_identity = (
    runtime_audit[
        "raw_file"
    ]
)
# ------------------------------------------------------------
# 5) Full DBN decode
#
# ตรงนี้คือการอ่าน record จริงทั้งไฟล์
# ถ้า DBN / compression เสียจน decode ไม่ได้
# จะหยุดตรงนี้
# ------------------------------------------------------------
print(
    "Decoding full DBN file..."
)
try:
    dbn_store_cell2 = (
        db.DBNStore.from_file(
            MES_DBN_PATH
        )
    )
    mes_1m = (
        dbn_store_cell2.to_df()
    )
except Exception as e:
    raise RuntimeError(
        "FULL DBN DECODE FAILED\n\n"
        f"{type(e).__name__}: {e}\n\n"
        "Raw source ยังไม่ผ่าน integrity gate"
    ) from e
print(
    "DBN decode completed."
)
# ------------------------------------------------------------
# 6) General structure
# ------------------------------------------------------------
failures = []
decoded_rows = (
    len(mes_1m)
)
decoded_columns = (
    list(mes_1m.columns)
)
if decoded_rows == 0:
    failures.append(
        "Decoded DBN contains zero rows"
    )
if not isinstance(
    mes_1m.index,
    pd.DatetimeIndex,
):
    failures.append(
        "DBN index is not pandas.DatetimeIndex"
    )
# ------------------------------------------------------------
# 7) Timestamp audit
# ------------------------------------------------------------
if isinstance(
    mes_1m.index,
    pd.DatetimeIndex,
):
    index_name = (
        mes_1m.index.name
    )
    timezone_name = (
        str(mes_1m.index.tz)
        if mes_1m.index.tz is not None
        else None
    )
    first_ts = (
        mes_1m.index[0]
        if decoded_rows
        else None
    )
    last_ts = (
        mes_1m.index[-1]
        if decoded_rows
        else None
    )
    is_monotonic = (
        mes_1m.index.is_monotonic_increasing
    )
    duplicate_timestamps = int(
        mes_1m.index.duplicated().sum()
    )
    if mes_1m.index.tz is None:
        failures.append(
            "DBN timestamp index is timezone-naive"
        )
    elif (
        str(mes_1m.index.tz).upper()
        !=
        "UTC"
    ):
        failures.append(
            "DBN timestamp timezone is not UTC: "
            f"{mes_1m.index.tz}"
        )
    if not is_monotonic:
        failures.append(
            "DBN timestamps are not monotonic increasing"
        )
    if duplicate_timestamps != 0:
        failures.append(
            "DBN contains duplicate timestamps: "
            f"{duplicate_timestamps}"
        )
else:
    index_name = None
    timezone_name = None
    first_ts = None
    last_ts = None
    is_monotonic = False
    duplicate_timestamps = None
# ------------------------------------------------------------
# 8) Required market-data columns
# ------------------------------------------------------------
OHLCV_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
]
CRITICAL_COLUMNS = [
    "instrument_id",
    *OHLCV_COLUMNS,
]
missing_critical_columns = [
    col
    for col in CRITICAL_COLUMNS
    if col not in mes_1m.columns
]
if missing_critical_columns:
    failures.append(
        "Missing critical DBN columns: "
        + ", ".join(
            missing_critical_columns
        )
    )
# ------------------------------------------------------------
# 9) OHLCV integrity
# ------------------------------------------------------------
ohlcv_nan_counts = {}
if not missing_critical_columns:
    ohlcv_nan_counts = {
        col:
            int(
                mes_1m[
                    col
                ].isna().sum()
            )
        for col in OHLCV_COLUMNS
    }
    total_ohlcv_nan = (
        sum(
            ohlcv_nan_counts.values()
        )
    )
    if total_ohlcv_nan != 0:
        failures.append(
            "OHLCV contains NaN values: "
            f"{ohlcv_nan_counts}"
        )
    # --------------------------------------------------------
    # Price structure
    #
    # ทุก bar:
    # high ต้องไม่ต่ำกว่า open/close/low
    # low ต้องไม่สูงกว่า open/close/high
    # --------------------------------------------------------
    high_violation = int(
        (
            (
                mes_1m["high"]
                <
                mes_1m[
                    ["open", "close", "low"]
                ].max(axis=1)
            )
        ).sum()
    )
    low_violation = int(
        (
            (
                mes_1m["low"]
                >
                mes_1m[
                    ["open", "close", "high"]
                ].min(axis=1)
            )
        ).sum()
    )
    negative_volume = int(
        (
            mes_1m["volume"]
            <
            0
        ).sum()
    )
    if high_violation:
        failures.append(
            f"OHLC high violations: {high_violation}"
        )
    if low_violation:
        failures.append(
            f"OHLC low violations: {low_violation}"
        )
    if negative_volume:
        failures.append(
            f"Negative volume rows: {negative_volume}"
        )
else:
    high_violation = None
    low_violation = None
    negative_volume = None
# ------------------------------------------------------------
# 10) Instrument / roll audit
# ------------------------------------------------------------
if (
    "instrument_id"
    in mes_1m.columns
):
    instrument_nan = int(
        mes_1m[
            "instrument_id"
        ].isna().sum()
    )
    unique_instruments = int(
        mes_1m[
            "instrument_id"
        ].nunique(
            dropna=True
        )
    )
    instrument_array = (
        mes_1m[
            "instrument_id"
        ].to_numpy()
    )
    if len(
        instrument_array
    ) > 1:
        roll_transition_count = int(
            np.count_nonzero(
                instrument_array[1:]
                !=
                instrument_array[:-1]
            )
        )
    else:
        roll_transition_count = 0
    if instrument_nan:
        failures.append(
            "instrument_id contains NaN: "
            f"{instrument_nan}"
        )
else:
    instrument_nan = None
    unique_instruments = None
    roll_transition_count = None
# ------------------------------------------------------------
# 11) Dtype record
#
# Dtype ถูกบันทึกไว้
# แต่ dtype ต่างกันกับ parquet
# ไม่ได้ถือว่า values ต่างกันโดยอัตโนมัติ
# ------------------------------------------------------------
dbn_dtypes = {
    col:
        str(dtype)
    for col, dtype
    in mes_1m.dtypes.items()
}
# ------------------------------------------------------------
# 12) DBN ↔ OLD PARQUET CROSS-CHECK
# ------------------------------------------------------------
cross_check = {
    "status":
        "SKIPPED",
    "reason":
        None,
    "compared_columns":
        [],
    "column_results":
        {},
    "index_equal":
        None,
    "rows_equal":
        None,
}
if HAS_OLD_PARQUET:
    print(
        "Loading old parquet for cross-check..."
    )
    try:
        old_1m = pd.read_parquet(
            OLD_PARQUET_PATH
        )
    except Exception as e:
        failures.append(
            "Old parquet could not be read: "
            f"{type(e).__name__}: {e}"
        )
        old_1m = None
    if old_1m is not None:
        cross_check[
            "status"
        ] = "RUNNING"
        # ----------------------------------------------------
        # Row-count comparison
        # ----------------------------------------------------
        cross_check[
            "dbn_rows"
        ] = int(
            len(mes_1m)
        )
        cross_check[
            "parquet_rows"
        ] = int(
            len(old_1m)
        )
        cross_check[
            "rows_equal"
        ] = (
            len(mes_1m)
            ==
            len(old_1m)
        )
        if not cross_check[
            "rows_equal"
        ]:
            failures.append(
                "DBN ↔ Parquet row count mismatch: "
                f"{len(mes_1m):,} vs {len(old_1m):,}"
            )
        # ----------------------------------------------------
        # Timestamp comparison
        # ----------------------------------------------------
        if (
            isinstance(
                old_1m.index,
                pd.DatetimeIndex,
            )
            and
            isinstance(
                mes_1m.index,
                pd.DatetimeIndex,
            )
            and
            len(old_1m)
            ==
            len(mes_1m)
        ):
            index_equal = bool(
                np.array_equal(
                    mes_1m.index.asi8,
                    old_1m.index.asi8,
                )
            )
        else:
            index_equal = False
        cross_check[
            "index_equal"
        ] = index_equal
        cross_check[
            "dbn_index_timezone"
        ] = (
            str(
                mes_1m.index.tz
            )
            if isinstance(
                mes_1m.index,
                pd.DatetimeIndex,
            )
            else None
        )
        cross_check[
            "parquet_index_timezone"
        ] = (
            str(
                old_1m.index.tz
            )
            if isinstance(
                old_1m.index,
                pd.DatetimeIndex,
            )
            else None
        )
        if not index_equal:
            failures.append(
                "DBN ↔ Parquet timestamp index mismatch"
            )
        # ----------------------------------------------------
        # Critical column availability
        # ----------------------------------------------------
        missing_dbn_compare = [
            col
            for col in CRITICAL_COLUMNS
            if col not in mes_1m.columns
        ]
        missing_parquet_compare = [
            col
            for col in CRITICAL_COLUMNS
            if col not in old_1m.columns
        ]
        cross_check[
            "missing_dbn_columns"
        ] = (
            missing_dbn_compare
        )
        cross_check[
            "missing_parquet_columns"
        ] = (
            missing_parquet_compare
        )
        if missing_dbn_compare:
            failures.append(
                "DBN cross-check columns missing: "
                + ", ".join(
                    missing_dbn_compare
                )
            )
        if missing_parquet_compare:
            failures.append(
                "Old parquet cross-check columns missing: "
                + ", ".join(
                    missing_parquet_compare
                )
            )
        # ----------------------------------------------------
        # Observation-by-observation comparison
        #
        # สำคัญ:
        # ใช้ .to_numpy()
        #
        # dtype ต่างกันไม่ทำให้ FAIL
        # ถ้าค่าจริงยังเท่ากัน
        # ----------------------------------------------------
        if (
            not missing_dbn_compare
            and
            not missing_parquet_compare
            and
            cross_check[
                "rows_equal"
            ]
            and
            index_equal
        ):
            cross_check[
                "compared_columns"
            ] = CRITICAL_COLUMNS.copy()
            for col in CRITICAL_COLUMNS:
                dbn_values = (
                    mes_1m[
                        col
                    ].to_numpy()
                )
                parquet_values = (
                    old_1m[
                        col
                    ].to_numpy()
                )
                try:
                    arrays_equal = bool(
                        np.array_equal(
                            dbn_values,
                            parquet_values,
                            equal_nan=True,
                        )
                    )
                except TypeError:
                    arrays_equal = bool(
                        np.array_equal(
                            dbn_values,
                            parquet_values,
                        )
                    )
                # --------------------------------------------
                # Count / sample mismatches
                # --------------------------------------------
                if arrays_equal:
                    mismatch_count = 0
                    mismatch_examples = []
                else:
                    # numeric comparison
                    if (
                        np.issubdtype(
                            dbn_values.dtype,
                            np.number,
                        )
                        and
                        np.issubdtype(
                            parquet_values.dtype,
                            np.number,
                        )
                    ):
                        equal_mask = (
                            dbn_values
                            ==
                            parquet_values
                        )
                        # NaN == NaN สำหรับ audit
                        try:
                            equal_mask = (
                                equal_mask
                                |
                                (
                                    np.isnan(
                                        dbn_values
                                    )
                                    &
                                    np.isnan(
                                        parquet_values
                                    )
                                )
                            )
                        except TypeError:
                            pass
                    else:
                        equal_mask = (
                            dbn_values
                            ==
                            parquet_values
                        )
                    mismatch_positions = (
                        np.flatnonzero(
                            ~equal_mask
                        )
                    )
                    mismatch_count = int(
                        len(
                            mismatch_positions
                        )
                    )
                    mismatch_examples = []
                    for pos in (
                        mismatch_positions[:5]
                    ):
                        mismatch_examples.append(
                            {
                                "position":
                                    int(pos),
                                "timestamp":
                                    str(
                                        mes_1m.index[
                                            pos
                                        ]
                                    ),
                                "dbn":
                                    str(
                                        dbn_values[
                                            pos
                                        ]
                                    ),
                                "parquet":
                                    str(
                                        parquet_values[
                                            pos
                                        ]
                                    ),
                            }
                        )
                cross_check[
                    "column_results"
                ][col] = {
                    "dbn_dtype":
                        str(
                            mes_1m[
                                col
                            ].dtype
                        ),
                    "parquet_dtype":
                        str(
                            old_1m[
                                col
                            ].dtype
                        ),
                    "values_equal":
                        arrays_equal,
                    "mismatch_count":
                        mismatch_count,
                    "examples":
                        mismatch_examples,
                }
                if not arrays_equal:
                    failures.append(
                        f"DBN ↔ Parquet values mismatch "
                        f"in '{col}': "
                        f"{mismatch_count:,} rows"
                    )
        # ----------------------------------------------------
        # Optional shared columns
        #
        # บันทึกไว้เพื่อดูว่าไฟล์สองฝั่งมี schema ต่างกันไหม
        # แต่ไม่ใช้เป็น Critical Gate
        # ----------------------------------------------------
        cross_check[
            "dbn_columns"
        ] = list(
            mes_1m.columns
        )
        cross_check[
            "parquet_columns"
        ] = list(
            old_1m.columns
        )
        # ----------------------------------------------------
        # Final cross-check state
        # ----------------------------------------------------
        parquet_related_failures = [
            failure
            for failure
            in failures
            if (
                "Parquet"
                in failure
                or
                "parquet"
                in failure
            )
        ]
        if parquet_related_failures:
            cross_check[
                "status"
            ] = "FAIL"
        else:
            cross_check[
                "status"
            ] = "PASS"
else:
    cross_check[
        "status"
    ] = "SKIPPED"
    cross_check[
        "reason"
    ] = (
        "Old parquet artifact not found"
    )
# ------------------------------------------------------------
# 13) Metadata / mapping consistency note
#
# ไม่ใช้เป็น hard assert เพราะ metadata request period
# อาจครอบคลุมช่วงที่ไม่มี market records
# ------------------------------------------------------------
mapping_summary = (
    runtime_audit
    .get(
        "dbn_metadata_bulky_summary",
        {}
    )
    .get(
        "mappings",
        {}
    )
)
mapping_intervals = (
    mapping_summary.get(
        "total_mapping_intervals"
    )
)
# ------------------------------------------------------------
# 14) Build Cell 2 audit
# ------------------------------------------------------------
cell2_audit = {
    "audit_written_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "raw_file":
        raw_identity,
    "decoded": {
        "rows":
            int(
                decoded_rows
            ),
        "columns":
            decoded_columns,
        "dtypes":
            dbn_dtypes,
        "index_name":
            index_name,
        "timezone":
            timezone_name,
        "first_timestamp":
            (
                first_ts.isoformat()
                if first_ts is not None
                else None
            ),
        "last_timestamp":
            (
                last_ts.isoformat()
                if last_ts is not None
                else None
            ),
        "monotonic_increasing":
            bool(
                is_monotonic
            ),
        "duplicate_timestamps":
            duplicate_timestamps,
        "ohlcv_nan_counts":
            ohlcv_nan_counts,
        "high_violations":
            high_violation,
        "low_violations":
            low_violation,
        "negative_volume_rows":
            negative_volume,
        "instrument_id_nan":
            instrument_nan,
        "unique_instrument_ids":
            unique_instruments,
        "roll_transition_count":
            roll_transition_count,
        "metadata_mapping_intervals":
            mapping_intervals,
    },
    "dbn_vs_old_parquet":
        cross_check,
    "failures":
        failures,
}
# ------------------------------------------------------------
# 15) Save audit even if a gate fails
# ------------------------------------------------------------
with open(
    CELL2_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        cell2_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )
# ------------------------------------------------------------
# 16) Integrity gate
# ------------------------------------------------------------
if failures:
    print(
        "\n=== CELL 2 FAILURES ==="
    )
    for failure in failures:
        print(
            " -",
            failure
        )
    raise RuntimeError(
        "\nCELL 2 RAW INTEGRITY: FAIL\n\n"
        "Audit ถูกบันทึกไว้ที่:\n"
        f"{CELL2_AUDIT_PATH}\n\n"
        "ห้ามสร้าง baseline และห้ามไป Cell ถัดไป"
    )
# ------------------------------------------------------------
# 17) Freeze or verify baseline
#
# เขียน baseline หลังทุก Gate ผ่านแล้วเท่านั้น
# ------------------------------------------------------------
baseline_payload = {
    "baseline_version":
        1,
    "baseline_created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
    "raw_file":
        raw_identity,
    "decoded_integrity": {
        "rows":
            int(
                decoded_rows
            ),
        "first_timestamp":
            (
                first_ts.isoformat()
                if first_ts is not None
                else None
            ),
        "last_timestamp":
            (
                last_ts.isoformat()
                if last_ts is not None
                else None
            ),
        "timezone":
            timezone_name,
        "duplicate_timestamps":
            duplicate_timestamps,
        "ohlcv_nan_counts":
            ohlcv_nan_counts,
        "unique_instrument_ids":
            unique_instruments,
        "roll_transition_count":
            roll_transition_count,
    },
    "old_parquet_cross_check": {
        "status":
            cross_check[
                "status"
            ],
        "parquet_file":
            runtime_audit.get(
                "old_parquet_reference"
            ),
        "compared_columns":
            cross_check.get(
                "compared_columns",
                [],
            ),
    },
    "environment_at_freeze":
        runtime_audit.get(
            "environment"
        ),
}
if BASELINE_PATH.exists():
    with open(
        BASELINE_PATH,
        "r",
        encoding="utf-8",
    ) as f:
        existing_baseline = (
            json.load(f)
        )
    existing_raw = (
        existing_baseline[
            "raw_file"
        ]
    )
    if (
        existing_raw[
            "sha256"
        ]
        !=
        raw_identity[
            "sha256"
        ]
    ):
        raise RuntimeError(
            "EXISTING BASELINE HASH DOES NOT MATCH CURRENT DBN"
        )
    if (
        int(
            existing_raw[
                "size_bytes"
            ]
        )
        !=
        int(
            raw_identity[
                "size_bytes"
            ]
        )
    ):
        raise RuntimeError(
            "EXISTING BASELINE SIZE DOES NOT MATCH CURRENT DBN"
        )
    existing_decoded = (
        existing_baseline.get(
            "decoded_integrity",
            {}
        )
    )
    baseline_checks = {
        "rows":
            int(
                decoded_rows
            ),
        "first_timestamp":
            (
                first_ts.isoformat()
                if first_ts is not None
                else None
            ),
        "last_timestamp":
            (
                last_ts.isoformat()
                if last_ts is not None
                else None
            ),
        "duplicate_timestamps":
            duplicate_timestamps,
    }
    for key, current_value in (
        baseline_checks.items()
    ):
        old_value = (
            existing_decoded.get(
                key
            )
        )
        if (
            old_value is not None
            and
            old_value != current_value
        ):
            raise RuntimeError(
                "EXISTING BASELINE DECODED "
                f"INTEGRITY MISMATCH: {key}\n"
                f"baseline={old_value}\n"
                f"current ={current_value}"
            )
    baseline_action = (
        "VERIFIED — existing baseline unchanged"
    )
else:
    # Atomic write:
    # เขียน temp ก่อน แล้วค่อย replace
    # ลดความเสี่ยง baseline ครึ่งไฟล์
    BASELINE_TEMP_PATH = (
        BASELINE_PATH.with_suffix(
            ".tmp"
        )
    )
    with open(
        BASELINE_TEMP_PATH,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            baseline_payload,
            f,
            indent=2,
            ensure_ascii=False,
        )
    BASELINE_TEMP_PATH.replace(
        BASELINE_PATH
    )
    baseline_action = (
        "CREATED — new immutable raw baseline"
    )
# ------------------------------------------------------------
# 18) Free old parquet from memory
#
# mes_1m จาก DBN ยังเก็บไว้
# เพราะ Cell ถัดไปจะใช้ต่อ
# ------------------------------------------------------------
if (
    "old_1m"
    in globals()
    and
    old_1m is not None
):
    del old_1m
    gc.collect()
# ------------------------------------------------------------
# 19) OUTPUT — decoded DBN
# ------------------------------------------------------------
print(
    "\n"
    + "=" * 70
)
print(
    "DBN FULL DECODE & RAW INTEGRITY"
)
print(
    "=" * 70
)
print(
    "Rows                  :",
    f"{decoded_rows:,}"
)
print(
    "First timestamp       :",
    first_ts
)
print(
    "Last timestamp        :",
    last_ts
)
print(
    "Timezone              :",
    timezone_name
)
print(
    "Monotonic timestamps  :",
    is_monotonic
)
print(
    "Duplicate timestamps  :",
    duplicate_timestamps
)
print(
    "\nOHLCV NaN counts:"
)
for col, count in (
    ohlcv_nan_counts.items()
):
    print(
        f"  {col:10s}",
        count
    )
print(
    "\nOHLC structure:"
)
print(
    "  high violations     :",
    high_violation
)
print(
    "  low violations      :",
    low_violation
)
print(
    "  negative volume     :",
    negative_volume
)
print(
    "\nInstrument / Roll:"
)
print(
    "  unique instrument_id:",
    unique_instruments
)
print(
    "  roll transitions    :",
    roll_transition_count
)
print(
    "  metadata intervals  :",
    mapping_intervals
)
# ------------------------------------------------------------
# 20) OUTPUT — DBN vs Parquet
# ------------------------------------------------------------
print(
    "\n=== DBN ↔ OLD PARQUET CROSS-CHECK ==="
)
print(
    "Status:",
    cross_check[
        "status"
    ]
)
if cross_check[
    "status"
] != "SKIPPED":
    print(
        "Rows equal :",
        cross_check[
            "rows_equal"
        ]
    )
    print(
        "Index equal:",
        cross_check[
            "index_equal"
        ]
    )
    for col in (
        cross_check.get(
            "compared_columns",
            []
        )
    ):
        result = (
            cross_check[
                "column_results"
            ][col]
        )
        print(
            f"  {col:15s}"
            f"values_equal="
            f"{result['values_equal']} "
            f"mismatch="
            f"{result['mismatch_count']:,} "
            f"dtype="
            f"{result['dbn_dtype']}"
            f" ↔ "
            f"{result['parquet_dtype']}"
        )
# ------------------------------------------------------------
# 21) OUTPUT — baseline
# ------------------------------------------------------------
print(
    "\n=== RAW SOURCE BASELINE ==="
)
print(
    baseline_action
)
print(
    "Baseline file:",
    BASELINE_PATH
)
print(
    "Cell 2 audit :",
    CELL2_AUDIT_PATH
)
print(
    "\n"
    + "=" * 70
)
print(
    "CELL 2 RAW INTEGRITY: PASS"
)
print(
    "=" * 70
)


# ------------------------------------------------------------
# CELL 2 INFRASTRUCTURE ADDENDUM — decoded-memory fingerprint
#
# This does not alter market data. It fingerprints the exact decoded
# frame held in this runtime so downstream path calculations can prove
# that they consumed the same Cell 2 object, not merely a frame with the
# same row count and endpoints.
# ------------------------------------------------------------
import hashlib as _cell2_hashlib

CELL2_PROVENANCE_POLICY_VERSION = "MES_V1_RAW_INTEGRITY_1.1"
CELL2_MEMORY_HASH_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "instrument_id",
]

_cell2_memory_row_hashes = pd.util.hash_pandas_object(
    mes_1m[CELL2_MEMORY_HASH_COLUMNS],
    index=True,
    categorize=False,
).to_numpy(dtype="uint64", copy=False)

_cell2_base_status = cell2_audit.get("status")
if _cell2_base_status not in {None, "PASS"} or cell2_audit.get("failures", []):
    raise RuntimeError(
        "CELL 2 PROVENANCE ADDENDUM STOPPED — base Cell 2 audit is not clean PASS."
    )

CELL2_MES_1M_CONTENT_SHA256 = _cell2_hashlib.sha256(
    _cell2_memory_row_hashes.tobytes()
).hexdigest()
del _cell2_memory_row_hashes

cell2_audit["status"] = "PASS"
cell2_audit["policy_version"] = CELL2_PROVENANCE_POLICY_VERSION
cell2_audit["decoded"]["content_sha256"] = CELL2_MES_1M_CONTENT_SHA256
cell2_audit["decoded"]["content_hash_columns"] = CELL2_MEMORY_HASH_COLUMNS
cell2_audit["decoded"]["content_hash_includes_index"] = True
cell2_audit["decoded"]["content_hash_algorithm"] = (
    "SHA256_OVER_PANDAS_UINT64_ROW_HASHES"
)

with open(CELL2_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        cell2_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Cell 2 decoded-memory SHA256:", CELL2_MES_1M_CONTENT_SHA256)
print("CELL 2 PROVENANCE ADDENDUM: PASS")


Decoding full DBN file...
DBN decode completed.
Loading old parquet for cross-check...

DBN FULL DECODE & RAW INTEGRITY
Rows                  : 2,551,123
First timestamp       : 2019-05-05 22:00:00+00:00
Last timestamp        : 2026-07-31 20:59:00+00:00
Timezone              : UTC
Monotonic timestamps  : True
Duplicate timestamps  : 0

OHLCV NaN counts:
  open       0
  high       0
  low        0
  close      0
  volume     0

OHLC structure:
  high violations     : 0
  low violations      : 0
  negative volume     : 0

Instrument / Roll:
  unique instrument_id: 30
  roll transitions    : 29
  metadata intervals  : 30

=== DBN ↔ OLD PARQUET CROSS-CHECK ===
Status: PASS
Rows equal : True
Index equal: True
  instrument_id  values_equal=True mismatch=0 dtype=uint32 ↔ uint32
  open           values_equal=True mismatch=0 dtype=float64 ↔ float64
  high           values_equal=True mismatch=0 dtype=float64 ↔ float64
  low            values_equal=True mismatch=0 dtype=float64 ↔ float64
  close

In [40]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 3 — Supplemental Raw Audit
# Provenance Gate + Zero-Volume + Raw Time-Gap Distribution
# ============================================================
#
# PURPOSE
# -------
# Cell นี้ปิด Data Audit ที่ยังค้างอยู่ก่อน resample 1m -> 15m
#
# ทำ 3 เรื่อง:
#
#   A) PROVENANCE HARD GATE
#      ยืนยันจาก DBN metadata ว่าไฟล์คือ:
#        dataset   = GLBX.MDP3
#        schema    = ohlcv-1m
#        stype_in  = continuous
#        stype_out = instrument_id
#        symbol    = MES.v.0
#        start     = 2019-04-15
#        end       = 2026-08-01 (exclusive)
#
#   B) ZERO-VOLUME AUDIT
#      OHLCV-1m ของ Databento เป็น trade-based bars
#      จึงตรวจว่ามี bar volume == 0 หรือไม่
#
#   C) RAW TIME-GAP AUDIT
#      วิเคราะห์ระยะห่างระหว่าง raw 1-minute records
#      ก่อนทำ resample
#
# IMPORTANT
# ---------
# - ไม่มีการดาวน์โหลด Market Data ใหม่
# - ไม่ forward-fill
# - ไม่ลบ gap
# - ไม่แก้ raw baseline เดิม
# - gap > 1 นาที ไม่ถือว่า data error อัตโนมัติ
# - Cell นี้เป็น AUDIT เท่านั้น
# ============================================================


# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import json
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2) Paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)

DATA_DIR = (
    PROJECT_DIR /
    "Data"
)

CLEAN_OUTPUT_DIR = (
    DATA_DIR /
    "MES_Clean_Pipeline_V1"
)

RUNTIME_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "runtime_source_audit.json"
)

CELL2_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell2_raw_integrity_audit.json"
)

BASELINE_PATH = (
    CLEAN_OUTPUT_DIR /
    "raw_source_baseline.json"
)

CELL3_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell3_supplemental_raw_audit.json"
)

GAP_DISTRIBUTION_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell3_gap_distribution.csv"
)

GAP_EVENTS_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell3_gap_events.parquet"
)


# ------------------------------------------------------------
# 3) Upstream dependency gates
# ------------------------------------------------------------

required_files = [
    RUNTIME_AUDIT_PATH,
    CELL2_AUDIT_PATH,
    BASELINE_PATH,
]


for required_path in required_files:

    if not required_path.exists():

        raise RuntimeError(
            "CELL 3 STOPPED — missing upstream audit file:\n\n"
            f"{required_path}\n\n"
            "ให้ Run CELL 0 → CELL 1 → CELL 2 ก่อน"
        )


# ------------------------------------------------------------
# mes_1m ต้องมาจาก full DBN decode ใน CELL 2
# เราไม่ decode ซ้ำโดยไม่จำเป็น
# ------------------------------------------------------------

if "mes_1m" not in globals():

    raise RuntimeError(
        "CELL 3 STOPPED — mes_1m not found in runtime.\n\n"
        "ให้ Run CELL 2 ก่อน แล้ว Run CELL 3 "
        "โดยไม่ Restart session"
    )


if not isinstance(
    mes_1m,
    pd.DataFrame,
):

    raise RuntimeError(
        "CELL 3 STOPPED — mes_1m is not a DataFrame"
    )


if not isinstance(
    mes_1m.index,
    pd.DatetimeIndex,
):

    raise RuntimeError(
        "CELL 3 STOPPED — mes_1m index "
        "is not DatetimeIndex"
    )


# ------------------------------------------------------------
# 4) Load upstream audit artifacts
# ------------------------------------------------------------

with open(
    RUNTIME_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    runtime_audit = (
        json.load(f)
    )


with open(
    CELL2_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    cell2_audit = (
        json.load(f)
    )


with open(
    BASELINE_PATH,
    "r",
    encoding="utf-8",
) as f:

    baseline = (
        json.load(f)
    )


# ------------------------------------------------------------
# 5) Bind CELL 3 to exact frozen raw source
#
# Cell 3 ต้อง audit DBN identity เดียวกับ baseline
# ------------------------------------------------------------

runtime_raw = (
    runtime_audit[
        "raw_file"
    ]
)

baseline_raw = (
    baseline[
        "raw_file"
    ]
)


identity_failures = []


if (
    runtime_raw[
        "sha256"
    ]
    !=
    baseline_raw[
        "sha256"
    ]
):

    identity_failures.append(
        "Runtime raw SHA256 != frozen baseline SHA256"
    )


if (
    int(
        runtime_raw[
            "size_bytes"
        ]
    )
    !=
    int(
        baseline_raw[
            "size_bytes"
        ]
    )
):

    identity_failures.append(
        "Runtime raw size != frozen baseline size"
    )


if identity_failures:

    raise RuntimeError(
        "CELL 3 RAW IDENTITY GATE: FAIL\n\n"
        + "\n".join(
            identity_failures
        )
    )


# ------------------------------------------------------------
# 6) Metadata from CELL 1
# ------------------------------------------------------------

metadata = (
    runtime_audit.get(
        "dbn_metadata",
        {}
    )
)

mapping_summary = (
    runtime_audit
    .get(
        "dbn_metadata_bulky_summary",
        {}
    )
    .get(
        "mappings",
        {}
    )
)


# ------------------------------------------------------------
# 7) Normalization helpers
#
# รองรับทั้ง:
#   "ohlcv-1m"
#   "Schema.OHLCV_1M"
#
# และ:
#   "continuous"
#   "SType.CONTINUOUS"
# ------------------------------------------------------------

def normalize_token(value):

    if value is None:
        return None

    text = (
        str(value)
        .strip()
        .lower()
    )

    # ถ้าเป็น enum เช่น Schema.OHLCV_1M
    if "." in text:
        text = text.split(".")[-1]

    # ตัด punctuation เพื่อเทียบ semantic token
    return re.sub(
        r"[^a-z0-9]+",
        "",
        text,
    )


def normalize_dataset(value):

    if value is None:
        return None

    text = (
        str(value)
        .strip()
    )

    # รองรับ enum-like representation
    if text.lower().startswith("dataset."):

        text = (
            text
            .split(".", 1)[1]
        )

    return text.upper()


def normalize_symbols(value):

    if value is None:
        return []

    if isinstance(
        value,
        (list, tuple, set),
    ):

        return [
            str(x)
            for x in value
        ]

    return [
        str(value)
    ]


def metadata_timestamp_utc(value):

    if value is None:
        return None


    # DBN metadata อาจเก็บ timestamp เป็น integer nanoseconds
    if isinstance(
        value,
        (int, np.integer),
    ):

        return pd.to_datetime(
            int(value),
            unit="ns",
            utc=True,
        )


    # หรือเป็น string/datetime
    return pd.to_datetime(
        value,
        utc=True,
    )


# ------------------------------------------------------------
# 8) Expected provenance
# ------------------------------------------------------------

EXPECTED_PROVENANCE = {

    "dataset":
        "GLBX.MDP3",

    "schema":
        "ohlcv1m",

    "stype_in":
        "continuous",

    "stype_out":
        "instrumentid",

    "symbol":
        "MES.v.0",

    "start":
        pd.Timestamp(
            "2019-04-15T00:00:00Z"
        ),

    "end":
        pd.Timestamp(
            "2026-08-01T00:00:00Z"
        ),
}


# ------------------------------------------------------------
# 9) Observed provenance
# ------------------------------------------------------------

observed_dataset = (
    normalize_dataset(
        metadata.get(
            "dataset"
        )
    )
)

observed_schema = (
    normalize_token(
        metadata.get(
            "schema"
        )
    )
)

observed_stype_in = (
    normalize_token(
        metadata.get(
            "stype_in"
        )
    )
)

observed_stype_out = (
    normalize_token(
        metadata.get(
            "stype_out"
        )
    )
)

observed_symbols = (
    normalize_symbols(
        metadata.get(
            "symbols"
        )
    )
)


try:

    observed_start = (
        metadata_timestamp_utc(
            metadata.get(
                "start"
            )
        )
    )

except Exception as e:

    observed_start = None

    provenance_timestamp_start_error = (
        f"{type(e).__name__}: {e}"
    )

else:

    provenance_timestamp_start_error = None


try:

    observed_end = (
        metadata_timestamp_utc(
            metadata.get(
                "end"
            )
        )
    )

except Exception as e:

    observed_end = None

    provenance_timestamp_end_error = (
        f"{type(e).__name__}: {e}"
    )

else:

    provenance_timestamp_end_error = None


mapping_first_key = (
    mapping_summary.get(
        "first_symbol_key"
    )
)

mapping_last_key = (
    mapping_summary.get(
        "last_symbol_key"
    )
)


# ------------------------------------------------------------
# 10) Provenance hard gate
# ------------------------------------------------------------

provenance_checks = {

    "dataset":
        (
            observed_dataset
            ==
            EXPECTED_PROVENANCE[
                "dataset"
            ]
        ),

    "schema":
        (
            observed_schema
            ==
            EXPECTED_PROVENANCE[
                "schema"
            ]
        ),

    "stype_in":
        (
            observed_stype_in
            ==
            EXPECTED_PROVENANCE[
                "stype_in"
            ]
        ),

    "stype_out":
        (
            observed_stype_out
            ==
            EXPECTED_PROVENANCE[
                "stype_out"
            ]
        ),

    "symbol_in_metadata":
        (
            EXPECTED_PROVENANCE[
                "symbol"
            ]
            in
            observed_symbols
        ),

    "symbol_in_mapping_first":
        (
            str(
                mapping_first_key
            )
            ==
            EXPECTED_PROVENANCE[
                "symbol"
            ]
        ),

    "symbol_in_mapping_last":
        (
            str(
                mapping_last_key
            )
            ==
            EXPECTED_PROVENANCE[
                "symbol"
            ]
        ),

    "start":
        (
            observed_start
            ==
            EXPECTED_PROVENANCE[
                "start"
            ]
        ),

    "end_exclusive":
        (
            observed_end
            ==
            EXPECTED_PROVENANCE[
                "end"
            ]
        ),
}


failed_provenance = [

    key
    for key, passed
    in provenance_checks.items()

    if not passed
]


# ------------------------------------------------------------
# 11) Print actual provenance values
#
# รอบนี้เราไม่พิมพ์แค่ชื่อ field
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "DBN PROVENANCE HARD GATE"
)

print(
    "=" * 72
)


print(
    "dataset          :",
    metadata.get(
        "dataset"
    )
)

print(
    "schema           :",
    metadata.get(
        "schema"
    )
)

print(
    "stype_in         :",
    metadata.get(
        "stype_in"
    )
)

print(
    "stype_out        :",
    metadata.get(
        "stype_out"
    )
)

print(
    "symbols          :",
    metadata.get(
        "symbols"
    )
)

print(
    "start            :",
    observed_start
)

print(
    "end (exclusive)  :",
    observed_end
)

print(
    "mapping first key:",
    mapping_first_key
)

print(
    "mapping last key :",
    mapping_last_key
)


print(
    "\nProvenance checks:"
)


for (
    check_name,
    passed,
) in (
    provenance_checks.items()
):

    print(
        f"  {check_name:25s}",
        "PASS"
        if passed
        else "FAIL"
    )


if failed_provenance:

    raise RuntimeError(
        "\nCELL 3 PROVENANCE GATE: FAIL\n\n"
        "Failed fields:\n  - "
        +
        "\n  - ".join(
            failed_provenance
        )
        +
        "\n\n"
        "ห้าม resample จนกว่าจะตรวจ provenance จบ"
    )


print(
    "\nDBN PROVENANCE GATE: PASS"
)


# ------------------------------------------------------------
# 12) Zero-volume audit
# ------------------------------------------------------------

if "volume" not in mes_1m.columns:

    raise RuntimeError(
        "CELL 3 STOPPED — volume column not found"
    )


zero_volume_count = int(
    (
        mes_1m[
            "volume"
        ]
        ==
        0
    ).sum()
)


positive_volume_count = int(
    (
        mes_1m[
            "volume"
        ]
        >
        0
    ).sum()
)


print(
    "\n"
    + "=" * 72
)

print(
    "ZERO-VOLUME AUDIT"
)

print(
    "=" * 72
)


print(
    "Total bars       :",
    f"{len(mes_1m):,}"
)

print(
    "Volume > 0 bars  :",
    f"{positive_volume_count:,}"
)

print(
    "Volume == 0 bars :",
    f"{zero_volume_count:,}"
)


# สำหรับ raw Databento OHLCV-1m
# เราต้องการยืนยันว่าไม่มี synthetic zero-volume bar
if zero_volume_count != 0:

    raise RuntimeError(
        "\nCELL 3 ZERO-VOLUME GATE: FAIL\n\n"
        f"พบ volume == 0 จำนวน {zero_volume_count:,} bars\n\n"
        "ห้าม resample จนกว่าจะตรวจสาเหตุ"
    )


print(
    "ZERO-VOLUME GATE: PASS"
)


# ------------------------------------------------------------
# 13) Timestamp alignment audit
#
# OHLCV-1m timestamp ควรอยู่บน minute boundary
# ------------------------------------------------------------

MINUTE_NS = int(
    pd.Timedelta(
        minutes=1
    ).value
)


index_ns = (
    mes_1m.index.asi8
)


minute_alignment_violations = int(
    np.count_nonzero(
        index_ns
        %
        MINUTE_NS
    )
)


if minute_alignment_violations != 0:

    raise RuntimeError(
        "\nCELL 3 MINUTE-ALIGNMENT GATE: FAIL\n\n"
        f"พบ timestamp ที่ไม่ตรง minute boundary "
        f"{minute_alignment_violations:,} rows"
    )


# ------------------------------------------------------------
# 14) Calculate raw time gaps efficiently
#
# ใช้ numpy.diff บน nanoseconds
# ไม่สร้าง reindex / synthetic bars
# ------------------------------------------------------------

delta_ns = (
    np.diff(
        index_ns
    )
)


non_positive_gap_count = int(
    np.count_nonzero(
        delta_ns
        <=
        0
    )
)


if non_positive_gap_count != 0:

    raise RuntimeError(
        "\nCELL 3 GAP ORDER GATE: FAIL\n\n"
        f"พบ non-positive gaps "
        f"{non_positive_gap_count:,}"
    )


non_integer_minute_gap_count = int(
    np.count_nonzero(
        delta_ns
        %
        MINUTE_NS
    )
)


if non_integer_minute_gap_count != 0:

    raise RuntimeError(
        "\nCELL 3 GAP ALIGNMENT GATE: FAIL\n\n"
        f"พบ gap ที่ไม่ใช่จำนวนเต็มนาที "
        f"{non_integer_minute_gap_count:,}"
    )


gap_minutes = (
    delta_ns
    //
    MINUTE_NS
).astype(
    np.int64
)


# ------------------------------------------------------------
# 15) Gap distribution
# ------------------------------------------------------------

unique_gap_minutes, gap_counts = (
    np.unique(
        gap_minutes,
        return_counts=True,
    )
)


gap_distribution = pd.DataFrame(
    {
        "gap_minutes":
            unique_gap_minutes,

        "count":
            gap_counts,
    }
)


gap_distribution[
    "gap_duration"
] = pd.to_timedelta(
    gap_distribution[
        "gap_minutes"
    ],
    unit="m",
).astype(
    str
)


gap_distribution[
    "pct_of_transitions"
] = (
    gap_distribution[
        "count"
    ]
    /
    len(
        gap_minutes
    )
    *
    100.0
)


# เรียงตามจำนวนครั้งมากที่สุด
gap_distribution_by_count = (
    gap_distribution
    .sort_values(
        [
            "count",
            "gap_minutes",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


gap_distribution.to_csv(
    GAP_DISTRIBUTION_PATH,
    index=False,
)


# ------------------------------------------------------------
# 16) Gap summary
# ------------------------------------------------------------

one_minute_count = int(
    np.count_nonzero(
        gap_minutes
        ==
        1
    )
)


gap_gt_1_mask = (
    gap_minutes
    >
    1
)


gap_gt_1_count = int(
    np.count_nonzero(
        gap_gt_1_mask
    )
)


gap_gt_1_pct = (
    gap_gt_1_count
    /
    len(
        gap_minutes
    )
    *
    100.0
)


max_gap_minutes = int(
    gap_minutes.max()
)


# ------------------------------------------------------------
# 17) Build event-level gap table
#
# เก็บเฉพาะ transition ที่ > 1 นาที
# ------------------------------------------------------------

gap_positions = (
    np.flatnonzero(
        gap_gt_1_mask
    )
)


prev_positions = (
    gap_positions
)

next_positions = (
    gap_positions
    +
    1
)


gap_events = pd.DataFrame(
    {
        "prev_timestamp":
            mes_1m.index[
                prev_positions
            ],

        "next_timestamp":
            mes_1m.index[
                next_positions
            ],

        "gap_minutes":
            gap_minutes[
                gap_positions
            ],
    }
)


# ------------------------------------------------------------
# Instrument IDs around each gap
# ------------------------------------------------------------

if (
    "instrument_id"
    in mes_1m.columns
):

    instrument_values = (
        mes_1m[
            "instrument_id"
        ].to_numpy()
    )


    gap_events[
        "instrument_before"
    ] = (
        instrument_values[
            prev_positions
        ]
    )


    gap_events[
        "instrument_after"
    ] = (
        instrument_values[
            next_positions
        ]
    )


    gap_events[
        "roll_boundary"
    ] = (
        gap_events[
            "instrument_before"
        ].to_numpy()
        !=
        gap_events[
            "instrument_after"
        ].to_numpy()
    )


else:

    gap_events[
        "instrument_before"
    ] = None

    gap_events[
        "instrument_after"
    ] = None

    gap_events[
        "roll_boundary"
    ] = False


gap_events.to_parquet(
    GAP_EVENTS_PATH,
    index=False,
)


gap_events_crossing_roll = int(
    gap_events[
        "roll_boundary"
    ].sum()
)


# ------------------------------------------------------------
# 18) Specific gap sizes worth surfacing
#
# ไม่ hardcode ว่าต้องมีจำนวนเท่าใด
# แค่รายงาน observation จริง
# ------------------------------------------------------------

WATCH_GAPS_MINUTES = [
    2,
    3,
    16,
    61,
]


watch_gap_counts = {}


for minutes in WATCH_GAPS_MINUTES:

    watch_gap_counts[
        str(minutes)
    ] = int(
        np.count_nonzero(
            gap_minutes
            ==
            minutes
        )
    )


# ------------------------------------------------------------
# 19) Load roll facts from CELL 2 for cross-reference
# ------------------------------------------------------------

decoded_cell2 = (
    cell2_audit.get(
        "decoded",
        {}
    )
)


unique_instrument_ids = (
    decoded_cell2.get(
        "unique_instrument_ids"
    )
)

roll_transition_count = (
    decoded_cell2.get(
        "roll_transition_count"
    )
)

metadata_mapping_intervals = (
    decoded_cell2.get(
        "metadata_mapping_intervals"
    )
)


# ------------------------------------------------------------
# 20) Supplemental audit artifact
# ------------------------------------------------------------

cell3_audit = {

    "audit_written_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "raw_identity_binding": {

        "sha256":
            runtime_raw[
                "sha256"
            ],

        "size_bytes":
            int(
                runtime_raw[
                    "size_bytes"
                ]
            ),

        "matches_frozen_baseline":
            True,
    },


    "provenance": {

        "expected": {
            "dataset":
                EXPECTED_PROVENANCE[
                    "dataset"
                ],

            "schema":
                "ohlcv-1m",

            "stype_in":
                EXPECTED_PROVENANCE[
                    "stype_in"
                ],

            "stype_out":
                "instrument_id",

            "symbol":
                EXPECTED_PROVENANCE[
                    "symbol"
                ],

            "start":
                EXPECTED_PROVENANCE[
                    "start"
                ].isoformat(),

            "end_exclusive":
                EXPECTED_PROVENANCE[
                    "end"
                ].isoformat(),
        },


        "observed": {

            "dataset":
                metadata.get(
                    "dataset"
                ),

            "schema":
                metadata.get(
                    "schema"
                ),

            "stype_in":
                metadata.get(
                    "stype_in"
                ),

            "stype_out":
                metadata.get(
                    "stype_out"
                ),

            "symbols":
                metadata.get(
                    "symbols"
                ),

            "start":
                (
                    observed_start.isoformat()
                    if observed_start is not None
                    else None
                ),

            "end":
                (
                    observed_end.isoformat()
                    if observed_end is not None
                    else None
                ),

            "mapping_first_symbol":
                mapping_first_key,

            "mapping_last_symbol":
                mapping_last_key,

            "mapping_intervals":
                mapping_summary.get(
                    "total_mapping_intervals"
                ),
        },


        "checks":
            provenance_checks,

        "status":
            "PASS",
    },


    "zero_volume": {

        "total_bars":
            int(
                len(
                    mes_1m
                )
            ),

        "positive_volume_bars":
            positive_volume_count,

        "zero_volume_bars":
            zero_volume_count,

        "status":
            "PASS",
    },


    "raw_gap_audit": {

        "total_transitions":
            int(
                len(
                    gap_minutes
                )
            ),

        "one_minute_transitions":
            one_minute_count,

        "gap_gt_1_minute_events":
            gap_gt_1_count,

        "gap_gt_1_minute_pct":
            float(
                gap_gt_1_pct
            ),

        "maximum_gap_minutes":
            max_gap_minutes,

        "minute_alignment_violations":
            minute_alignment_violations,

        "non_integer_minute_gaps":
            non_integer_minute_gap_count,

        "non_positive_gaps":
            non_positive_gap_count,

        "watch_gap_counts":
            watch_gap_counts,

        "gap_events_crossing_roll":
            gap_events_crossing_roll,

        "distribution_file":
            str(
                GAP_DISTRIBUTION_PATH
            ),

        "event_file":
            str(
                GAP_EVENTS_PATH
            ),
    },


    "roll_cross_reference": {

        "unique_instrument_ids":
            unique_instrument_ids,

        "roll_transition_count":
            roll_transition_count,

        "metadata_mapping_intervals":
            metadata_mapping_intervals,
    },
}


with open(
    CELL3_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell3_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 21) Output — Raw gap distribution
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "RAW 1-MINUTE TIME-GAP AUDIT"
)

print(
    "=" * 72
)


print(
    "Total timestamp transitions :",
    f"{len(gap_minutes):,}"
)

print(
    "Exactly 1-minute transitions:",
    f"{one_minute_count:,}"
)

print(
    "Gap > 1 minute events       :",
    f"{gap_gt_1_count:,}"
)

print(
    "Gap > 1 minute %            :",
    f"{gap_gt_1_pct:.4f}%"
)

print(
    "Maximum observed gap        :",
    pd.Timedelta(
        minutes=max_gap_minutes
    )
)

print(
    "Minute alignment violations :",
    minute_alignment_violations
)

print(
    "Non-integer-minute gaps     :",
    non_integer_minute_gap_count
)

print(
    "Gap events crossing roll    :",
    gap_events_crossing_roll
)


# ------------------------------------------------------------
# 22) Output — watched gap sizes
# ------------------------------------------------------------

print(
    "\nSelected gap sizes:"
)


for minutes in WATCH_GAPS_MINUTES:

    count = (
        watch_gap_counts[
            str(minutes)
        ]
    )

    print(
        f"  {minutes:4d} minute gap : "
        f"{count:,}"
    )


# ------------------------------------------------------------
# 23) Output — most common gap durations
# ------------------------------------------------------------

print(
    "\nMost common gap durations:"
)


print(
    gap_distribution_by_count[
        [
            "gap_minutes",
            "gap_duration",
            "count",
            "pct_of_transitions",
        ]
    ]
    .head(15)
    .to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 24) Example gap events
#
# แสดงเฉพาะตัวอย่างเพื่อให้มนุษย์อ่านได้
# full list ถูก save ลง parquet แล้ว
# ------------------------------------------------------------

print(
    "\nFirst 10 gap events (>1 minute):"
)


if len(
    gap_events
) > 0:

    print(
        gap_events
        .head(10)
        .to_string(
            index=False
        )
    )

else:

    print(
        "None"
    )


# ------------------------------------------------------------
# 25) Artifact locations
# ------------------------------------------------------------

print(
    "\n=== CELL 3 AUDIT ARTIFACTS ==="
)

print(
    "Supplemental audit :",
    CELL3_AUDIT_PATH
)

print(
    "Gap distribution   :",
    GAP_DISTRIBUTION_PATH
)

print(
    "Gap events         :",
    GAP_EVENTS_PATH
)


# ------------------------------------------------------------
# 26) Final gate
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "CELL 3 SUPPLEMENTAL RAW AUDIT: PASS"
)

print(
    "=" * 72
)


DBN PROVENANCE HARD GATE
dataset          : GLBX.MDP3
schema           : ohlcv-1m
stype_in         : continuous
stype_out        : instrument_id
symbols          : ['MES.v.0']
start            : 2019-04-15 00:00:00+00:00
end (exclusive)  : 2026-08-01 00:00:00+00:00
mapping first key: MES.v.0
mapping last key : MES.v.0

Provenance checks:
  dataset                   PASS
  schema                    PASS
  stype_in                  PASS
  stype_out                 PASS
  symbol_in_metadata        PASS
  symbol_in_mapping_first   PASS
  symbol_in_mapping_last    PASS
  start                     PASS
  end_exclusive             PASS

DBN PROVENANCE GATE: PASS

ZERO-VOLUME AUDIT
Total bars       : 2,551,123
Volume > 0 bars  : 2,551,123
Volume == 0 bars : 0
ZERO-VOLUME GATE: PASS

RAW 1-MINUTE TIME-GAP AUDIT
Total timestamp transitions : 2,551,122
Exactly 1-minute transitions: 2,543,191
Gap > 1 minute events       : 7,931
Gap > 1 minute %            : 0.3109%
Maximum observed gap        : 3

In [41]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 4 — Databento Dataset Condition Registry
# ============================================================
#
# PURPOSE
# -------
# ขอ "สถานะคุณภาพข้อมูลรายวัน" ของ GLBX.MDP3 จาก Databento
#
# IMPORTANT
# ---------
# - ไม่ download MES OHLCV ใหม่
# - Raw market data ยังอ่านจาก Google Drive เท่านั้น
# - API key ใช้เฉพาะ Metadata request
# - Query สำเร็จแล้วจะ cache ลง Drive
# - รอบต่อไปใช้ cache ไม่ต้องเรียก API ซ้ำ
# ============================================================


# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import json
import hashlib

import pandas as pd
import databento as db


# ------------------------------------------------------------
# 2) Paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)

DATA_DIR = (
    PROJECT_DIR /
    "Data"
)

CLEAN_OUTPUT_DIR = (
    DATA_DIR /
    "MES_Clean_Pipeline_V1"
)

BASELINE_PATH = (
    CLEAN_OUTPUT_DIR /
    "raw_source_baseline.json"
)

CELL3_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell3_supplemental_raw_audit.json"
)

CONDITION_REGISTRY_PATH = (
    CLEAN_OUTPUT_DIR /
    "databento_glbx_mdp3_condition_registry.csv"
)

FLAGGED_CONDITIONS_PATH = (
    CLEAN_OUTPUT_DIR /
    "databento_glbx_mdp3_flagged_conditions.csv"
)

CONDITION_META_PATH = (
    CLEAN_OUTPUT_DIR /
    "databento_glbx_mdp3_condition_registry_meta.json"
)

CELL4_AUDIT_PATH = (
    CLEAN_OUTPUT_DIR /
    "cell4_dataset_condition_audit.json"
)


# ------------------------------------------------------------
# 3) Upstream gates
# ------------------------------------------------------------

for required_path in [
    BASELINE_PATH,
    CELL3_AUDIT_PATH,
]:

    if not required_path.exists():

        raise RuntimeError(
            "CELL 4 STOPPED — missing upstream audit:\n\n"
            f"{required_path}\n\n"
            "Run CELL 0 → CELL 3 first."
        )


# ------------------------------------------------------------
# 4) Condition request
#
# Databento condition API ใช้ end_date แบบ inclusive
#
# Raw data request เดิม:
# start = 2019-04-15
# end   = 2026-08-01 exclusive
#
# ดังนั้น condition request:
# end_date = 2026-07-31 inclusive
# ------------------------------------------------------------

CONDITION_REQUEST = {
    "dataset": "GLBX.MDP3",
    "start_date": "2019-04-15",
    "end_date_inclusive": "2026-07-31",
}


VALID_CONDITIONS = {
    "available",
    "degraded",
    "pending",
    "missing",
}


# ------------------------------------------------------------
# 5) Historical warning evidence
# ------------------------------------------------------------

ORIGINAL_WARNING_EVIDENCE = {
    "known_flagged_dates": [
        "2020-02-27",
        "2020-02-28",
        "2020-06-30",
    ],

    "note":
        (
            "Original Databento warning was truncated. "
            "These dates are preserved only as known "
            "historical warning evidence."
        ),
}


# ------------------------------------------------------------
# 6) Helper — SHA256
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(
                8 * 1024 * 1024
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# ------------------------------------------------------------
# 7) Helper — validate registry
# ------------------------------------------------------------

def validate_condition_registry(df):

    required_columns = {
        "date",
        "condition",
        "last_modified_date",
    }


    missing = (
        required_columns
        -
        set(df.columns)
    )


    if missing:

        raise RuntimeError(
            "Condition registry missing columns: "
            + ", ".join(
                sorted(missing)
            )
        )


    work = df.copy()


    work["date"] = pd.to_datetime(
        work["date"],
        errors="raise",
    ).dt.date


    work["condition"] = (
        work["condition"]
        .astype(str)
        .str.lower()
        .str.strip()
    )


    invalid = (
        set(
            work["condition"]
        )
        -
        VALID_CONDITIONS
    )


    if invalid:

        raise RuntimeError(
            "Unknown Databento condition: "
            + ", ".join(
                sorted(invalid)
            )
        )


    duplicate_dates = int(
        work["date"]
        .duplicated()
        .sum()
    )


    if duplicate_dates:

        raise RuntimeError(
            "Duplicate condition dates: "
            f"{duplicate_dates}"
        )


    work = (
        work
        .sort_values("date")
        .reset_index(drop=True)
    )


    return work


# ------------------------------------------------------------
# 8) Prefer existing local cache
#
# ถ้ามี cache แล้ว:
# ไม่ใช้ API key
# ไม่เรียก Databento
# ------------------------------------------------------------

USE_CACHE = (
    CONDITION_REGISTRY_PATH.exists()
    and
    CONDITION_META_PATH.exists()
)


if USE_CACHE:

    print(
        "Existing condition registry cache found."
    )

    print(
        "Using Drive cache — Databento API will NOT be called."
    )


    condition_df = pd.read_csv(
        CONDITION_REGISTRY_PATH
    )


    condition_df = (
        validate_condition_registry(
            condition_df
        )
    )


    with open(
        CONDITION_META_PATH,
        "r",
        encoding="utf-8",
    ) as f:

        registry_meta = (
            json.load(f)
        )


    registry_source = (
        "local_drive_cache"
    )


# ------------------------------------------------------------
# 9) No cache → read API key from Colab Secret
# ------------------------------------------------------------

else:

    print(
        "No local condition registry cache found."
    )


    try:

        from google.colab import userdata

        API_KEY = userdata.get(
            "DATABENTO_API_KEY"
        )


    except Exception as e:

        raise RuntimeError(
            "Cannot read Colab Secret "
            "'DATABENTO_API_KEY'.\n\n"
            "เปิด Secrets (รูปกุญแจ) ใน Colab "
            "แล้วเพิ่ม DATABENTO_API_KEY "
            "และเปิด Notebook access."
        ) from e


    if not API_KEY:

        raise RuntimeError(
            "DATABENTO_API_KEY is empty.\n\n"
            "เพิ่ม API key ใน Colab Secrets ก่อน."
        )


    print(
        "Databento API key detected securely."
    )

    print(
        "Querying dataset-condition metadata only..."
    )


    # --------------------------------------------------------
    # 10) Authenticated Databento metadata request
    #
    # นี่ไม่ใช่ market-data download
    # --------------------------------------------------------

    try:

        client = db.Historical(
            API_KEY
        )


        conditions = (
            client.metadata.get_dataset_condition(
                dataset=
                    CONDITION_REQUEST[
                        "dataset"
                    ],

                start_date=
                    CONDITION_REQUEST[
                        "start_date"
                    ],

                end_date=
                    CONDITION_REQUEST[
                        "end_date_inclusive"
                    ],
            )
        )


    except Exception as e:

        raise RuntimeError(
            "Databento condition metadata request failed.\n\n"
            f"{type(e).__name__}: {e}"
        ) from e


    # --------------------------------------------------------
    # 11) Validate API response
    # --------------------------------------------------------

    condition_df = pd.DataFrame(
        conditions
    )


    condition_df = (
        validate_condition_registry(
            condition_df
        )
    )


    # --------------------------------------------------------
    # 12) Save registry to Drive
    #
    # หลังจากนี้รอบต่อไปจะใช้ cache
    # --------------------------------------------------------

    condition_df.to_csv(
        CONDITION_REGISTRY_PATH,
        index=False,
    )


    retrieved_utc = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )


    registry_meta = {

        "request":
            CONDITION_REQUEST,

        "retrieved_utc":
            retrieved_utc,

        "source":
            (
                "Databento "
                "Historical.metadata."
                "get_dataset_condition"
            ),

        "registry_sha256":
            sha256_file(
                CONDITION_REGISTRY_PATH
            ),

        "original_warning_evidence":
            ORIGINAL_WARNING_EVIDENCE,
    }


    with open(
        CONDITION_META_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            registry_meta,
            f,
            indent=2,
            ensure_ascii=False,
        )


    registry_source = (
        "databento_metadata_api"
    )


    # ลบ reference ของ key ออกจากตัวแปร
    del API_KEY


# ------------------------------------------------------------
# 13) Analyze conditions
# ------------------------------------------------------------

condition_counts = {

    str(condition):
        int(count)

    for condition, count
    in (
        condition_df[
            "condition"
        ]
        .value_counts()
        .to_dict()
        .items()
    )
}


flagged_df = (

    condition_df[
        condition_df[
            "condition"
        ]
        !=
        "available"
    ]
    .copy()
)


flagged_df.to_csv(
    FLAGGED_CONDITIONS_PATH,
    index=False,
)


flagged_count = int(
    len(flagged_df)
)


# ------------------------------------------------------------
# 14) Compare with original warning evidence
# ------------------------------------------------------------

current_flagged_dates = set(
    flagged_df[
        "date"
    ].astype(str)
)


original_known_dates = set(
    ORIGINAL_WARNING_EVIDENCE[
        "known_flagged_dates"
    ]
)


warning_comparison = {

    "known_original_warning_dates":
        sorted(
            original_known_dates
        ),

    "still_flagged_now":
        sorted(
            original_known_dates
            &
            current_flagged_dates
        ),

    "not_currently_flagged":
        sorted(
            original_known_dates
            -
            current_flagged_dates
        ),
}


# ------------------------------------------------------------
# 15) Save CELL 4 audit
# ------------------------------------------------------------

cell4_audit = {

    "audit_written_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "request":
        CONDITION_REQUEST,

    "registry_status":
        "AVAILABLE",

    "registry_source":
        registry_source,

    "rows":
        int(
            len(condition_df)
        ),

    "condition_counts":
        condition_counts,

    "flagged_rows":
        flagged_count,

    "historical_warning_comparison":
        warning_comparison,

    "policy": {

        "condition_scope":
            "GLBX.MDP3 dataset-level",

        "automatic_mes_row_deletion":
            False,

        "automatic_mes_day_exclusion":
            False,
    },
}


with open(
    CELL4_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell4_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 16) Output
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "DATABENTO DATASET CONDITION REGISTRY"
)

print(
    "=" * 72
)


print(
    "Dataset         :",
    CONDITION_REQUEST[
        "dataset"
    ]
)

print(
    "Registry source :",
    registry_source
)

print(
    "Rows            :",
    f"{len(condition_df):,}"
)

print(
    "First date      :",
    condition_df[
        "date"
    ].min()
)

print(
    "Last date       :",
    condition_df[
        "date"
    ].max()
)


print(
    "\nCondition counts:"
)


for condition_name in [
    "available",
    "degraded",
    "pending",
    "missing",
]:

    print(
        f"  {condition_name:10s}:",
        f"{condition_counts.get(condition_name, 0):,}"
    )


print(
    "\nNon-available dates:",
    f"{flagged_count:,}"
)


if flagged_count:

    print(
        "\nFlagged dates:"
    )

    print(
        flagged_df
        .to_string(
            index=False
        )
    )


print(
    "\nOriginal-warning comparison:"
)

print(
    warning_comparison
)


print(
    "\nRegistry saved:"
)

print(
    CONDITION_REGISTRY_PATH
)

print(
    "\nCELL 4 audit:"
)

print(
    CELL4_AUDIT_PATH
)


print(
    "\n"
    + "=" * 72
)

print(
    "CELL 4 DATASET CONDITION REGISTRY: PASS"
)

print(
    "=" * 72
)

Existing condition registry cache found.
Using Drive cache — Databento API will NOT be called.

DATABENTO DATASET CONDITION REGISTRY
Dataset         : GLBX.MDP3
Registry source : local_drive_cache
Rows            : 2,319
First date      : 2019-04-15
Last date       : 2026-07-31

Condition counts:
  available : 2,302
  degraded  : 17
  pending   : 0
  missing   : 0

Non-available dates: 17

Flagged dates:
      date condition last_modified_date
2020-02-27  degraded         2026-05-27
2020-02-28  degraded         2026-05-28
2020-06-30  degraded         2026-05-22
2020-07-01  degraded         2026-05-22
2021-12-05  degraded         2026-05-18
2022-01-02  degraded         2026-04-03
2024-09-18  degraded         2026-04-21
2025-09-17  degraded         2026-06-11
2025-09-24  degraded         2026-06-12
2025-11-28  degraded         2026-06-09
2026-01-31  degraded         2026-06-04
2026-03-15  degraded         2026-06-05
2026-03-16  degraded         2026-06-06
2026-03-21  degraded         202

In [42]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 5 — 1m → 15m RESAMPLE + DATA INTEGRITY AUDIT
# ============================================================
#
# PURPOSE
# -------
# 1) Resample canonical MES 1-minute data → 15-minute bars
# 2) Never forward-fill or create synthetic price bars
# 3) Count actual 1-minute observations inside every 15m bar
# 4) Preserve / restore instrument_id dtype
# 5) Detect mixed-contract (roll-crossing) 15m bars
# 6) Build a first data-integrity gate for V1 decision times
# 7) Audit partial bars inside 09:45–15:00 New York
# 8) Attach Databento dataset-condition flags
# 9) Compare degraded dates with actual MES observations
# 10) Audit early MES liquidity without choosing a cutoff yet
# 11) Cross-check clean 15m OHLCV against old audited parquet
#
# IMPORTANT
# ---------
# - This cell does NOT create features.
# - This cell does NOT create labels.
# - This cell does NOT remove degraded days automatically.
# - This cell does NOT choose a warm-up cutoff.
# - Raw-gap attribution belongs to CELL 6.
# ============================================================


# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2) Project paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)

DATA_DIR = (
    PROJECT_DIR
    / "Data"
)

CLEAN_DIR = (
    DATA_DIR
    / "MES_Clean_Pipeline_V1"
)


# ------------------------------------------------------------
# 3) Upstream audit / registry paths
# ------------------------------------------------------------

BASELINE_PATH = (
    CLEAN_DIR
    / "raw_source_baseline.json"
)

CELL3_AUDIT_PATH = (
    CLEAN_DIR
    / "cell3_supplemental_raw_audit.json"
)

CELL4_AUDIT_PATH = (
    CLEAN_DIR
    / "cell4_dataset_condition_audit.json"
)

CONDITION_PATH = (
    CLEAN_DIR
    / "databento_glbx_mdp3_condition_registry.csv"
)


# ------------------------------------------------------------
# 4) Old research artifact
#
# Optional.
# Used only for semantic cross-check.
# It is NOT the source of truth.
# ------------------------------------------------------------

OLD_15M_PATH = (
    DATA_DIR
    / "MES_2019_2026_15m.parquet"
)


# ------------------------------------------------------------
# 5) CELL 5 output artifacts
# ------------------------------------------------------------

MES_15M_PATH = (
    CLEAN_DIR
    / "MES_2019_2026_15m_clean.parquet"
)

CELL5_AUDIT_PATH = (
    CLEAN_DIR
    / "cell5_15m_resample_audit.json"
)

V1_PARTIAL_PATH = (
    CLEAN_DIR
    / "cell5_v1_clock_partial_bars.parquet"
)

DEGRADED_IMPACT_PATH = (
    CLEAN_DIR
    / "cell5_degraded_day_mes_impact.csv"
)

LAUNCH_LIQUIDITY_PATH = (
    CLEAN_DIR
    / "cell5_launch_liquidity_monthly.csv"
)


# ------------------------------------------------------------
# 6) Policy constants
# ------------------------------------------------------------

NY_TZ = "America/New_York"

RESAMPLE_RULE = "15min"

# V1 clock-only decision window
#
# IMPORTANT:
# Calendar / holidays / early closes are NOT applied here.
# Those belong to the next decision-universe layer.
#
# Decision time:
# 09:45 → 15:00 New York

V1_START_MINUTE = (
    9 * 60
    + 45
)

V1_END_MINUTE = (
    15 * 60
)


# ------------------------------------------------------------
# 7) Upstream artifact gates
# ------------------------------------------------------------

required_artifacts = [
    BASELINE_PATH,
    CELL3_AUDIT_PATH,
    CELL4_AUDIT_PATH,
    CONDITION_PATH,
]


for path in required_artifacts:

    if not path.exists():

        raise RuntimeError(
            "CELL 5 STOPPED — missing upstream artifact:\n"
            f"{path}\n\n"
            "Run CELL 0 → CELL 4 first."
        )


# ------------------------------------------------------------
# 8) mes_1m runtime gate
#
# mes_1m must come from canonical DBN decode in CELL 2.
# ------------------------------------------------------------

if "mes_1m" not in globals():

    raise RuntimeError(
        "CELL 5 STOPPED — mes_1m is not in memory.\n\n"
        "Run CELL 0 → CELL 4 first."
    )


if not isinstance(
    mes_1m,
    pd.DataFrame,
):

    raise RuntimeError(
        "CELL 5 STOPPED — mes_1m is not a pandas DataFrame."
    )


if not isinstance(
    mes_1m.index,
    pd.DatetimeIndex,
):

    raise RuntimeError(
        "CELL 5 STOPPED — mes_1m index is not DatetimeIndex."
    )


# ------------------------------------------------------------
# 9) Required raw columns
# ------------------------------------------------------------

REQUIRED_RAW_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "instrument_id",
]


missing_raw_columns = [
    col
    for col in REQUIRED_RAW_COLUMNS
    if col not in mes_1m.columns
]


if missing_raw_columns:

    raise RuntimeError(
        "CELL 5 STOPPED — missing raw columns:\n"
        + ", ".join(
            missing_raw_columns
        )
    )


# ------------------------------------------------------------
# 10) Raw timestamp integrity
# ------------------------------------------------------------

if not mes_1m.index.is_monotonic_increasing:

    raise RuntimeError(
        "CELL 5 STOPPED — raw timestamps are not monotonic."
    )


raw_duplicate_timestamps = int(
    mes_1m.index
    .duplicated()
    .sum()
)


if raw_duplicate_timestamps != 0:

    raise RuntimeError(
        "CELL 5 STOPPED — raw duplicate timestamps found: "
        f"{raw_duplicate_timestamps:,}"
    )


if mes_1m.index.tz is None:

    raise RuntimeError(
        "CELL 5 STOPPED — raw index has no timezone."
    )


if str(
    mes_1m.index.tz
).upper() != "UTC":

    raise RuntimeError(
        "CELL 5 STOPPED — raw timezone is not UTC."
    )


# ------------------------------------------------------------
# 11) Load CELL 4 audit
# ------------------------------------------------------------

with open(
    CELL4_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    cell4_audit = json.load(f)


if (
    cell4_audit.get(
        "registry_status"
    )
    != "AVAILABLE"
):

    raise RuntimeError(
        "CELL 5 STOPPED — Databento condition registry "
        "is not AVAILABLE."
    )


# ------------------------------------------------------------
# 12) Load Databento condition registry
# ------------------------------------------------------------

condition_df = pd.read_csv(
    CONDITION_PATH
)


required_condition_columns = {
    "date",
    "condition",
    "last_modified_date",
}


if not required_condition_columns.issubset(
    condition_df.columns
):

    raise RuntimeError(
        "CELL 5 STOPPED — condition registry columns incomplete."
    )


condition_df[
    "date"
] = (
    pd.to_datetime(
        condition_df[
            "date"
        ],
        utc=True,
    )
    .dt
    .floor(
        "D"
    )
)


condition_df[
    "condition"
] = (
    condition_df[
        "condition"
    ]
    .astype(str)
    .str
    .lower()
    .str
    .strip()
)


if condition_df[
    "date"
].duplicated().any():

    raise RuntimeError(
        "CELL 5 STOPPED — duplicate condition dates found."
    )


condition_map = (
    condition_df
    .set_index(
        "date"
    )[
        "condition"
    ]
)


# ------------------------------------------------------------
# 13) Resample OHLCV: 1m → 15m
#
# Databento ts_event represents bar START.
#
# Therefore:
#
# index 22:00 represents:
# [22:00, 22:15)
#
# decision_time becomes:
# 22:15
#
# No forward-fill.
# ------------------------------------------------------------

print(
    "CELL 5 — Resampling canonical MES 1m → 15m..."
)


ohlcv_15m = (

    mes_1m[
        [
            "open",
            "high",
            "low",
            "close",
            "volume",
        ]
    ]

    .resample(
        RESAMPLE_RULE,
        label="left",
        closed="left",
    )

    .agg(
        {
            "open":
                "first",

            "high":
                "max",

            "low":
                "min",

            "close":
                "last",

            "volume":
                "sum",
        }
    )
)


# ------------------------------------------------------------
# 14) Count actual raw 1m bars
#
# active_1m_count:
#
# 15 = complete 15-minute interval
#
# <15 = at least one expected minute does not have an
#       OHLCV trade record
#
# IMPORTANT:
# This does NOT by itself tell us WHY the minute is absent.
# ------------------------------------------------------------

active_1m_count = (

    mes_1m[
        "close"
    ]

    .resample(
        RESAMPLE_RULE,
        label="left",
        closed="left",
    )

    .count()
)


# ------------------------------------------------------------
# 15) Instrument aggregation
# ------------------------------------------------------------

instrument_first = (

    mes_1m[
        "instrument_id"
    ]

    .resample(
        RESAMPLE_RULE,
        label="left",
        closed="left",
    )

    .first()
)


instrument_count = (

    mes_1m[
        "instrument_id"
    ]

    .resample(
        RESAMPLE_RULE,
        label="left",
        closed="left",
    )

    .nunique()
)


# ------------------------------------------------------------
# 16) Assemble initial 15m dataframe
# ------------------------------------------------------------

mes_15m = (
    ohlcv_15m
    .copy()
)


mes_15m[
    "active_1m_count"
] = (
    active_1m_count
)


mes_15m[
    "instrument_id"
] = (
    instrument_first
)


mes_15m[
    "instrument_count"
] = (
    instrument_count
)


# ------------------------------------------------------------
# 17) Empty-bin audit
#
# Pandas resample creates clock bins even when there are
# no raw OHLCV observations.
#
# We count those BEFORE removing them.
# ------------------------------------------------------------

bins_before_drop = int(
    len(
        mes_15m
    )
)


empty_bin_mask = (
    mes_15m[
        "active_1m_count"
    ]
    .eq(
        0
    )
)


empty_bins = int(
    empty_bin_mask.sum()
)


# ------------------------------------------------------------
# 18) Remove truly empty bins
#
# We do NOT synthesize a price.
# ------------------------------------------------------------

mes_15m = (

    mes_15m.loc[
        ~empty_bin_mask
    ]

    .copy()
)


# ------------------------------------------------------------
# 19) Validate counts BEFORE dtype conversion
# ------------------------------------------------------------

if mes_15m[
    "instrument_id"
].isna().any():

    raise RuntimeError(
        "CELL 5 STOPPED — instrument_id contains NaN "
        "after empty-bin removal."
    )


if (
    mes_15m[
        "active_1m_count"
    ]
    .lt(
        1
    )
    .any()
):

    raise RuntimeError(
        "CELL 5 STOPPED — active_1m_count < 1 detected."
    )


if (
    mes_15m[
        "active_1m_count"
    ]
    .gt(
        15
    )
    .any()
):

    raise RuntimeError(
        "CELL 5 STOPPED — active_1m_count > 15 detected."
    )


# ------------------------------------------------------------
# 20) Restore semantic dtypes
#
# Why:
#
# Temporary empty bins can force Pandas to convert
# instrument_id:
#
# uint32 → float64
#
# Example:
#
# 7849 → 7849.0
#
# After empty bins are removed and NaN == 0,
# restore the original raw dtype.
# ------------------------------------------------------------

RAW_INSTRUMENT_DTYPE = (
    mes_1m[
        "instrument_id"
    ].dtype
)


mes_15m[
    "instrument_id"
] = (
    mes_15m[
        "instrument_id"
    ]
    .astype(
        RAW_INSTRUMENT_DTYPE
    )
)


mes_15m[
    "active_1m_count"
] = (
    mes_15m[
        "active_1m_count"
    ]
    .astype(
        "uint8"
    )
)


mes_15m[
    "instrument_count"
] = (
    mes_15m[
        "instrument_count"
    ]
    .astype(
        "uint8"
    )
)


# ------------------------------------------------------------
# 21) Roll-crossing flag
#
# True means one 15m bar contains >1 contract.
# ------------------------------------------------------------

mes_15m[
    "crosses_roll"
] = (
    mes_15m[
        "instrument_count"
    ]
    .gt(
        1
    )
)


# ------------------------------------------------------------
# 22) Decision time
#
# Index = bar start
# decision_time = bar end
# ------------------------------------------------------------

mes_15m[
    "decision_time"
] = (
    mes_15m.index
    +
    pd.Timedelta(
        minutes=15
    )
)


# ------------------------------------------------------------
# 23) New York decision time
#
# IMPORTANT:
# Series values require .dt.tz_convert(...)
# ------------------------------------------------------------

mes_15m[
    "decision_time_ny"
] = (
    mes_15m[
        "decision_time"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)


# ------------------------------------------------------------
# 24) Bar completeness
#
# Full bar:
# all 15 one-minute trade-bars exist
# ------------------------------------------------------------

mes_15m[
    "bar_complete_15m"
] = (
    mes_15m[
        "active_1m_count"
    ]
    .eq(
        15
    )
)


# ------------------------------------------------------------
# 25) First data-integrity policy
#
# A bar is structurally clean if:
#
# - complete 15/15
# - not mixed across futures contracts
# ------------------------------------------------------------

mes_15m[
    "data_integrity_ok"
] = (

    mes_15m[
        "bar_complete_15m"
    ]

    &

    ~mes_15m[
        "crosses_roll"
    ]
)


# ------------------------------------------------------------
# 26) V1 clock-only window
#
# This is NOT yet the final Decision Universe.
#
# NYSE calendar / holidays / early close comes later.
# ------------------------------------------------------------

decision_minute_ny = (

    mes_15m[
        "decision_time_ny"
    ]
    .dt.hour
    *
    60

    +

    mes_15m[
        "decision_time_ny"
    ]
    .dt.minute
)


mes_15m[
    "v1_clock_window"
] = (

    decision_minute_ny
    .between(
        V1_START_MINUTE,
        V1_END_MINUTE,
        inclusive="both",
    )
)


# ------------------------------------------------------------
# 27) Partial bars inside V1 clock window
# ------------------------------------------------------------

mes_15m[
    "v1_clock_partial"
] = (

    mes_15m[
        "v1_clock_window"
    ]

    &

    ~mes_15m[
        "bar_complete_15m"
    ]
)


# ------------------------------------------------------------
# 28) Integrity eligibility
#
# IMPORTANT POLICY:
#
# Partial bars remain in dataset as market context,
# but cannot become an ENTRY decision observation.
# ------------------------------------------------------------

mes_15m[
    "v1_clock_integrity_eligible"
] = (

    mes_15m[
        "v1_clock_window"
    ]

    &

    mes_15m[
        "data_integrity_ok"
    ]
)


# ------------------------------------------------------------
# 29) Attach Databento dataset-condition flag
#
# Registry date is interpreted in UTC.
#
# degraded is NOT automatic exclusion.
# ------------------------------------------------------------

bar_utc_day = pd.Series(
    mes_15m.index.floor(
        "D"
    ),
    index=mes_15m.index,
)


mes_15m[
    "dataset_condition_utc"
] = (
    bar_utc_day
    .map(
        condition_map
    )
)


missing_condition_count = int(
    mes_15m[
        "dataset_condition_utc"
    ]
    .isna()
    .sum()
)


if missing_condition_count != 0:

    raise RuntimeError(
        "CELL 5 STOPPED — missing Databento condition "
        f"for {missing_condition_count:,} 15m bars."
    )


mes_15m[
    "dataset_degraded_utc"
] = (
    mes_15m[
        "dataset_condition_utc"
    ]
    .eq(
        "degraded"
    )
)


# ------------------------------------------------------------
# 30) Structural audit
# ------------------------------------------------------------

failures = []


if not mes_15m.index.is_monotonic_increasing:

    failures.append(
        "15m index is not monotonic"
    )


duplicate_15m = int(
    mes_15m.index
    .duplicated()
    .sum()
)


if duplicate_15m != 0:

    failures.append(
        f"15m duplicate timestamps: {duplicate_15m}"
    )


# ------------------------------------------------------------
# 31) OHLCV NaN audit
# ------------------------------------------------------------

OHLCV_COLUMNS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
]


ohlcv_nan_counts = {

    col:
        int(
            mes_15m[
                col
            ]
            .isna()
            .sum()
        )

    for col in OHLCV_COLUMNS
}


if sum(
    ohlcv_nan_counts.values()
) != 0:

    failures.append(
        "15m OHLCV contains NaN: "
        f"{ohlcv_nan_counts}"
    )


# ------------------------------------------------------------
# 32) OHLC consistency audit
# ------------------------------------------------------------

high_violation = int(
    (
        mes_15m[
            "high"
        ]
        <
        mes_15m[
            [
                "open",
                "close",
                "low",
            ]
        ]
        .max(
            axis=1
        )
    )
    .sum()
)


low_violation = int(
    (
        mes_15m[
            "low"
        ]
        >
        mes_15m[
            [
                "open",
                "close",
                "high",
            ]
        ]
        .min(
            axis=1
        )
    )
    .sum()
)


if high_violation != 0:

    failures.append(
        f"15m high violations: {high_violation}"
    )


if low_violation != 0:

    failures.append(
        f"15m low violations: {low_violation}"
    )


# ------------------------------------------------------------
# 33) Roll audit
#
# For this dataset we expect zero mixed-contract bars.
# ------------------------------------------------------------

roll_crossing_bars = int(
    mes_15m[
        "crosses_roll"
    ]
    .sum()
)


max_instruments_per_bar = int(
    mes_15m[
        "instrument_count"
    ]
    .max()
)


if roll_crossing_bars != 0:

    failures.append(
        "Mixed-contract 15m bars detected: "
        f"{roll_crossing_bars}"
    )


# ------------------------------------------------------------
# 34) instrument_id dtype hard gate
# ------------------------------------------------------------

clean_instrument_dtype = (
    mes_15m[
        "instrument_id"
    ].dtype
)


instrument_dtype_match = (
    clean_instrument_dtype
    ==
    RAW_INSTRUMENT_DTYPE
)


if not instrument_dtype_match:

    failures.append(
        "instrument_id dtype mismatch: "
        f"{clean_instrument_dtype} "
        f"!= {RAW_INSTRUMENT_DTYPE}"
    )


# ------------------------------------------------------------
# 35) Completeness statistics
# ------------------------------------------------------------

full_15m_bars = int(
    mes_15m[
        "bar_complete_15m"
    ]
    .sum()
)


partial_15m_bars = int(
    (
        ~mes_15m[
            "bar_complete_15m"
        ]
    )
    .sum()
)


active_count_distribution = (

    mes_15m[
        "active_1m_count"
    ]

    .value_counts()

    .sort_index()
)


# ------------------------------------------------------------
# 36) V1 clock integrity statistics
# ------------------------------------------------------------

v1_clock_bars = int(
    mes_15m[
        "v1_clock_window"
    ]
    .sum()
)


v1_partial_bars = int(
    mes_15m[
        "v1_clock_partial"
    ]
    .sum()
)


v1_integrity_eligible_bars = int(
    mes_15m[
        "v1_clock_integrity_eligible"
    ]
    .sum()
)


# ------------------------------------------------------------
# 37) Save V1 partial-bar examples
# ------------------------------------------------------------

partial_v1 = (

    mes_15m.loc[
        mes_15m[
            "v1_clock_partial"
        ],
        [
            "open",
            "high",
            "low",
            "close",
            "volume",
            "active_1m_count",
            "instrument_id",
            "instrument_count",
            "crosses_roll",
            "dataset_condition_utc",
            "dataset_degraded_utc",
            "decision_time",
            "decision_time_ny",
        ],
    ]

    .copy()
)


partial_v1.to_parquet(
    V1_PARTIAL_PATH,
    index=True,
)


# ------------------------------------------------------------
# 38) Old 15m semantic cross-check
#
# IMPORTANT:
#
# Same row count alone is NOT enough.
#
# Compare:
# - row count
# - timestamps
# - open
# - high
# - low
# - close
# - volume
#
# observation-by-observation
# ------------------------------------------------------------

old_15m_check = {
    "status":
        "SKIPPED",

    "reason":
        "Old 15m parquet not found",
}


if OLD_15M_PATH.exists():

    old_15m = pd.read_parquet(
        OLD_15M_PATH
    )


    rows_equal = (
        len(
            old_15m
        )
        ==
        len(
            mes_15m
        )
    )


    index_equal = (

        isinstance(
            old_15m.index,
            pd.DatetimeIndex,
        )

        and

        rows_equal

        and

        old_15m.index.equals(
            mes_15m.index
        )
    )


    column_results = {}


    for col in OHLCV_COLUMNS:

        if col not in old_15m.columns:

            values_equal = False

        else:

            values_equal = (
                old_15m[
                    col
                ]
                .equals(
                    mes_15m[
                        col
                    ]
                )
            )


        column_results[
            col
        ] = {
            "values_equal":
                bool(
                    values_equal
                )
        }


    all_ohlcv_equal = all(

        result[
            "values_equal"
        ]

        for result
        in column_results.values()
    )


    old_15m_pass = (

        rows_equal
        and
        index_equal
        and
        all_ohlcv_equal
    )


    old_15m_check = {

        "status":
            (
                "PASS"
                if old_15m_pass
                else "FAIL"
            ),

        "rows_equal":
            bool(
                rows_equal
            ),

        "index_equal":
            bool(
                index_equal
            ),

        "columns":
            column_results,
    }


    if not old_15m_pass:

        failures.append(
            "Old 15m ↔ clean 15m semantic cross-check failed"
        )


    del old_15m


# ------------------------------------------------------------
# 39) MES daily raw statistics
#
# Used only to investigate Databento degraded dates.
# ------------------------------------------------------------

raw_daily = (

    mes_1m[
        [
            "close",
            "volume",
        ]
    ]

    .resample(
        "1D"
    )

    .agg(
        raw_1m_bars=(
            "close",
            "count",
        ),

        raw_volume=(
            "volume",
            "sum",
        ),
    )
)


# ------------------------------------------------------------
# 40) MES daily 15m statistics
# ------------------------------------------------------------

daily15 = pd.DataFrame(
    index=(
        mes_15m[
            "close"
        ]
        .resample(
            "1D"
        )
        .count()
        .index
    )
)


daily15[
    "bars_15m"
] = (
    mes_15m[
        "close"
    ]
    .resample(
        "1D"
    )
    .count()
)


daily15[
    "partial_15m_bars"
] = (
    (
        ~mes_15m[
            "bar_complete_15m"
        ]
    )
    .resample(
        "1D"
    )
    .sum()
)


daily15[
    "v1_clock_bars"
] = (
    mes_15m[
        "v1_clock_window"
    ]
    .resample(
        "1D"
    )
    .sum()
)


daily15[
    "v1_clock_partial_bars"
] = (
    mes_15m[
        "v1_clock_partial"
    ]
    .resample(
        "1D"
    )
    .sum()
)


# ------------------------------------------------------------
# 41) Databento degraded dates ↔ actual MES observations
#
# NOTE:
#
# Databento condition is dataset-level.
#
# We therefore do NOT assume degraded means MES is defective.
# ------------------------------------------------------------

degraded_condition_rows = (

    condition_df.loc[
        condition_df[
            "condition"
        ]
        .eq(
            "degraded"
        ),
        [
            "date",
            "condition",
            "last_modified_date",
        ],
    ]

    .copy()
)


degraded_days = (

    degraded_condition_rows

    .set_index(
        "date"
    )

    .join(
        raw_daily.join(
            daily15,
            how="outer",
        ),
        how="left",
    )

    .fillna(
        {
            "raw_1m_bars":
                0,

            "raw_volume":
                0,

            "bars_15m":
                0,

            "partial_15m_bars":
                0,

            "v1_clock_bars":
                0,

            "v1_clock_partial_bars":
                0,
        }
    )

    .reset_index()
)


degraded_days.to_csv(
    DEGRADED_IMPACT_PATH,
    index=False,
)


# ------------------------------------------------------------
# 42) Early MES liquidity diagnostic
#
# Diagnostic only.
#
# We do NOT decide:
# "remove first month"
# or
# "use ES instead"
#
# until the actual statistics are reviewed.
# ------------------------------------------------------------

liquidity_df = (
    mes_15m.copy()
)


liquidity_df[
    "ny_month"
] = (

    liquidity_df[
        "decision_time_ny"
    ]

    .dt
    .tz_localize(
        None
    )

    .dt
    .to_period(
        "M"
    )

    .astype(
        str
    )
)


liquidity_df[
    "range_points"
] = (
    liquidity_df[
        "high"
    ]
    -
    liquidity_df[
        "low"
    ]
)


# ------------------------------------------------------------
# 43) Monthly all-session liquidity statistics
# ------------------------------------------------------------

monthly_all = (

    liquidity_df

    .groupby(
        "ny_month"
    )

    .agg(
        all_bars=(
            "close",
            "size",
        ),

        all_median_volume=(
            "volume",
            "median",
        ),

        all_median_range_points=(
            "range_points",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 44) Monthly V1-clock liquidity statistics
# ------------------------------------------------------------

monthly_v1 = (

    liquidity_df.loc[
        liquidity_df[
            "v1_clock_window"
        ]
    ]

    .groupby(
        "ny_month"
    )

    .agg(
        v1_bars=(
            "close",
            "size",
        ),

        v1_partial_bars=(
            "v1_clock_partial",
            "sum",
        ),

        v1_median_volume=(
            "volume",
            "median",
        ),

        v1_p10_volume=(
            "volume",
            lambda s: float(
                s.quantile(
                    0.10
                )
            ),
        ),

        v1_median_range_points=(
            "range_points",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 45) Combine monthly liquidity audit
# ------------------------------------------------------------

launch_monthly = (
    monthly_all
    .join(
        monthly_v1,
        how="outer",
    )
)


launch_monthly[
    "v1_partial_pct"
] = (
    launch_monthly[
        "v1_partial_bars"
    ]
    /
    launch_monthly[
        "v1_bars"
    ]
    *
    100.0
)


launch_monthly.to_csv(
    LAUNCH_LIQUIDITY_PATH,
    index=True,
)


# ------------------------------------------------------------
# 46) Build CELL 5 audit artifact
# ------------------------------------------------------------

cell5_audit = {

    "audit_written_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),


    "resample": {

        "raw_rows":
            int(
                len(
                    mes_1m
                )
            ),

        "bins_before_drop":
            bins_before_drop,

        "empty_bins_dropped":
            empty_bins,

        "final_15m_bars":
            int(
                len(
                    mes_15m
                )
            ),

        "full_15m_bars":
            full_15m_bars,

        "partial_15m_bars":
            partial_15m_bars,

        "roll_crossing_bars":
            roll_crossing_bars,

        "maximum_instruments_per_bar":
            max_instruments_per_bar,

        "raw_instrument_dtype":
            str(
                RAW_INSTRUMENT_DTYPE
            ),

        "clean_instrument_dtype":
            str(
                clean_instrument_dtype
            ),

        "instrument_dtype_match":
            bool(
                instrument_dtype_match
            ),

        "active_1m_count_distribution": {

            str(
                int(k)
            ):
                int(v)

            for k, v
            in active_count_distribution.items()
        },
    },


    "v1_clock_integrity": {

        "definition":
            "decision_time_ny 09:45–15:00 before NYSE calendar",

        "clock_bars":
            v1_clock_bars,

        "partial_bars":
            v1_partial_bars,

        "integrity_eligible_bars":
            v1_integrity_eligible_bars,

        "policy":
            (
                "Partial bars are retained as context "
                "but are not decision-eligible."
            ),
    },


    "dataset_condition_impact": {

        "degraded_dates":
            int(
                len(
                    degraded_days
                )
            ),

        "automatic_exclusion":
            False,

        "table":
            str(
                DEGRADED_IMPACT_PATH
            ),
    },


    "launch_liquidity": {

        "automatic_warmup_cut":
            False,

        "table":
            str(
                LAUNCH_LIQUIDITY_PATH
            ),
    },


    "old_15m_cross_check":
        old_15m_check,


    "failures":
        failures,
}


# ------------------------------------------------------------
# 47) Save audit BEFORE hard gate
#
# Even a failure leaves forensic evidence.
# ------------------------------------------------------------

with open(
    CELL5_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell5_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 48) Hard gate
# ------------------------------------------------------------

if failures:

    print(
        "\nCELL 5 FAILURES"
    )

    print(
        "-" * 72
    )


    for failure in failures:

        print(
            " -",
            failure
        )


    raise RuntimeError(
        "\nCELL 5 15m INTEGRITY AUDIT: FAIL\n\n"
        "Audit artifact saved at:\n"
        f"{CELL5_AUDIT_PATH}"
    )


# ------------------------------------------------------------
# 49) Save clean 15m dataset
#
# Only after all structural hard gates pass.
# ------------------------------------------------------------

mes_15m.to_parquet(
    MES_15M_PATH,
    index=True,
)


# ------------------------------------------------------------
# 50) Compact output
#
# Deliberately avoid huge output dumps.
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "CELL 5 — 15m RESAMPLE + DATA INTEGRITY AUDIT"
)

print(
    "=" * 72
)


print(
    "\n[1] RESAMPLE"
)

print(
    "Raw 1m rows             :",
    f"{len(mes_1m):,}"
)

print(
    "15m bins before drop    :",
    f"{bins_before_drop:,}"
)

print(
    "Empty bins dropped      :",
    f"{empty_bins:,}"
)

print(
    "Final 15m bars          :",
    f"{len(mes_15m):,}"
)

print(
    "Full 15m bars           :",
    f"{full_15m_bars:,}"
)

print(
    "Partial 15m bars        :",
    f"{partial_15m_bars:,}"
)


print(
    "\n[2] CONTRACT / DTYPE"
)

print(
    "Roll-crossing bars      :",
    f"{roll_crossing_bars:,}"
)

print(
    "Max instruments / bar   :",
    max_instruments_per_bar
)

print(
    "Raw instrument_id dtype :",
    RAW_INSTRUMENT_DTYPE
)

print(
    "15m instrument_id dtype :",
    clean_instrument_dtype
)

print(
    "Dtype match             :",
    instrument_dtype_match
)


print(
    "\n[3] V1 CLOCK INTEGRITY — BEFORE NYSE CALENDAR"
)

print(
    "09:45–15:00 NY bars     :",
    f"{v1_clock_bars:,}"
)

print(
    "Partial V1-clock bars   :",
    f"{v1_partial_bars:,}"
)

print(
    "Integrity-eligible bars :",
    f"{v1_integrity_eligible_bars:,}"
)

print(
    "Policy                  : "
    "partial bars retained, "
    "but NOT decision-eligible"
)


print(
    "\n[4] ACTIVE 1m COUNT DISTRIBUTION"
)

print(
    active_count_distribution
    .to_string()
)


if v1_partial_bars > 0:

    print(
        "\n[5] FIRST 10 V1 PARTIAL BARS"
    )

    print(
        partial_v1
        .head(
            10
        )
        .to_string()
    )


print(
    "\n[6] DATABENTO DEGRADED DATES ↔ MES"
)

print(
    "Degraded dates          :",
    f"{len(degraded_days):,}"
)

print(
    degraded_days
    .to_string(
        index=False
    )
)


print(
    "\n[7] EARLY MES LIQUIDITY — FIRST 8 NY MONTHS"
)

print(
    launch_monthly
    .head(
        8
    )
    .to_string()
)


print(
    "\n[8] OLD 15m SEMANTIC CROSS-CHECK"
)

print(
    json.dumps(
        old_15m_check,
        indent=2,
        ensure_ascii=False,
    )
)


print(
    "\n[9] SAVED ARTIFACTS"
)

print(
    "Clean 15m parquet       :",
    MES_15M_PATH
)

print(
    "CELL 5 audit            :",
    CELL5_AUDIT_PATH
)

print(
    "V1 partial bars         :",
    V1_PARTIAL_PATH
)

print(
    "Degraded-date impact    :",
    DEGRADED_IMPACT_PATH
)

print(
    "Launch liquidity        :",
    LAUNCH_LIQUIDITY_PATH
)


print(
    "\n"
    + "=" * 72
)

print(
    "CELL 5 15m INTEGRITY AUDIT: PASS"
)

print(
    "=" * 72
)

CELL 5 — Resampling canonical MES 1m → 15m...

CELL 5 — 15m RESAMPLE + DATA INTEGRITY AUDIT

[1] RESAMPLE
Raw 1m rows             : 2,551,123
15m bins before drop    : 253,820
Empty bins dropped      : 83,234
Final 15m bars          : 170,586
Full 15m bars           : 167,030
Partial 15m bars        : 3,556

[2] CONTRACT / DTYPE
Roll-crossing bars      : 0
Max instruments / bar   : 1
Raw instrument_id dtype : uint32
15m instrument_id dtype : uint32
Dtype match             : True

[3] V1 CLOCK INTEGRITY — BEFORE NYSE CALENDAR
09:45–15:00 NY bars     : 40,621
Partial V1-clock bars   : 47
Integrity-eligible bars : 40,574
Policy                  : partial bars retained, but NOT decision-eligible

[4] ACTIVE 1m COUNT DISTRIBUTION
active_1m_count
1         34
2         23
3         15
4         19
5         18
6         19
7         17
8         28
9         58
10        92
11       149
12       288
13       756
14      2040
15    167030

[5] FIRST 10 V1 PARTIAL BARS
                        

In [43]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 6 — RAW GAP ATTRIBUTION AUDIT
# ============================================================
#
# PURPOSE
# -------
# อธิบาย raw timestamp gaps > 1 minute ที่ CELL 3 ตรวจพบ
#
# CELL นี้ทำ 7 เรื่อง:
#
# 1) Bind กับ gap events จาก CELL 3
# 2) แปลงเวลา gap เป็น America/New_York
# 3) ตรวจ clock pattern ของ CME:
#       - 16:15–16:30 ET trading halt
#       - 17:00–18:00 ET daily closed period
# 4) แยก weekend / multi-day / special-session candidates
# 5) Flag overlap กับ Databento degraded dates
# 6) ตรวจ gap ที่กระทบ V1 bar-input window
# 7) ยืนยันว่า 47 V1 partial bars จาก CELL 5
#    สามารถ trace กลับมายัง raw gap events ได้หรือไม่
#
# IMPORTANT
# ---------
# - Audit only
# - ไม่ forward-fill
# - ไม่ impute
# - ไม่ลบ raw rows
# - ไม่ลบ degraded days
# - ไม่กล่าวว่า short gap = data error
# - UNCLASSIFIED ต้องเป็น 0
#   แต่ไม่ได้หมายความว่า causal explanation = 100%
# ============================================================


# ------------------------------------------------------------
# Colab/Jupyter warning guard
#
# Prevent jupyter_client's Python 3.12 utcnow deprecation
# from recursively flooding cell output and making execution
# appear to run forever. This does not suppress pip stderr or
# any pipeline audit failure.
# ------------------------------------------------------------

import warnings as _warnings

_warnings.filterwarnings(
    "ignore",
    message=(
        r"datetime\.datetime\.utcnow\(\) is deprecated.*"
    ),
    category=DeprecationWarning,
    module=r"jupyter_client(\..*)?",
)


# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone, time, timedelta

import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 2) Paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)

DATA_DIR = (
    PROJECT_DIR
    / "Data"
)

CLEAN_DIR = (
    DATA_DIR
    / "MES_Clean_Pipeline_V1"
)


CELL3_AUDIT_PATH = (
    CLEAN_DIR
    / "cell3_supplemental_raw_audit.json"
)

CELL4_AUDIT_PATH = (
    CLEAN_DIR
    / "cell4_dataset_condition_audit.json"
)

CELL5_AUDIT_PATH = (
    CLEAN_DIR
    / "cell5_15m_resample_audit.json"
)

GAP_EVENTS_PATH = (
    CLEAN_DIR
    / "cell3_gap_events.parquet"
)

CONDITION_PATH = (
    CLEAN_DIR
    / "databento_glbx_mdp3_condition_registry.csv"
)

MES_15M_PATH = (
    CLEAN_DIR
    / "MES_2019_2026_15m_clean.parquet"
)


# ------------------------------------------------------------
# CELL 6 outputs
# ------------------------------------------------------------

CELL6_EVENTS_PATH = (
    CLEAN_DIR
    / "cell6_gap_attribution_events.parquet"
)

CELL6_SUMMARY_PATH = (
    CLEAN_DIR
    / "cell6_gap_attribution_summary.csv"
)

CELL6_CLOCK_PATTERNS_PATH = (
    CLEAN_DIR
    / "cell6_gap_clock_patterns.csv"
)

CELL6_V1_GAPS_PATH = (
    CLEAN_DIR
    / "cell6_v1_partial_gap_events.parquet"
)

CELL6_AUDIT_PATH = (
    CLEAN_DIR
    / "cell6_gap_attribution_audit.json"
)


# ------------------------------------------------------------
# 3) Constants
# ------------------------------------------------------------

NY_TZ = "America/New_York"

ONE_MINUTE = pd.Timedelta(
    minutes=1
)

ONE_NS = pd.Timedelta(
    nanoseconds=1
)


# ------------------------------------------------------------
# 4) Upstream artifact gates
# ------------------------------------------------------------

required_paths = [
    CELL3_AUDIT_PATH,
    CELL4_AUDIT_PATH,
    CELL5_AUDIT_PATH,
    GAP_EVENTS_PATH,
    CONDITION_PATH,
    MES_15M_PATH,
]


for path in required_paths:

    if not path.exists():

        raise RuntimeError(
            "CELL 6 STOPPED — missing upstream artifact:\n"
            f"{path}\n\n"
            "Run CELL 0 → CELL 5 first."
        )


# ------------------------------------------------------------
# 5) Load upstream audits
# ------------------------------------------------------------

with open(
    CELL3_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    cell3_audit = json.load(f)


with open(
    CELL4_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    cell4_audit = json.load(f)


with open(
    CELL5_AUDIT_PATH,
    "r",
    encoding="utf-8",
) as f:

    cell5_audit = json.load(f)


# ------------------------------------------------------------
# 6) Upstream status gates
# ------------------------------------------------------------

expected_gap_count = (
    cell3_audit
    .get(
        "raw_gap_audit",
        {},
    )
    .get(
        "gap_gt_1_minute_events"
    )
)


if expected_gap_count is None:

    raise RuntimeError(
        "CELL 6 STOPPED — CELL 3 audit does not expose "
        "gap_gt_1_minute_events."
    )


if (
    cell4_audit.get(
        "registry_status"
    )
    !=
    "AVAILABLE"
):

    raise RuntimeError(
        "CELL 6 STOPPED — CELL 4 condition registry "
        "is not AVAILABLE."
    )


if (
    cell5_audit.get(
        "failures",
        [],
    )
    !=
    []
):

    raise RuntimeError(
        "CELL 6 STOPPED — CELL 5 audit contains failures."
    )


# ------------------------------------------------------------
# 7) Load raw gap-event table
#
# Important:
# This is the event-level artifact created by CELL 3.
# ------------------------------------------------------------

gaps = pd.read_parquet(
    GAP_EVENTS_PATH
)


REQUIRED_GAP_COLUMNS = {
    "prev_timestamp",
    "next_timestamp",
    "gap_minutes",
    "instrument_before",
    "instrument_after",
    "roll_boundary",
}


missing_gap_columns = (
    REQUIRED_GAP_COLUMNS
    -
    set(
        gaps.columns
    )
)


if missing_gap_columns:

    raise RuntimeError(
        "CELL 6 STOPPED — missing gap-event columns:\n"
        + ", ".join(
            sorted(
                missing_gap_columns
            )
        )
    )


# ------------------------------------------------------------
# 8) Normalize timestamps
# ------------------------------------------------------------

for col in [
    "prev_timestamp",
    "next_timestamp",
]:

    gaps[
        col
    ] = pd.to_datetime(
        gaps[
            col
        ],
        utc=True,
        errors="raise",
    )


gaps[
    "gap_minutes"
] = (
    pd.to_numeric(
        gaps[
            "gap_minutes"
        ],
        errors="raise",
    )
    .astype(
        "int64"
    )
)


# ------------------------------------------------------------
# 9) Bind exactly to CELL 3
# ------------------------------------------------------------

if (
    len(
        gaps
    )
    !=
    int(
        expected_gap_count
    )
):

    raise RuntimeError(
        "CELL 6 STOPPED — gap-event count mismatch.\n\n"
        f"Loaded : {len(gaps):,}\n"
        f"CELL 3 : {int(expected_gap_count):,}"
    )


if (
    gaps[
        "gap_minutes"
    ]
    .le(
        1
    )
    .any()
):

    raise RuntimeError(
        "CELL 6 STOPPED — gap_minutes <= 1 "
        "found in gap-event artifact."
    )


if (
    gaps[
        "next_timestamp"
    ]
    <=
    gaps[
        "prev_timestamp"
    ]
).any():

    raise RuntimeError(
        "CELL 6 STOPPED — non-positive gap interval."
    )


duplicate_gap_events = int(
    gaps[
        [
            "prev_timestamp",
            "next_timestamp",
        ]
    ]
    .duplicated()
    .sum()
)


if duplicate_gap_events != 0:

    raise RuntimeError(
        "CELL 6 STOPPED — duplicate gap endpoints: "
        f"{duplicate_gap_events:,}"
    )


# ------------------------------------------------------------
# 10) Verify stored gap_minutes against timestamps
# ------------------------------------------------------------

observed_gap_minutes = (

    (
        gaps[
            "next_timestamp"
        ]
        -
        gaps[
            "prev_timestamp"
        ]
    )

    /
    ONE_MINUTE
)


observed_gap_minutes = (
    observed_gap_minutes
    .astype(
        "int64"
    )
)


if not np.array_equal(
    observed_gap_minutes
    .to_numpy(),

    gaps[
        "gap_minutes"
    ]
    .to_numpy(),
):

    raise RuntimeError(
        "CELL 6 STOPPED — stored gap_minutes "
        "does not match timestamp difference."
    )


# ------------------------------------------------------------
# 11) Missing interval semantics
#
# Example:
#
# Existing:
# 10:00
# 10:02
#
# gap_minutes = 2
#
# Missing raw timestamp:
# 10:01
#
# Therefore:
# missing_minutes = gap_minutes - 1
# ------------------------------------------------------------

gaps[
    "missing_minutes"
] = (
    gaps[
        "gap_minutes"
    ]
    -
    1
)


gaps[
    "missing_start_utc"
] = (
    gaps[
        "prev_timestamp"
    ]
    +
    ONE_MINUTE
)


gaps[
    "missing_end_utc"
] = (
    gaps[
        "next_timestamp"
    ]
)


# ------------------------------------------------------------
# 12) Convert endpoints to New York
#
# DST is handled by timezone conversion.
# ------------------------------------------------------------

gaps[
    "prev_timestamp_ny"
] = (
    gaps[
        "prev_timestamp"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)


gaps[
    "next_timestamp_ny"
] = (
    gaps[
        "next_timestamp"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)


gaps[
    "missing_start_ny"
] = (
    gaps[
        "missing_start_utc"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)


gaps[
    "missing_end_ny"
] = (
    gaps[
        "missing_end_utc"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)


# ------------------------------------------------------------
# 13) Readable clock fields
# ------------------------------------------------------------

gaps[
    "prev_time_ny"
] = (
    gaps[
        "prev_timestamp_ny"
    ]
    .dt
    .strftime(
        "%H:%M"
    )
)


gaps[
    "next_time_ny"
] = (
    gaps[
        "next_timestamp_ny"
    ]
    .dt
    .strftime(
        "%H:%M"
    )
)


gaps[
    "prev_date_ny"
] = (
    gaps[
        "prev_timestamp_ny"
    ]
    .dt
    .strftime(
        "%Y-%m-%d"
    )
)


gaps[
    "next_date_ny"
] = (
    gaps[
        "next_timestamp_ny"
    ]
    .dt
    .strftime(
        "%Y-%m-%d"
    )
)


gaps[
    "same_ny_date"
] = (
    gaps[
        "prev_date_ny"
    ]
    .eq(
        gaps[
            "next_date_ny"
        ]
    )
)


# ------------------------------------------------------------
# 14) Load Databento condition registry
#
# Used only as an independent context flag.
# NOT proof that MES itself is defective.
# ------------------------------------------------------------

condition_df = pd.read_csv(
    CONDITION_PATH
)


condition_df[
    "date"
] = (
    pd.to_datetime(
        condition_df[
            "date"
        ],
        utc=True,
        errors="raise",
    )
    .dt
    .date
)


condition_df[
    "condition"
] = (
    condition_df[
        "condition"
    ]
    .astype(str)
    .str
    .lower()
    .str
    .strip()
)


degraded_utc_dates = set(

    condition_df.loc[
        condition_df[
            "condition"
        ]
        .eq(
            "degraded"
        ),
        "date",
    ]
)


# ------------------------------------------------------------
# 15) Time-overlap helpers
#
# We classify CLOCK PATTERNS.
#
# This is deliberately different from saying:
# "we know the causal reason for every missing trade bar."
# ------------------------------------------------------------

def local_timestamp(
    day,
    hour,
    minute,
):

    return (
        pd.Timestamp(
            datetime.combine(
                day,
                time(
                    hour,
                    minute,
                ),
            )
        )
        .tz_localize(
            NY_TZ
        )
    )


def overlaps_local_window(
    start_utc,
    end_utc,
    start_hm,
    end_hm,
):

    if end_utc <= start_utc:

        return False


    start_local = (
        start_utc
        .tz_convert(
            NY_TZ
        )
    )


    end_local = (
        end_utc
        .tz_convert(
            NY_TZ
        )
    )


    last_local = (
        end_local
        -
        ONE_NS
    )


    day = (
        start_local
        .date()
    )


    last_day = (
        last_local
        .date()
    )


    while day <= last_day:

        window_start = (
            local_timestamp(
                day,
                *start_hm,
            )
        )


        window_end = (
            local_timestamp(
                day,
                *end_hm,
            )
        )


        if (
            start_local
            <
            window_end

            and

            end_local
            >
            window_start
        ):

            return True


        day += timedelta(
            days=1
        )


    return False


def overlaps_local_weekday_window(
    start_utc,
    end_utc,
    start_hm,
    end_hm,
):

    if end_utc <= start_utc:

        return False


    start_local = (
        start_utc
        .tz_convert(
            NY_TZ
        )
    )


    end_local = (
        end_utc
        .tz_convert(
            NY_TZ
        )
    )


    last_local = (
        end_local
        -
        ONE_NS
    )


    day = (
        start_local
        .date()
    )


    last_day = (
        last_local
        .date()
    )


    while day <= last_day:

        # Monday = 0 ... Friday = 4
        if day.weekday() < 5:

            window_start = (
                local_timestamp(
                    day,
                    *start_hm,
                )
            )


            window_end = (
                local_timestamp(
                    day,
                    *end_hm,
                )
            )


            if (
                start_local
                <
                window_end

                and

                end_local
                >
                window_start
            ):

                return True


        day += timedelta(
            days=1
        )


    return False


def touches_weekend_local(
    start_utc,
    end_utc,
):

    if end_utc <= start_utc:

        return False


    start_local = (
        start_utc
        .tz_convert(
            NY_TZ
        )
    )


    last_local = (
        end_utc
        .tz_convert(
            NY_TZ
        )
        -
        ONE_NS
    )


    day = (
        start_local
        .date()
    )


    last_day = (
        last_local
        .date()
    )


    while day <= last_day:

        if day.weekday() >= 5:

            return True


        day += timedelta(
            days=1
        )


    return False


def overlaps_degraded_utc_date(
    start_utc,
    end_utc,
):

    if end_utc <= start_utc:

        return False


    day = (
        start_utc
        .date()
    )


    last_day = (
        (
            end_utc
            -
            ONE_NS
        )
        .date()
    )


    while day <= last_day:

        if day in degraded_utc_dates:

            return True


        day += timedelta(
            days=1
        )


    return False


# ------------------------------------------------------------
# 16) Evaluate overlap flags
#
# CME clock references:
#
# 16:15–16:30 ET
#     scheduled intraday halt
#
# 17:00–18:00 ET
#     regular daily closed period
#
# V1 bar-input interval:
#
# 09:30–15:00 ET
#
# Why 09:30?
# A decision at 09:45 uses the 09:30–09:44 bar.
# ------------------------------------------------------------

gap_intervals = list(
    zip(
        gaps[
            "missing_start_utc"
        ],
        gaps[
            "missing_end_utc"
        ],
    )
)


gaps[
    "cme_1615_1630_overlap"
] = [

    overlaps_local_window(
        start,
        end,
        (16, 15),
        (16, 30),
    )

    for start, end
    in gap_intervals
]


gaps[
    "cme_1700_1800_overlap"
] = [

    overlaps_local_window(
        start,
        end,
        (17, 0),
        (18, 0),
    )

    for start, end
    in gap_intervals
]


gaps[
    "v1_bar_input_overlap"
] = [

    overlaps_local_weekday_window(
        start,
        end,
        (9, 30),
        (15, 0),
    )

    for start, end
    in gap_intervals
]


gaps[
    "touches_weekend_ny"
] = [

    touches_weekend_local(
        start,
        end,
    )

    for start, end
    in gap_intervals
]


gaps[
    "degraded_utc_date_overlap"
] = [

    overlaps_degraded_utc_date(
        start,
        end,
    )

    for start, end
    in gap_intervals
]


# ------------------------------------------------------------
# 17) Exact New York clock patterns
# ------------------------------------------------------------

prev_minute_ny = (

    gaps[
        "prev_timestamp_ny"
    ]
    .dt.hour
    *
    60

    +

    gaps[
        "prev_timestamp_ny"
    ]
    .dt.minute
)


next_minute_ny = (

    gaps[
        "next_timestamp_ny"
    ]
    .dt.hour
    *
    60

    +

    gaps[
        "next_timestamp_ny"
    ]
    .dt.minute
)


# ------------------------------------------------------------
# Exact CME halt:
#
# 16:14 raw bar exists
# 16:15–16:29 absent
# 16:30 raw bar exists
#
# Timestamp difference = 16 minutes
# ------------------------------------------------------------

gaps[
    "exact_cme_halt_pattern"
] = (

    gaps[
        "same_ny_date"
    ]

    &

    gaps[
        "gap_minutes"
    ]
    .eq(
        16
    )

    &

    prev_minute_ny
    .eq(
        16 * 60
        + 14
    )

    &

    next_minute_ny
    .eq(
        16 * 60
        + 30
    )
)


# ------------------------------------------------------------
# Exact regular daily close/reopen:
#
# 16:59 raw bar exists
# next raw bar = 18:00
#
# Timestamp difference = 61 minutes
# ------------------------------------------------------------

gaps[
    "exact_daily_closed_period_pattern"
] = (

    gaps[
        "same_ny_date"
    ]

    &

    gaps[
        "gap_minutes"
    ]
    .eq(
        61
    )

    &

    prev_minute_ny
    .eq(
        16 * 60
        + 59
    )

    &

    next_minute_ny
    .eq(
        18 * 60
    )
)


# ------------------------------------------------------------
# Longer same-day closure that reopens at 18:00
#
# Examples might include early/special closes.
#
# Candidate only:
# We do NOT label the holiday cause automatically.
# ------------------------------------------------------------

gaps[
    "special_close_to_1800_candidate"
] = (

    gaps[
        "same_ny_date"
    ]

    &

    gaps[
        "gap_minutes"
    ]
    .gt(
        61
    )

    &

    gaps[
        "gap_minutes"
    ]
    .lt(
        24 * 60
    )

    &

    next_minute_ny
    .eq(
        18 * 60
    )
)


# ------------------------------------------------------------
# 18) Primary attribution taxonomy
#
# IMPORTANT:
#
# Every event gets ONE primary category.
#
# Some categories are high-confidence schedule matches.
# Others are deliberately called "candidate".
#
# Therefore:
#
# UNCLASSIFIED = 0
#
# does NOT mean:
#
# causal uncertainty = 0
# ------------------------------------------------------------

gaps[
    "primary_attribution"
] = (
    "UNCLASSIFIED"
)


# ------------------------------------------------------------
# Priority 1:
# Weekend / multi-day closure
# ------------------------------------------------------------

multiday_closure_candidate = (

    gaps[
        "gap_minutes"
    ]
    .ge(
        24 * 60
    )
)


gaps.loc[
    multiday_closure_candidate,
    "primary_attribution",
] = (
    "WEEKEND_OR_MULTIDAY_CLOSURE_CANDIDATE"
)



# ------------------------------------------------------------
# Priority 2:
# Exact daily 17:00 → 18:00 closed period
# ------------------------------------------------------------

gaps.loc[
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    )

    &

    gaps[
        "exact_daily_closed_period_pattern"
    ],

    "primary_attribution",
] = (
    "CME_DAILY_CLOSED_PERIOD_EXACT"
)


# ------------------------------------------------------------
# Priority 3:
# Longer close that reopens at 18:00
# ------------------------------------------------------------

gaps.loc[
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    )

    &

    gaps[
        "special_close_to_1800_candidate"
    ],

    "primary_attribution",
] = (
    "SPECIAL_SESSION_CLOSE_TO_1800_CANDIDATE"
)


# ------------------------------------------------------------
# Priority 4:
# Exact 16:15 → 16:30 halt
# ------------------------------------------------------------

gaps.loc[
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    )

    &

    gaps[
        "exact_cme_halt_pattern"
    ],

    "primary_attribution",
] = (
    "CME_1615_1630_HALT_EXACT"
)


# ------------------------------------------------------------
# Priority 5:
# Gap overlaps halt but is longer than exact halt
#
# Example:
# no trade immediately before/after the scheduled halt.
# ------------------------------------------------------------

gaps.loc[
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    )

    &

    gaps[
        "cme_1615_1630_overlap"
    ],

    "primary_attribution",
] = (
    "CME_HALT_PLUS_ADJACENT_GAP_CANDIDATE"
)


# ------------------------------------------------------------
# Priority 6:
# Other > 61-minute intraday gap
# ------------------------------------------------------------

gaps.loc[
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    )

    &

    gaps[
        "gap_minutes"
    ]
    .gt(
        61
    ),

    "primary_attribution",
] = (
    "OTHER_INTRADAY_OR_SPECIAL_CLOSURE_CANDIDATE"
)


# ------------------------------------------------------------
# Priority 7:
# Remaining short gaps
#
# Because Databento OHLCV is trade-based,
# OHLCV alone cannot prove whether these are:
#
# - no-trade minutes
# - interruption
# - data-quality issue
#
# So we deliberately do NOT call them "errors".
# ------------------------------------------------------------

gaps.loc[
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    ),

    "primary_attribution",
] = (
    "SHORT_NO_TRADE_OR_DATA_GAP_CANDIDATE"
)


# ------------------------------------------------------------
# 19) Load clean 15m integrity artifact
#
# Main goal:
# connect raw gaps back to the V1 partial bars from CELL 5.
# ------------------------------------------------------------

mes_15m = pd.read_parquet(
    MES_15M_PATH
)


required_15m_columns = {
    "v1_clock_partial",
    "bar_complete_15m",
}


if not required_15m_columns.issubset(
    mes_15m.columns
):

    raise RuntimeError(
        "CELL 6 STOPPED — clean 15m file "
        "does not contain CELL 5 integrity columns."
    )


# ------------------------------------------------------------
# 20) Extract V1 partial 15m bars
# ------------------------------------------------------------

v1_partial_starts = (
    mes_15m.index[
        mes_15m[
            "v1_clock_partial"
        ]
        .astype(
            bool
        )
    ]
)


v1_partial_start_ns = (
    v1_partial_starts
    .asi8
)


v1_partial_end_ns = (

    v1_partial_start_ns
    +
    pd.Timedelta(
        minutes=15
    )
    .value
)


# ------------------------------------------------------------
# 21) Trace every raw gap to V1 partial bars
#
# There are only a small number of V1 partial bars,
# so this check is intentionally simple and auditable.
# ------------------------------------------------------------

explained_partial_mask = np.zeros(
    len(
        v1_partial_starts
    ),
    dtype=bool,
)


gap_hits_v1_partial = np.zeros(
    len(
        gaps
    ),
    dtype=bool,
)


for i, (
    missing_start,
    missing_end,
) in enumerate(
    zip(
        gaps[
            "missing_start_utc"
        ],
        gaps[
            "missing_end_utc"
        ],
    )
):

    start_ns = (
        missing_start.value
    )


    end_ns = (
        missing_end.value
    )


    overlap = (

        v1_partial_start_ns
        <
        end_ns

        &

        (
            v1_partial_end_ns
            >
            start_ns
        )
    )


    # Keep explicit parentheses for numpy boolean logic
    overlap = (

        (
            v1_partial_start_ns
            <
            end_ns
        )

        &

        (
            v1_partial_end_ns
            >
            start_ns
        )
    )


    if overlap.any():

        gap_hits_v1_partial[
            i
        ] = True


        explained_partial_mask |= overlap


gaps[
    "impacts_v1_partial_bar"
] = (
    gap_hits_v1_partial
)


# ------------------------------------------------------------
# 22) Cross-check against CELL 5
# ------------------------------------------------------------

expected_v1_partial = (

    cell5_audit
    .get(
        "v1_clock_integrity",
        {},
    )
    .get(
        "partial_bars"
    )
)


observed_v1_partial = int(
    len(
        v1_partial_starts
    )
)


explained_v1_partial = int(
    explained_partial_mask
    .sum()
)


unexplained_v1_partial = int(
    observed_v1_partial
    -
    explained_v1_partial
)


# ------------------------------------------------------------
# 23) Hard audit failures
# ------------------------------------------------------------

failures = []


if expected_v1_partial is None:

    failures.append(
        "CELL 5 audit missing V1 partial-bar count"
    )


elif (
    observed_v1_partial
    !=
    int(
        expected_v1_partial
    )
):

    failures.append(
        "Clean 15m V1 partial count mismatch: "
        f"{observed_v1_partial} "
        f"!= CELL 5 {int(expected_v1_partial)}"
    )


# ------------------------------------------------------------
# Strong traceability gate:
#
# Every V1 partial bar must trace to at least one raw gap.
# ------------------------------------------------------------

if unexplained_v1_partial != 0:

    failures.append(
        "V1 partial bars not linked to raw gap events: "
        f"{unexplained_v1_partial}"
    )


# ------------------------------------------------------------
# Every gap must belong to the taxonomy.
# ------------------------------------------------------------

unclassified_count = int(
    gaps[
        "primary_attribution"
    ]
    .eq(
        "UNCLASSIFIED"
    )
    .sum()
)


if unclassified_count != 0:

    failures.append(
        "Primary gap attribution has "
        f"{unclassified_count} UNCLASSIFIED events"
    )


short_gap_misclassified_as_multiday = int(
    (
        gaps[
            "primary_attribution"
        ]
        .eq(
            "WEEKEND_OR_MULTIDAY_CLOSURE_CANDIDATE"
        )

        &

        gaps[
            "gap_minutes"
        ]
        .lt(
            24 * 60
        )
    )
    .sum()
)


if short_gap_misclassified_as_multiday != 0:

    failures.append(
        "Short gaps incorrectly classified "
        "as multiday closure: "
        f"{short_gap_misclassified_as_multiday}"
    )


# ------------------------------------------------------------
# 24) Attribution summary
# ------------------------------------------------------------

summary = (

    gaps

    .groupby(
        "primary_attribution",
        dropna=False,
    )

    .agg(
        events=(
            "gap_minutes",
            "size",
        ),

        missing_minutes=(
            "missing_minutes",
            "sum",
        ),

        median_gap_minutes=(
            "gap_minutes",
            "median",
        ),

        max_gap_minutes=(
            "gap_minutes",
            "max",
        ),

        degraded_overlap_events=(
            "degraded_utc_date_overlap",
            "sum",
        ),

        v1_input_overlap_events=(
            "v1_bar_input_overlap",
            "sum",
        ),

        v1_partial_impact_events=(
            "impacts_v1_partial_bar",
            "sum",
        ),

        roll_boundary_events=(
            "roll_boundary",
            "sum",
        ),
    )

    .sort_values(
        [
            "events",
            "missing_minutes",
        ],
        ascending=[
            False,
            False,
        ],
    )

    .reset_index()
)


summary[
    "pct_of_gap_events"
] = (

    summary[
        "events"
    ]

    /
    len(
        gaps
    )

    *
    100.0
)


if (
    int(
        summary[
            "events"
        ]
        .sum()
    )
    !=
    len(
        gaps
    )
):

    failures.append(
        "Attribution summary does not sum "
        "to total gap events"
    )


# ------------------------------------------------------------
# 25) New York clock-pattern table
#
# This table is especially useful for explaining:
#
# 16-minute
# 61-minute
# 286-minute
# 301-minute
# etc.
# ------------------------------------------------------------

clock_patterns = (

    gaps

    .groupby(
        [
            "gap_minutes",
            "prev_time_ny",
            "next_time_ny",
            "primary_attribution",
        ],
        dropna=False,
    )

    .size()

    .rename(
        "count"
    )

    .reset_index()

    .sort_values(
        [
            "count",
            "gap_minutes",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


# ------------------------------------------------------------
# 26) Save event-level artifacts
# ------------------------------------------------------------

gaps.to_parquet(
    CELL6_EVENTS_PATH,
    index=False,
)


summary.to_csv(
    CELL6_SUMMARY_PATH,
    index=False,
)


clock_patterns.to_csv(
    CELL6_CLOCK_PATTERNS_PATH,
    index=False,
)


gaps.loc[
    gaps[
        "impacts_v1_partial_bar"
    ]
].to_parquet(
    CELL6_V1_GAPS_PATH,
    index=False,
)


# ------------------------------------------------------------
# 27) Core result counts
# ------------------------------------------------------------

exact_halt_count = int(
    gaps[
        "exact_cme_halt_pattern"
    ]
    .sum()
)


exact_daily_closed_count = int(
    gaps[
        "exact_daily_closed_period_pattern"
    ]
    .sum()
)


special_close_1800_count = int(
    gaps[
        "special_close_to_1800_candidate"
    ]
    .sum()
)


degraded_overlap_count = int(
    gaps[
        "degraded_utc_date_overlap"
    ]
    .sum()
)


v1_input_overlap_count = int(
    gaps[
        "v1_bar_input_overlap"
    ]
    .sum()
)


roll_gap_count = int(
    gaps[
        "roll_boundary"
    ]
    .sum()
)


# ------------------------------------------------------------
# 28) Build forensic audit
# ------------------------------------------------------------

cell6_audit = {

    "audit_written_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),


    "input_binding": {

        "cell3_expected_gap_events":
            int(
                expected_gap_count
            ),

        "loaded_gap_events":
            int(
                len(
                    gaps
                )
            ),

        "cell5_expected_v1_partial_bars":
            (
                None
                if expected_v1_partial is None
                else int(
                    expected_v1_partial
                )
            ),

        "observed_v1_partial_bars":
            observed_v1_partial,
    },


    "schedule_clock_basis": {

        "timezone":
            NY_TZ,

        "regular_globex_hours_reference":
            "Sunday-Friday 18:00-17:00 ET",

        "intraday_halt_reference":
            "16:15-16:30 ET",

        "regular_daily_closed_period_reference":
            "17:00-18:00 ET",

        "classification_note":
            (
                "Exact clock matches are strong schedule "
                "attributions. Extended and short-gap "
                "categories are candidates, not causal proof "
                "from trade-based OHLCV alone."
            ),
    },


    "results": {

        "total_gap_events":
            int(
                len(
                    gaps
                )
            ),

        "total_missing_minutes":
            int(
                gaps[
                    "missing_minutes"
                ]
                .sum()
            ),

        "exact_cme_halt_events":
            exact_halt_count,

        "exact_daily_closed_period_events":
            exact_daily_closed_count,

        "special_close_to_1800_candidates":
            special_close_1800_count,

        "degraded_utc_date_overlap_events":
            degraded_overlap_count,

        "v1_bar_input_overlap_events":
            v1_input_overlap_count,

        "raw_gap_events_impacting_v1_partial_bars":
            int(
                gaps[
                    "impacts_v1_partial_bar"
                ]
                .sum()
            ),

        "v1_partial_bars_explained_by_raw_gaps":
            explained_v1_partial,

        "v1_partial_bars_unexplained_by_raw_gaps":
            unexplained_v1_partial,

        "roll_boundary_gap_events":
            roll_gap_count,

        "primary_unclassified_events":
            unclassified_count,
    },


    "policy": {

        "automatic_raw_row_deletion":
            False,

        "automatic_gap_imputation":
            False,

        "automatic_degraded_day_exclusion":
            False,

        "short_gap_causal_claim":
            False,
    },


    "artifacts": {

        "events":
            str(
                CELL6_EVENTS_PATH
            ),

        "summary":
            str(
                CELL6_SUMMARY_PATH
            ),

        "clock_patterns":
            str(
                CELL6_CLOCK_PATTERNS_PATH
            ),

        "v1_partial_gap_events":
            str(
                CELL6_V1_GAPS_PATH
            ),
    },


    "failures":
        failures,
}


# ------------------------------------------------------------
# 29) Save audit BEFORE hard gate
# ------------------------------------------------------------

with open(
    CELL6_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell6_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 30) Final hard gate
# ------------------------------------------------------------

if failures:

    print(
        "\nCELL 6 FAILURES"
    )

    print(
        "-" * 72
    )


    for failure in failures:

        print(
            " -",
            failure
        )


    raise RuntimeError(
        "\nCELL 6 GAP ATTRIBUTION AUDIT: FAIL\n"
        f"{CELL6_AUDIT_PATH}"
    )


# ------------------------------------------------------------
# 31) Compact output
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "CELL 6 — RAW GAP ATTRIBUTION AUDIT"
)

print(
    "=" * 72
)


print(
    "\n[1] INPUT BINDING"
)

print(
    "Gap events from CELL 3          :",
    f"{len(gaps):,}"
)

print(
    "V1 partial bars from CELL 5     :",
    f"{observed_v1_partial:,}"
)

print(
    "V1 partial bars explained       :",
    f"{explained_v1_partial:,}"
)

print(
    "V1 partial bars unexplained     :",
    f"{unexplained_v1_partial:,}"
)


print(
    "\n[2] HIGH-CONFIDENCE CLOCK PATTERNS"
)

print(
    "Exact 16:15–16:30 halt events   :",
    f"{exact_halt_count:,}"
)

print(
    "Exact 17:00–18:00 closed period :",
    f"{exact_daily_closed_count:,}"
)

print(
    "Special close → 18:00 candidate :",
    f"{special_close_1800_count:,}"
)


print(
    "\n[3] CONTEXT FLAGS"
)

print(
    "Degraded-date overlap events    :",
    f"{degraded_overlap_count:,}"
)

print(
    "V1 bar-input overlap events     :",
    f"{v1_input_overlap_count:,}"
)

print(
    "Roll-boundary gap events        :",
    f"{roll_gap_count:,}"
)


print(
    "\n[4] PRIMARY ATTRIBUTION SUMMARY"
)

print(
    summary
    .to_string(
        index=False
    )
)


print(
    "\n[5] TOP 20 NEW YORK CLOCK PATTERNS"
)

print(
    clock_patterns
    .head(
        20
    )
    .to_string(
        index=False
    )
)


print(
    "\n[6] POLICY"
)

print(
    "UNCLASSIFIED events             :",
    unclassified_count
)

print(
    "Automatic deletion              : False"
)

print(
    "Automatic imputation            : False"
)

print(
    "Short-gap causal claim          : False"
)


print(
    "\n[7] SAVED ARTIFACTS"
)

print(
    "Event-level attribution         :",
    CELL6_EVENTS_PATH
)

print(
    "Attribution summary             :",
    CELL6_SUMMARY_PATH
)

print(
    "Clock patterns                  :",
    CELL6_CLOCK_PATTERNS_PATH
)

print(
    "V1-impacting gap events         :",
    CELL6_V1_GAPS_PATH
)

print(
    "CELL 6 audit                    :",
    CELL6_AUDIT_PATH
)


print(
    "\n"
    + "=" * 72
)

print(
    "CELL 6 GAP ATTRIBUTION AUDIT: PASS"
)

print(
    "=" * 72
)


CELL 6 — RAW GAP ATTRIBUTION AUDIT

[1] INPUT BINDING
Gap events from CELL 3          : 7,931
V1 partial bars from CELL 5     : 47
V1 partial bars explained       : 47
V1 partial bars unexplained     : 0

[2] HIGH-CONFIDENCE CLOCK PATTERNS
Exact 16:15–16:30 halt events   : 531
Exact 17:00–18:00 closed period : 1,436
Special close → 18:00 candidate : 50

[3] CONTEXT FLAGS
Degraded-date overlap events    : 26
V1 bar-input overlap events     : 134
Roll-boundary gap events        : 2

[4] PRIMARY ATTRIBUTION SUMMARY
                        primary_attribution  events  missing_minutes  median_gap_minutes  max_gap_minutes  degraded_overlap_events  v1_input_overlap_events  v1_partial_impact_events  roll_boundary_events  pct_of_gap_events
       SHORT_NO_TRADE_OR_DATA_GAP_CANDIDATE    5522             7931                 2.0               60                        4                       53                        53                     2          69.625520
              CME_DAILY_CLOSED_PERI

In [44]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 7 — POINT-IN-TIME DECISION UNIVERSE
# ============================================================
#
# PURPOSE
# -------
# Freeze the timestamps at which MES V1 is allowed to make an
# entry decision before any feature, label, or model is built.
#
# POLICY
# ------
# 1) Decision time is the end of a completed 15-minute input bar.
# 2) Base clock window is 09:45–15:00 America/New_York.
# 3) NYSE regular sessions are a research/entry policy filter.
#    They are NOT treated as the authority for CME tradability.
# 4) A +60 minute research horizon must fit before the scheduled
#    NYSE close. On a normal 16:00 close the last decision is 15:00;
#    on a 13:00 early close the last decision is 12:00.
# 5) Partial or mixed-contract input bars remain in the ledger as
#    context but cannot become decision observations.
# 6) Databento degraded-date metadata is retained as a context flag;
#    it is NOT an automatic exclusion rule.
# 7) No future return or future-bar quality is used to decide whether
#    a timestamp belongs to the live point-in-time universe.
# ============================================================


# ------------------------------------------------------------
# Colab/Jupyter warning guard
# ------------------------------------------------------------

import warnings as _warnings

_warnings.filterwarnings(
    "ignore",
    message=(
        r"datetime\.datetime\.utcnow\(\) is deprecated.*"
    ),
    category=DeprecationWarning,
)


# ------------------------------------------------------------
# 1) Imports
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal


# ------------------------------------------------------------
# 2) Paths
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant_Lab"
)

DATA_DIR = (
    PROJECT_DIR
    / "Data"
)

CLEAN_DIR = (
    DATA_DIR
    / "MES_Clean_Pipeline_V1"
)


MES_15M_PATH = (
    CLEAN_DIR
    / "MES_2019_2026_15m_clean.parquet"
)

CELL4_AUDIT_PATH = (
    CLEAN_DIR
    / "cell4_dataset_condition_audit.json"
)

CELL5_AUDIT_PATH = (
    CLEAN_DIR
    / "cell5_15m_resample_audit.json"
)

CELL6_AUDIT_PATH = (
    CLEAN_DIR
    / "cell6_gap_attribution_audit.json"
)


# ------------------------------------------------------------
# CELL 7 outputs
# ------------------------------------------------------------

CELL7_UNIVERSE_PATH = (
    CLEAN_DIR
    / "cell7_decision_universe_v1.parquet"
)

CELL7_LEDGER_PATH = (
    CLEAN_DIR
    / "cell7_decision_universe_ledger.parquet"
)

CELL7_DAILY_PATH = (
    CLEAN_DIR
    / "cell7_decision_universe_daily_summary.csv"
)

CELL7_AUDIT_PATH = (
    CLEAN_DIR
    / "cell7_decision_universe_audit.json"
)


# ------------------------------------------------------------
# 3) Frozen V1 policy constants
# ------------------------------------------------------------

POLICY_VERSION = "MES_V1_DECISION_UNIVERSE_1.0"

NY_TZ = "America/New_York"
CALENDAR_NAME = "NYSE"

BAR_MINUTES = 15
LABEL_HORIZON_MINUTES = 60

V1_START_MINUTE = (
    9 * 60
    + 45
)

V1_END_MINUTE = (
    15 * 60
)


# ------------------------------------------------------------
# 4) Helpers
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# 5) Upstream artifact gates
# ------------------------------------------------------------

required_paths = [
    MES_15M_PATH,
    CELL4_AUDIT_PATH,
    CELL5_AUDIT_PATH,
    CELL6_AUDIT_PATH,
]


for path in required_paths:

    if not path.exists():

        raise RuntimeError(
            "CELL 7 STOPPED — missing upstream artifact:\n"
            f"{path}\n\n"
            "Run CELL 0 → CELL 6 first."
        )


cell4_audit = load_json(
    CELL4_AUDIT_PATH
)

cell5_audit = load_json(
    CELL5_AUDIT_PATH
)

cell6_audit = load_json(
    CELL6_AUDIT_PATH
)


if (
    cell4_audit.get(
        "registry_status"
    )
    !=
    "AVAILABLE"
):

    raise RuntimeError(
        "CELL 7 STOPPED — CELL 4 condition registry "
        "is not AVAILABLE."
    )


if cell5_audit.get(
    "failures",
    [],
):

    raise RuntimeError(
        "CELL 7 STOPPED — CELL 5 audit contains failures."
    )


if cell6_audit.get(
    "failures",
    [],
):

    raise RuntimeError(
        "CELL 7 STOPPED — CELL 6 audit contains failures."
    )


if (
    cell6_audit
    .get(
        "results",
        {},
    )
    .get(
        "primary_unclassified_events"
    )
    !=
    0
):

    raise RuntimeError(
        "CELL 7 STOPPED — CELL 6 still has "
        "unclassified gap events."
    )


# ------------------------------------------------------------
# 6) Load the clean 15-minute dataset
# ------------------------------------------------------------

mes_15m = pd.read_parquet(
    MES_15M_PATH
)


REQUIRED_COLUMNS = {
    "open",
    "high",
    "low",
    "close",
    "volume",
    "active_1m_count",
    "instrument_id",
    "instrument_count",
    "crosses_roll",
    "decision_time",
    "decision_time_ny",
    "bar_complete_15m",
    "data_integrity_ok",
    "v1_clock_window",
    "v1_clock_partial",
    "v1_clock_integrity_eligible",
    "dataset_condition_utc",
    "dataset_degraded_utc",
}


missing_columns = (
    REQUIRED_COLUMNS
    -
    set(
        mes_15m.columns
    )
)


if missing_columns:

    raise RuntimeError(
        "CELL 7 STOPPED — missing CELL 5 columns:\n"
        + ", ".join(
            sorted(
                missing_columns
            )
        )
    )


if not isinstance(
    mes_15m.index,
    pd.DatetimeIndex,
):

    raise RuntimeError(
        "CELL 7 STOPPED — 15m index is not DatetimeIndex."
    )


if mes_15m.index.tz is None:

    raise RuntimeError(
        "CELL 7 STOPPED — 15m index is timezone-naive."
    )


mes_15m = (
    mes_15m
    .sort_index()
    .copy()
)

mes_15m.index = (
    mes_15m.index
    .tz_convert(
        "UTC"
    )
)

mes_15m.index.name = (
    "bar_start_utc"
)


# ------------------------------------------------------------
# 7) Structural input gates
# ------------------------------------------------------------

if not mes_15m.index.is_monotonic_increasing:

    raise RuntimeError(
        "CELL 7 STOPPED — 15m index is not monotonic."
    )


duplicate_input_timestamps = int(
    mes_15m.index
    .duplicated()
    .sum()
)


if duplicate_input_timestamps != 0:

    raise RuntimeError(
        "CELL 7 STOPPED — duplicate 15m timestamps: "
        f"{duplicate_input_timestamps:,}"
    )


# ------------------------------------------------------------
# 8) Recompute point-in-time fields from the frozen index
# ------------------------------------------------------------

stored_decision_time = pd.to_datetime(
    mes_15m[
        "decision_time"
    ],
    utc=True,
    errors="raise",
)

expected_decision_time = (
    mes_15m.index
    +
    pd.Timedelta(
        minutes=BAR_MINUTES
    )
)


if not np.array_equal(
    stored_decision_time.array.asi8,
    expected_decision_time.asi8,
):

    raise RuntimeError(
        "CELL 7 STOPPED — decision_time does not equal "
        "bar_start_utc + 15 minutes."
    )


mes_15m[
    "decision_time"
] = expected_decision_time

mes_15m[
    "decision_time_ny"
] = (
    mes_15m[
        "decision_time"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)


decision_minute_ny = (
    mes_15m[
        "decision_time_ny"
    ]
    .dt.hour
    *
    60
    +
    mes_15m[
        "decision_time_ny"
    ]
    .dt.minute
)


recomputed_clock_window = (
    decision_minute_ny
    .between(
        V1_START_MINUTE,
        V1_END_MINUTE,
        inclusive="both",
    )
)


if not np.array_equal(
    recomputed_clock_window.to_numpy(
        dtype=bool
    ),
    mes_15m[
        "v1_clock_window"
    ]
    .astype(bool)
    .to_numpy(),
):

    raise RuntimeError(
        "CELL 7 STOPPED — stored v1_clock_window does not "
        "match the frozen 09:45–15:00 policy."
    )


recomputed_integrity_ok = (
    mes_15m[
        "active_1m_count"
    ]
    .eq(
        BAR_MINUTES
    )
    &
    ~mes_15m[
        "crosses_roll"
    ]
    .astype(bool)
)


if not np.array_equal(
    recomputed_integrity_ok.to_numpy(
        dtype=bool
    ),
    mes_15m[
        "data_integrity_ok"
    ]
    .astype(bool)
    .to_numpy(),
):

    raise RuntimeError(
        "CELL 7 STOPPED — stored data_integrity_ok does not "
        "match complete-bar / no-cross-roll policy."
    )


# ------------------------------------------------------------
# 9) Build the NYSE policy calendar
#
# The calendar is a research-session policy filter only.
# It is not a claim about whether MES traded on CME.
# ------------------------------------------------------------

first_ny_date = (
    mes_15m[
        "decision_time_ny"
    ]
    .min()
    .date()
)

last_ny_date = (
    mes_15m[
        "decision_time_ny"
    ]
    .max()
    .date()
)


nyse = mcal.get_calendar(
    CALENDAR_NAME
)

nyse_schedule = nyse.schedule(
    start_date=first_ny_date,
    end_date=last_ny_date,
)


for col in [
    "market_open",
    "market_close",
]:

    nyse_schedule[
        col
    ] = pd.to_datetime(
        nyse_schedule[
            col
        ],
        utc=True,
        errors="raise",
    )


nyse_schedule[
    "nyse_session_date"
] = (
    nyse_schedule.index.date
)


market_open_map = dict(
    zip(
        nyse_schedule[
            "nyse_session_date"
        ],
        nyse_schedule[
            "market_open"
        ],
    )
)

market_close_map = dict(
    zip(
        nyse_schedule[
            "nyse_session_date"
        ],
        nyse_schedule[
            "market_close"
        ],
    )
)


schedule_close_ny = (
    nyse_schedule[
        "market_close"
    ]
    .dt
    .tz_convert(
        NY_TZ
    )
)

schedule_close_minute_ny = (
    schedule_close_ny.dt.hour
    *
    60
    +
    schedule_close_ny.dt.minute
)


early_close_dates = set(
    nyse_schedule.loc[
        schedule_close_minute_ny
        <
        16 * 60,
        "nyse_session_date",
    ]
)


# ------------------------------------------------------------
# 10) Build the full clock-candidate ledger
# ------------------------------------------------------------

ledger_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "active_1m_count",
    "instrument_id",
    "instrument_count",
    "crosses_roll",
    "decision_time",
    "decision_time_ny",
    "bar_complete_15m",
    "data_integrity_ok",
    "v1_clock_window",
    "v1_clock_partial",
    "dataset_condition_utc",
    "dataset_degraded_utc",
]


ledger = (
    mes_15m.loc[
        recomputed_clock_window,
        ledger_columns,
    ]
    .copy()
)

ledger.insert(
    0,
    "bar_start_utc",
    ledger.index,
)

ledger.reset_index(
    drop=True,
    inplace=True,
)


ledger[
    "nyse_session_date"
] = (
    ledger[
        "decision_time_ny"
    ]
    .dt.date
)

ledger[
    "nyse_market_open_utc"
] = pd.to_datetime(
    ledger[
        "nyse_session_date"
    ]
    .map(
        market_open_map
    ),
    utc=True,
)

ledger[
    "nyse_market_close_utc"
] = pd.to_datetime(
    ledger[
        "nyse_session_date"
    ]
    .map(
        market_close_map
    ),
    utc=True,
)

ledger[
    "nyse_policy_session"
] = (
    ledger[
        "nyse_market_open_utc"
    ]
    .notna()
    &
    ledger[
        "nyse_market_close_utc"
    ]
    .notna()
)

ledger[
    "policy_first_decision_utc"
] = (
    ledger[
        "nyse_market_open_utc"
    ]
    +
    pd.Timedelta(
        minutes=BAR_MINUTES
    )
)

ledger[
    "policy_last_decision_utc"
] = (
    ledger[
        "nyse_market_close_utc"
    ]
    -
    pd.Timedelta(
        minutes=LABEL_HORIZON_MINUTES
    )
)

ledger[
    "early_close_session"
] = (
    ledger[
        "nyse_session_date"
    ]
    .isin(
        early_close_dates
    )
)

ledger[
    "within_nyse_entry_policy"
] = (
    ledger[
        "nyse_policy_session"
    ]
    &
    ledger[
        "decision_time"
    ]
    .ge(
        ledger[
            "policy_first_decision_utc"
        ]
    )
    &
    ledger[
        "decision_time"
    ]
    .le(
        ledger[
            "policy_last_decision_utc"
        ]
    )
)

ledger[
    "input_bar_integrity_ok"
] = (
    ledger[
        "bar_complete_15m"
    ]
    .astype(bool)
    &
    ~ledger[
        "crosses_roll"
    ]
    .astype(bool)
)


# Degraded metadata is intentionally NOT in this expression.
ledger[
    "decision_eligible"
] = (
    ledger[
        "within_nyse_entry_policy"
    ]
    &
    ledger[
        "input_bar_integrity_ok"
    ]
)


# ------------------------------------------------------------
# 11) One primary reason per clock candidate
# ------------------------------------------------------------

ledger[
    "primary_exclusion_reason"
] = (
    "ELIGIBLE"
)


no_policy_session = (
    ~ledger[
        "nyse_policy_session"
    ]
)

before_policy_start = (
    ledger[
        "nyse_policy_session"
    ]
    &
    ledger[
        "decision_time"
    ]
    .lt(
        ledger[
            "policy_first_decision_utc"
        ]
    )
)

after_horizon_safe_close = (
    ledger[
        "nyse_policy_session"
    ]
    &
    ledger[
        "decision_time"
    ]
    .gt(
        ledger[
            "policy_last_decision_utc"
        ]
    )
)

input_bar_partial = (
    ~ledger[
        "bar_complete_15m"
    ]
    .astype(bool)
)

input_bar_crosses_roll = (
    ledger[
        "crosses_roll"
    ]
    .astype(bool)
)


ledger.loc[
    no_policy_session,
    "primary_exclusion_reason",
] = (
    "NO_NYSE_POLICY_SESSION"
)

ledger.loc[
    before_policy_start
    &
    ledger[
        "primary_exclusion_reason"
    ]
    .eq(
        "ELIGIBLE"
    ),
    "primary_exclusion_reason",
] = (
    "BEFORE_NYSE_POLICY_START"
)

ledger.loc[
    after_horizon_safe_close
    &
    ledger[
        "primary_exclusion_reason"
    ]
    .eq(
        "ELIGIBLE"
    ),
    "primary_exclusion_reason",
] = (
    "AFTER_HORIZON_SAFE_CLOSE"
)

ledger.loc[
    input_bar_partial
    &
    ledger[
        "primary_exclusion_reason"
    ]
    .eq(
        "ELIGIBLE"
    ),
    "primary_exclusion_reason",
] = (
    "INPUT_BAR_PARTIAL"
)

ledger.loc[
    input_bar_crosses_roll
    &
    ledger[
        "primary_exclusion_reason"
    ]
    .eq(
        "ELIGIBLE"
    ),
    "primary_exclusion_reason",
] = (
    "INPUT_BAR_CROSSES_ROLL"
)


# ------------------------------------------------------------
# 12) Stable decision identifier
# ------------------------------------------------------------

decision_time_key = (
    ledger[
        "decision_time"
    ]
    .dt
    .strftime(
        "%Y-%m-%dT%H:%M:%SZ"
    )
)

ledger[
    "decision_id"
] = (
    decision_time_key
    +
    "|instrument_id="
    +
    ledger[
        "instrument_id"
    ]
    .astype(str)
)

ledger[
    "policy_version"
] = (
    POLICY_VERSION
)


# ------------------------------------------------------------
# 13) Freeze eligible decision observations
# ------------------------------------------------------------

universe_columns = [
    "decision_id",
    "policy_version",
    "bar_start_utc",
    "decision_time",
    "decision_time_ny",
    "nyse_session_date",
    "nyse_market_open_utc",
    "nyse_market_close_utc",
    "policy_first_decision_utc",
    "policy_last_decision_utc",
    "early_close_session",
    "instrument_id",
    "active_1m_count",
    "bar_complete_15m",
    "crosses_roll",
    "dataset_condition_utc",
    "dataset_degraded_utc",
    "decision_eligible",
]


decision_universe = (
    ledger.loc[
        ledger[
            "decision_eligible"
        ],
        universe_columns,
    ]
    .sort_values(
        "decision_time"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 14) Daily policy summary
# ------------------------------------------------------------

daily_summary = (
    ledger
    .groupby(
        "nyse_session_date",
        dropna=False,
    )
    .agg(
        clock_candidate_rows=(
            "decision_id",
            "size",
        ),
        nyse_policy_session_rows=(
            "nyse_policy_session",
            "sum",
        ),
        eligible_rows=(
            "decision_eligible",
            "sum",
        ),
        partial_input_rows=(
            "bar_complete_15m",
            lambda s: int(
                (~s.astype(bool)).sum()
            ),
        ),
        degraded_context_rows=(
            "dataset_degraded_utc",
            "sum",
        ),
        early_close_session=(
            "early_close_session",
            "max",
        ),
        first_candidate_time=(
            "decision_time",
            "min",
        ),
        last_candidate_time=(
            "decision_time",
            "max",
        ),
    )
    .reset_index()
)

daily_summary[
    "excluded_rows"
] = (
    daily_summary[
        "clock_candidate_rows"
    ]
    -
    daily_summary[
        "eligible_rows"
    ]
)


exclusion_summary = (
    ledger[
        "primary_exclusion_reason"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "primary_exclusion_reason"
    )
    .rename(
        "rows"
    )
    .reset_index()
)


# ------------------------------------------------------------
# 15) Hard audit gates
# ------------------------------------------------------------

failures = []


candidate_rows = int(
    len(
        ledger
    )
)

eligible_rows = int(
    len(
        decision_universe
    )
)

excluded_rows = int(
    candidate_rows
    -
    eligible_rows
)


cell5_clock_rows = (
    cell5_audit
    .get(
        "v1_clock_integrity",
        {},
    )
    .get(
        "clock_bars"
    )
)


if (
    cell5_clock_rows is not None
    and
    candidate_rows
    !=
    int(
        cell5_clock_rows
    )
):

    failures.append(
        "Clock-candidate count mismatch: "
        f"CELL 7 {candidate_rows:,} != "
        f"CELL 5 {int(cell5_clock_rows):,}"
    )


if candidate_rows != eligible_rows + excluded_rows:

    failures.append(
        "Candidate accounting does not reconcile."
    )


if int(
    ledger[
        "primary_exclusion_reason"
    ]
    .isna()
    .sum()
) != 0:

    failures.append(
        "Missing primary exclusion reason."
    )


reason_eligible_mask = (
    ledger[
        "primary_exclusion_reason"
    ]
    .eq(
        "ELIGIBLE"
    )
)


if not np.array_equal(
    reason_eligible_mask.to_numpy(),
    ledger[
        "decision_eligible"
    ]
    .to_numpy(),
):

    failures.append(
        "ELIGIBLE reason does not match decision_eligible."
    )


if int(
    decision_universe[
        "decision_time"
    ]
    .duplicated()
    .sum()
) != 0:

    failures.append(
        "Duplicate decision_time in frozen universe."
    )


if int(
    decision_universe[
        "decision_id"
    ]
    .duplicated()
    .sum()
) != 0:

    failures.append(
        "Duplicate decision_id in frozen universe."
    )


if not decision_universe[
    "decision_time"
].is_monotonic_increasing:

    failures.append(
        "Frozen universe is not time-sorted."
    )


eligible_partial_rows = int(
    (
        decision_universe[
            "bar_complete_15m"
        ]
        .astype(bool)
        ==
        False
    )
    .sum()
)


if eligible_partial_rows != 0:

    failures.append(
        "Partial input bars entered the frozen universe: "
        f"{eligible_partial_rows:,}"
    )


eligible_roll_cross_rows = int(
    decision_universe[
        "crosses_roll"
    ]
    .astype(bool)
    .sum()
)


if eligible_roll_cross_rows != 0:

    failures.append(
        "Mixed-contract input bars entered the frozen universe: "
        f"{eligible_roll_cross_rows:,}"
    )


eligible_outside_session = int(
    (
        ~ledger.loc[
            ledger[
                "decision_eligible"
            ],
            "within_nyse_entry_policy",
        ]
    )
    .sum()
)


if eligible_outside_session != 0:

    failures.append(
        "Eligible rows outside NYSE entry policy: "
        f"{eligible_outside_session:,}"
    )


eligible_after_safe_close = int(
    (
        ledger.loc[
            ledger[
                "decision_eligible"
            ],
            "decision_time",
        ]
        >
        ledger.loc[
            ledger[
                "decision_eligible"
            ],
            "policy_last_decision_utc",
        ]
    )
    .sum()
)


if eligible_after_safe_close != 0:

    failures.append(
        "Eligible decisions violate +60m close buffer: "
        f"{eligible_after_safe_close:,}"
    )


eligible_decision_minutes = (
    decision_universe[
        "decision_time"
    ]
    .dt.minute
)


off_grid_rows = int(
    (
        ~eligible_decision_minutes
        .isin(
            [
                0,
                15,
                30,
                45,
            ]
        )
    )
    .sum()
)


if off_grid_rows != 0:

    failures.append(
        "Eligible decisions are off the 15-minute grid: "
        f"{off_grid_rows:,}"
    )


if int(
    exclusion_summary[
        "rows"
    ]
    .sum()
) != candidate_rows:

    failures.append(
        "Exclusion summary does not sum to clock candidates."
    )


# ------------------------------------------------------------
# 16) Save versioned artifacts
# ------------------------------------------------------------

ledger = ledger.sort_values(
    "decision_time"
).reset_index(
    drop=True
)

decision_universe.to_parquet(
    CELL7_UNIVERSE_PATH,
    index=False,
)

ledger.to_parquet(
    CELL7_LEDGER_PATH,
    index=False,
)

daily_summary.to_csv(
    CELL7_DAILY_PATH,
    index=False,
)


artifact_hashes = {
    "input_mes_15m_sha256": sha256_file(
        MES_15M_PATH
    ),
    "cell4_audit_sha256": sha256_file(
        CELL4_AUDIT_PATH
    ),
    "cell5_audit_sha256": sha256_file(
        CELL5_AUDIT_PATH
    ),
    "cell6_audit_sha256": sha256_file(
        CELL6_AUDIT_PATH
    ),
    "decision_universe_sha256": sha256_file(
        CELL7_UNIVERSE_PATH
    ),
    "decision_ledger_sha256": sha256_file(
        CELL7_LEDGER_PATH
    ),
    "daily_summary_sha256": sha256_file(
        CELL7_DAILY_PATH
    ),
}


# ------------------------------------------------------------
# 17) Build and save the forensic audit
# ------------------------------------------------------------

partial_candidate_rows = int(
    (
        ~ledger[
            "bar_complete_15m"
        ]
        .astype(bool)
    )
    .sum()
)

degraded_candidate_rows = int(
    ledger[
        "dataset_degraded_utc"
    ]
    .astype(bool)
    .sum()
)

degraded_eligible_rows = int(
    decision_universe[
        "dataset_degraded_utc"
    ]
    .astype(bool)
    .sum()
)

early_close_candidate_rows = int(
    ledger[
        "early_close_session"
    ]
    .astype(bool)
    .sum()
)

early_close_eligible_rows = int(
    decision_universe[
        "early_close_session"
    ]
    .astype(bool)
    .sum()
)


cell7_audit = {
    "audit_written_utc": (
        datetime.now(
            timezone.utc
        )
        .isoformat()
    ),
    "policy_version": POLICY_VERSION,
    "status": (
        "PASS"
        if not failures
        else "FAIL"
    ),
    "upstream_binding": {
        "cell4_registry_status": (
            cell4_audit.get(
                "registry_status"
            )
        ),
        "cell5_failures": (
            cell5_audit.get(
                "failures",
                [],
            )
        ),
        "cell6_failures": (
            cell6_audit.get(
                "failures",
                [],
            )
        ),
        "cell6_unclassified_gap_events": (
            cell6_audit
            .get(
                "results",
                {},
            )
            .get(
                "primary_unclassified_events"
            )
        ),
    },
    "decision_policy": {
        "bar_minutes": BAR_MINUTES,
        "decision_time_semantics": (
            "end of completed 15-minute input bar"
        ),
        "timezone": NY_TZ,
        "clock_window_inclusive": (
            "09:45–15:00 America/New_York"
        ),
        "calendar": CALENDAR_NAME,
        "calendar_role": (
            "research/entry policy filter; not CME "
            "tradability authority"
        ),
        "label_horizon_minutes": (
            LABEL_HORIZON_MINUTES
        ),
        "scheduled_close_buffer_applied": True,
        "partial_bar_decision_eligible": False,
        "mixed_contract_bar_decision_eligible": False,
        "degraded_date_auto_exclusion": False,
        "future_return_used_for_eligibility": False,
        "future_bar_quality_used_for_eligibility": False,
    },
    "counts": {
        "input_15m_rows": int(
            len(
                mes_15m
            )
        ),
        "clock_candidate_rows": candidate_rows,
        "eligible_decision_rows": eligible_rows,
        "excluded_clock_candidate_rows": excluded_rows,
        "eligible_sessions": int(
            decision_universe[
                "nyse_session_date"
            ]
            .nunique()
        ),
        "partial_clock_candidate_rows": partial_candidate_rows,
        "eligible_partial_rows": eligible_partial_rows,
        "eligible_roll_cross_rows": eligible_roll_cross_rows,
        "degraded_clock_candidate_rows": degraded_candidate_rows,
        "degraded_eligible_rows_retained": degraded_eligible_rows,
        "early_close_clock_candidate_rows": early_close_candidate_rows,
        "early_close_eligible_rows": early_close_eligible_rows,
    },
    "time_range": {
        "first_eligible_decision_utc": (
            None
            if decision_universe.empty
            else decision_universe[
                "decision_time"
            ]
            .min()
            .isoformat()
        ),
        "last_eligible_decision_utc": (
            None
            if decision_universe.empty
            else decision_universe[
                "decision_time"
            ]
            .max()
            .isoformat()
        ),
    },
    "primary_exclusion_summary": (
        exclusion_summary
        .to_dict(
            orient="records"
        )
    ),
    "artifacts": {
        "decision_universe": str(
            CELL7_UNIVERSE_PATH
        ),
        "decision_ledger": str(
            CELL7_LEDGER_PATH
        ),
        "daily_summary": str(
            CELL7_DAILY_PATH
        ),
    },
    "sha256": artifact_hashes,
    "failures": failures,
}


with open(
    CELL7_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        cell7_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 18) Final hard gate
# ------------------------------------------------------------

if failures:

    print(
        "\nCELL 7 FAILURES"
    )
    print(
        "-" * 72
    )

    for failure in failures:

        print(
            " -",
            failure,
        )

    raise RuntimeError(
        "\nCELL 7 DECISION UNIVERSE: FAIL\n"
        f"{CELL7_AUDIT_PATH}"
    )


# ------------------------------------------------------------
# 19) Compact output
# ------------------------------------------------------------

print(
    "\n"
    +
    "=" * 72
)
print(
    "CELL 7 — POINT-IN-TIME DECISION UNIVERSE"
)
print(
    "=" * 72
)

print(
    "\n[1] POLICY"
)
print(
    "Policy version             :",
    POLICY_VERSION,
)
print(
    "Decision clock             : 09:45–15:00 America/New_York"
)
print(
    "Label-horizon close buffer : +60 minutes"
)
print(
    "Calendar role              : NYSE policy filter, not CME authority"
)
print(
    "Degraded-date exclusion    : False"
)
print(
    "Future-data eligibility    : False"
)

print(
    "\n[2] UNIVERSE COUNTS"
)
print(
    "Input 15m rows             :",
    f"{len(mes_15m):,}",
)
print(
    "Clock candidates           :",
    f"{candidate_rows:,}",
)
print(
    "Eligible decisions         :",
    f"{eligible_rows:,}",
)
print(
    "Excluded candidates        :",
    f"{excluded_rows:,}",
)
print(
    "Eligible NYSE sessions     :",
    f"{decision_universe['nyse_session_date'].nunique():,}",
)

print(
    "\n[3] INTEGRITY / CONTEXT"
)
print(
    "Partial clock candidates   :",
    f"{partial_candidate_rows:,}",
)
print(
    "Eligible partial bars      :",
    eligible_partial_rows,
)
print(
    "Eligible roll-cross bars   :",
    eligible_roll_cross_rows,
)
print(
    "Degraded eligible retained :",
    f"{degraded_eligible_rows:,}",
)
print(
    "Early-close eligible rows  :",
    f"{early_close_eligible_rows:,}",
)

print(
    "\n[4] PRIMARY EXCLUSION SUMMARY"
)
print(
    exclusion_summary
    .to_string(
        index=False
    )
)

print(
    "\n[5] TIME RANGE"
)
print(
    "First eligible decision    :",
    decision_universe[
        "decision_time"
    ]
    .min(),
)
print(
    "Last eligible decision     :",
    decision_universe[
        "decision_time"
    ]
    .max(),
)

print(
    "\n[6] SAVED ARTIFACTS"
)
print(
    "Frozen decision universe   :",
    CELL7_UNIVERSE_PATH,
)
print(
    "Full candidate ledger      :",
    CELL7_LEDGER_PATH,
)
print(
    "Daily summary              :",
    CELL7_DAILY_PATH,
)
print(
    "CELL 7 audit               :",
    CELL7_AUDIT_PATH,
)
print(
    "Universe SHA256            :",
    artifact_hashes[
        "decision_universe_sha256"
    ],
)

print(
    "\n"
    +
    "=" * 72
)
print(
    "CELL 7 DECISION UNIVERSE: PASS"
)
print(
    "=" * 72
)



CELL 7 — POINT-IN-TIME DECISION UNIVERSE

[1] POLICY
Policy version             : MES_V1_DECISION_UNIVERSE_1.0
Decision clock             : 09:45–15:00 America/New_York
Label-horizon close buffer : +60 minutes
Calendar role              : NYSE policy filter, not CME authority
Degraded-date exclusion    : False
Future-data eligibility    : False

[2] UNIVERSE COUNTS
Input 15m rows             : 170,586
Clock candidates           : 40,621
Eligible decisions         : 39,847
Excluded candidates        : 774
Eligible NYSE sessions     : 1,820

[3] INTEGRITY / CONTEXT
Partial clock candidates   : 47
Eligible partial bars      : 0
Eligible roll-cross bars   : 0
Degraded eligible retained : 230
Early-close eligible rows  : 149

[4] PRIMARY EXCLUSION SUMMARY
primary_exclusion_reason  rows
                ELIGIBLE 39847
  NO_NYSE_POLICY_SESSION   686
AFTER_HORIZON_SAFE_CLOSE    75
       INPUT_BAR_PARTIAL    13

[5] TIME RANGE
First eligible decision    : 2019-05-06 13:45:00+00:00
Last eligibl

In [45]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 8 — PURGED CHRONOLOGICAL SPLITS
# ============================================================
#
# PURPOSE
# -------
# Freeze how the Cell 7 decision universe is divided through time
# before any feature, economic label, or model is fitted.
#
# This cell:
# 1) binds itself to the exact Cell 7 universe hash,
# 2) reserves 2025 onward as an untouched final test period,
# 3) creates expanding walk-forward validation folds for 2022–2024,
# 4) purges any training row whose +60m information interval reaches
#    into the following validation/test period,
# 5) records every assignment, boundary, count, and SHA-256 hash.
#
# IMPORTANT
# ---------
# label_end_time is used only to audit temporal overlap. No return,
# direction, P&L, or economic label is created in this cell.
# ============================================================

import warnings as _warnings

_warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
)

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1) Paths and frozen policy
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Quant_Lab")
CLEAN_DIR = PROJECT_DIR / "Data" / "MES_Clean_Pipeline_V1"

CELL7_UNIVERSE_PATH = CLEAN_DIR / "cell7_decision_universe_v1.parquet"
CELL7_AUDIT_PATH = CLEAN_DIR / "cell7_decision_universe_audit.json"

CELL8_ASSIGNMENTS_PATH = (
    CLEAN_DIR / "cell8_purged_split_assignments_v1.parquet"
)
CELL8_FOLDS_PATH = CLEAN_DIR / "cell8_walk_forward_folds_v1.csv"
CELL8_BOUNDARIES_PATH = CLEAN_DIR / "cell8_purge_boundaries_v1.csv"
CELL8_AUDIT_PATH = CLEAN_DIR / "cell8_purged_split_audit.json"

SPLIT_POLICY_VERSION = "MES_V1_PURGED_SPLIT_1.0"
EXPECTED_CELL7_POLICY = "MES_V1_DECISION_UNIVERSE_1.0"
LABEL_HORIZON_MINUTES = 60

# Stable calendar-year contract. Appending new data must not move
# these historical boundaries; it requires a new policy version.
OUTER_TRAIN_END_YEAR = 2023
OUTER_VALIDATION_YEAR = 2024
FINAL_TEST_START_YEAR = 2025

# Expanding train -> next calendar-year validation.
WALK_FORWARD_SPECS = [
    ("WF_2022", 2021, 2022),
    ("WF_2023", 2022, 2023),
    ("WF_2024", 2023, 2024),
]

# Embargo is deliberately not invented here. Purging is mandatory;
# any non-zero embargo remains an OPEN research decision.
EMBARGO_MINUTES = 0


# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def iso_or_none(value):
    if value is None or pd.isna(value):
        return None
    return pd.Timestamp(value).isoformat()


# ------------------------------------------------------------
# 3) Upstream artifact and hash gates
# ------------------------------------------------------------
for required_path in [CELL7_UNIVERSE_PATH, CELL7_AUDIT_PATH]:
    if not required_path.exists():
        raise RuntimeError(
            "CELL 8 STOPPED — missing Cell 7 artifact:\n"
            f"{required_path}\n\n"
            "Run Cell 0 → Cell 7 first."
        )

cell7_audit = load_json(CELL7_AUDIT_PATH)

if cell7_audit.get("status") != "PASS":
    raise RuntimeError(
        "CELL 8 STOPPED — Cell 7 audit status is not PASS."
    )

if cell7_audit.get("failures", []):
    raise RuntimeError(
        "CELL 8 STOPPED — Cell 7 audit contains failures."
    )

if cell7_audit.get("policy_version") != EXPECTED_CELL7_POLICY:
    raise RuntimeError(
        "CELL 8 STOPPED — unexpected Cell 7 policy version:\n"
        f"{cell7_audit.get('policy_version')}"
    )

expected_cell7_hash = (
    cell7_audit
    .get("sha256", {})
    .get("decision_universe_sha256")
)
actual_cell7_hash = sha256_file(CELL7_UNIVERSE_PATH)

if not expected_cell7_hash:
    raise RuntimeError(
        "CELL 8 STOPPED — Cell 7 audit has no universe SHA-256."
    )

if actual_cell7_hash != expected_cell7_hash:
    raise RuntimeError(
        "CELL 8 STOPPED — Cell 7 universe hash mismatch.\n"
        f"Audit : {expected_cell7_hash}\n"
        f"Actual: {actual_cell7_hash}"
    )


# ------------------------------------------------------------
# 4) Load and validate the frozen decision universe
# ------------------------------------------------------------
universe = pd.read_parquet(CELL7_UNIVERSE_PATH)

required_columns = {
    "decision_id",
    "policy_version",
    "decision_time",
    "nyse_session_date",
    "instrument_id",
    "bar_complete_15m",
    "crosses_roll",
    "decision_eligible",
    "dataset_degraded_utc",
}
missing_columns = required_columns - set(universe.columns)
if missing_columns:
    raise RuntimeError(
        "CELL 8 STOPPED — missing Cell 7 columns:\n"
        + ", ".join(sorted(missing_columns))
    )

universe = universe.copy()
universe["decision_time"] = pd.to_datetime(
    universe["decision_time"],
    utc=True,
    errors="raise",
)
universe["nyse_session_date"] = (
    pd.to_datetime(
        universe["nyse_session_date"],
        errors="raise",
    )
    .dt.date
)
universe = universe.sort_values("decision_time").reset_index(drop=True)

expected_rows = int(
    cell7_audit
    .get("counts", {})
    .get("eligible_decision_rows", -1)
)

structural_failures = []
if len(universe) != expected_rows:
    structural_failures.append(
        "Cell 7 row count does not match its audit: "
        f"{len(universe):,} != {expected_rows:,}"
    )
if universe.empty:
    structural_failures.append("Cell 7 universe is empty.")
if universe["decision_id"].duplicated().any():
    structural_failures.append("Duplicate decision_id in Cell 7 universe.")
if universe["decision_time"].duplicated().any():
    structural_failures.append("Duplicate decision_time in Cell 7 universe.")
if not universe["decision_time"].is_monotonic_increasing:
    structural_failures.append("Cell 7 universe is not time-sorted.")
if not universe["decision_eligible"].astype(bool).all():
    structural_failures.append("Ineligible row found in Cell 7 universe.")
if (~universe["bar_complete_15m"].astype(bool)).any():
    structural_failures.append("Partial bar found in Cell 7 universe.")
if universe["crosses_roll"].astype(bool).any():
    structural_failures.append("Roll-cross bar found in Cell 7 universe.")

if structural_failures:
    raise RuntimeError(
        "CELL 8 STOPPED — Cell 7 structural gate failed:\n- "
        + "\n- ".join(structural_failures)
    )


# ------------------------------------------------------------
# 5) Build immutable time assignments
# ------------------------------------------------------------
assignments = universe[
    [
        "decision_id",
        "decision_time",
        "nyse_session_date",
        "instrument_id",
        "dataset_degraded_utc",
    ]
].copy()

assignments["label_start_time"] = assignments["decision_time"]
assignments["label_end_time"] = (
    assignments["decision_time"]
    + pd.Timedelta(minutes=LABEL_HORIZON_MINUTES)
)

session_ts = pd.to_datetime(assignments["nyse_session_date"])
session_year = session_ts.dt.year

assignments["outer_partition"] = np.select(
    [
        session_year.le(OUTER_TRAIN_END_YEAR),
        session_year.eq(OUTER_VALIDATION_YEAR),
        session_year.ge(FINAL_TEST_START_YEAR),
    ],
    [
        "TRAIN",
        "VALIDATION",
        "FINAL_TEST",
    ],
    default="UNASSIGNED",
)


# ------------------------------------------------------------
# 6) Expanding walk-forward folds with overlap purging
# ------------------------------------------------------------
fold_rows = []
boundary_rows = []
failures = []

for fold_id, train_end_year, validation_year in WALK_FORWARD_SPECS:
    raw_train_mask = session_year.le(train_end_year)
    validation_mask = session_year.eq(validation_year)

    if not raw_train_mask.any():
        failures.append(f"{fold_id}: empty training period.")
        continue
    if not validation_mask.any():
        failures.append(f"{fold_id}: empty validation period.")
        continue

    validation_start = assignments.loc[
        validation_mask,
        "decision_time",
    ].min()

    # Purge train observations whose information interval touches or
    # crosses the first validation decision timestamp.
    purge_mask = (
        raw_train_mask
        & assignments["label_end_time"].ge(validation_start)
    )
    kept_train_mask = raw_train_mask & ~purge_mask

    role_column = "role_" + fold_id.lower()
    assignments[role_column] = "UNUSED"
    assignments.loc[kept_train_mask, role_column] = "TRAIN"
    assignments.loc[purge_mask, role_column] = "PURGED"
    assignments.loc[validation_mask, role_column] = "VALIDATION"

    max_kept_train_label_end = assignments.loc[
        kept_train_mask,
        "label_end_time",
    ].max()
    overlap_after_purge = int(
        assignments.loc[
            kept_train_mask,
            "label_end_time",
        ]
        .ge(validation_start)
        .sum()
    )

    if overlap_after_purge != 0:
        failures.append(
            f"{fold_id}: {overlap_after_purge:,} train labels "
            "still overlap validation after purging."
        )

    test_leakage_rows = int(
        (
            session_year.ge(FINAL_TEST_START_YEAR)
            & assignments[role_column].ne("UNUSED")
        ).sum()
    )
    if test_leakage_rows != 0:
        failures.append(
            f"{fold_id}: {test_leakage_rows:,} final-test rows leaked "
            "into walk-forward development."
        )

    train_sessions = int(
        assignments.loc[
            kept_train_mask,
            "nyse_session_date",
        ].nunique()
    )
    validation_sessions = int(
        assignments.loc[
            validation_mask,
            "nyse_session_date",
        ].nunique()
    )

    if train_sessions < 500:
        failures.append(
            f"{fold_id}: only {train_sessions:,} training sessions; "
            "minimum is 500."
        )
    if validation_sessions < 200:
        failures.append(
            f"{fold_id}: only {validation_sessions:,} validation "
            "sessions; minimum is 200."
        )

    natural_gap_minutes = (
        None
        if pd.isna(max_kept_train_label_end)
        else float(
            (
                validation_start
                - max_kept_train_label_end
            ).total_seconds()
            / 60.0
        )
    )

    fold_rows.append(
        {
            "fold_id": fold_id,
            "train_start_session": str(
                assignments.loc[
                    kept_train_mask,
                    "nyse_session_date",
                ].min()
            ),
            "train_end_session": str(
                assignments.loc[
                    kept_train_mask,
                    "nyse_session_date",
                ].max()
            ),
            "validation_start_session": str(
                assignments.loc[
                    validation_mask,
                    "nyse_session_date",
                ].min()
            ),
            "validation_end_session": str(
                assignments.loc[
                    validation_mask,
                    "nyse_session_date",
                ].max()
            ),
            "train_rows_before_purge": int(raw_train_mask.sum()),
            "purged_train_rows": int(purge_mask.sum()),
            "train_rows_after_purge": int(kept_train_mask.sum()),
            "validation_rows": int(validation_mask.sum()),
            "train_sessions": train_sessions,
            "validation_sessions": validation_sessions,
            "overlap_rows_after_purge": overlap_after_purge,
            "embargo_minutes": EMBARGO_MINUTES,
        }
    )

    boundary_rows.append(
        {
            "boundary_id": fold_id + "_VALIDATION_START",
            "left_role": "TRAIN",
            "right_role": "VALIDATION",
            "right_first_decision_utc": iso_or_none(validation_start),
            "left_last_label_end_after_purge_utc": iso_or_none(
                max_kept_train_label_end
            ),
            "purged_left_rows": int(purge_mask.sum()),
            "overlap_rows_after_purge": overlap_after_purge,
            "natural_gap_minutes_after_purge": natural_gap_minutes,
        }
    )


# ------------------------------------------------------------
# 7) Seal the final test boundary
# ------------------------------------------------------------
final_test_mask = assignments["outer_partition"].eq("FINAL_TEST")
development_mask = assignments["outer_partition"].isin(
    ["TRAIN", "VALIDATION"]
)

if not final_test_mask.any():
    failures.append("Final test period is empty.")
    final_test_start = pd.NaT
    pretest_purge_mask = pd.Series(False, index=assignments.index)
else:
    final_test_start = assignments.loc[
        final_test_mask,
        "decision_time",
    ].min()
    pretest_purge_mask = (
        development_mask
        & assignments["label_end_time"].ge(final_test_start)
    )

assignments["purged_before_final_test"] = pretest_purge_mask.astype(bool)
assignments["outer_modeling_eligible"] = (
    ~final_test_mask
    & ~assignments["purged_before_final_test"]
)

pretest_overlap_after = int(
    assignments.loc[
        development_mask
        & ~pretest_purge_mask,
        "label_end_time",
    ]
    .ge(final_test_start)
    .sum()
) if not pd.isna(final_test_start) else -1

if pretest_overlap_after != 0:
    failures.append(
        "Development labels still overlap final test after purging: "
        f"{pretest_overlap_after:,}"
    )

outer_counts = (
    assignments["outer_partition"]
    .value_counts()
    .reindex(
        ["TRAIN", "VALIDATION", "FINAL_TEST", "UNASSIGNED"],
        fill_value=0,
    )
)

if int(outer_counts["UNASSIGNED"]) != 0:
    failures.append(
        f"Unassigned outer rows: {int(outer_counts['UNASSIGNED']):,}"
    )
if int(outer_counts.sum()) != len(assignments):
    failures.append("Outer partition counts do not reconcile.")

final_test_sessions = int(
    assignments.loc[
        final_test_mask,
        "nyse_session_date",
    ].nunique()
)
if final_test_sessions < 250:
    failures.append(
        f"Final test has only {final_test_sessions:,} sessions; "
        "minimum is 250."
    )

max_dev_label_end = assignments.loc[
    development_mask & ~pretest_purge_mask,
    "label_end_time",
].max()
test_gap_minutes = (
    None
    if pd.isna(final_test_start) or pd.isna(max_dev_label_end)
    else float(
        (final_test_start - max_dev_label_end).total_seconds()
        / 60.0
    )
)

boundary_rows.append(
    {
        "boundary_id": "FINAL_TEST_START",
        "left_role": "MODEL_DEVELOPMENT",
        "right_role": "FINAL_TEST",
        "right_first_decision_utc": iso_or_none(final_test_start),
        "left_last_label_end_after_purge_utc": iso_or_none(
            max_dev_label_end
        ),
        "purged_left_rows": int(pretest_purge_mask.sum()),
        "overlap_rows_after_purge": pretest_overlap_after,
        "natural_gap_minutes_after_purge": test_gap_minutes,
    }
)


# ------------------------------------------------------------
# 8) Final audit gates
# ------------------------------------------------------------
if assignments["decision_id"].duplicated().any():
    failures.append("Duplicate decision_id in Cell 8 assignments.")
if len(assignments) != len(universe):
    failures.append("Cell 8 assignments changed the Cell 7 row count.")
if not assignments["decision_time"].is_monotonic_increasing:
    failures.append("Cell 8 assignments are not time-sorted.")
if int(assignments["label_end_time"].isna().sum()) != 0:
    failures.append("Missing label_end_time in Cell 8 assignments.")
if not assignments["label_end_time"].eq(
    assignments["decision_time"]
    + pd.Timedelta(minutes=LABEL_HORIZON_MINUTES)
).all():
    failures.append("Incorrect +60m label interval endpoint.")

folds = pd.DataFrame(fold_rows)
boundaries = pd.DataFrame(boundary_rows)

if len(folds) != len(WALK_FORWARD_SPECS):
    failures.append(
        f"Expected {len(WALK_FORWARD_SPECS)} walk-forward folds; "
        f"built {len(folds)}."
    )
if int(boundaries["overlap_rows_after_purge"].sum()) != 0:
    failures.append("A split boundary still has temporal overlap.")


# ------------------------------------------------------------
# 9) Save versioned artifacts and bind hashes
# ------------------------------------------------------------
assignments.to_parquet(
    CELL8_ASSIGNMENTS_PATH,
    index=False,
)
folds.to_csv(
    CELL8_FOLDS_PATH,
    index=False,
)
boundaries.to_csv(
    CELL8_BOUNDARIES_PATH,
    index=False,
)

artifact_hashes = {
    "input_cell7_universe_sha256": actual_cell7_hash,
    "input_cell7_audit_sha256": sha256_file(CELL7_AUDIT_PATH),
    "split_assignments_sha256": sha256_file(CELL8_ASSIGNMENTS_PATH),
    "walk_forward_folds_sha256": sha256_file(CELL8_FOLDS_PATH),
    "purge_boundaries_sha256": sha256_file(CELL8_BOUNDARIES_PATH),
}

partition_summary = []
for partition_name in ["TRAIN", "VALIDATION", "FINAL_TEST"]:
    mask = assignments["outer_partition"].eq(partition_name)
    partition_summary.append(
        {
            "outer_partition": partition_name,
            "rows": int(mask.sum()),
            "sessions": int(
                assignments.loc[mask, "nyse_session_date"].nunique()
            ),
            "first_session": str(
                assignments.loc[mask, "nyse_session_date"].min()
            ),
            "last_session": str(
                assignments.loc[mask, "nyse_session_date"].max()
            ),
            "first_decision_utc": iso_or_none(
                assignments.loc[mask, "decision_time"].min()
            ),
            "last_decision_utc": iso_or_none(
                assignments.loc[mask, "decision_time"].max()
            ),
        }
    )

cell8_audit = {
    "audit_written_utc": datetime.now(timezone.utc).isoformat(),
    "policy_version": SPLIT_POLICY_VERSION,
    "status": "PASS" if not failures else "FAIL",
    "upstream_binding": {
        "cell7_policy_version": cell7_audit.get("policy_version"),
        "cell7_status": cell7_audit.get("status"),
        "cell7_eligible_rows": expected_rows,
        "cell7_universe_sha256": actual_cell7_hash,
    },
    "split_contract": {
        "method": "calendar-year chronological expanding walk-forward",
        "outer_train_through_year": OUTER_TRAIN_END_YEAR,
        "outer_validation_year": OUTER_VALIDATION_YEAR,
        "final_test_from_year": FINAL_TEST_START_YEAR,
        "final_test_is_untouched": True,
        "label_horizon_minutes_for_purging": LABEL_HORIZON_MINUTES,
        "purge_rule": (
            "remove left-side observations when label_end_time >= "
            "first decision_time of the right-side period"
        ),
        "embargo_minutes": EMBARGO_MINUTES,
        "embargo_status": (
            "OPEN — no non-zero embargo is assumed; revisit after "
            "feature and label dependence analysis"
        ),
        "economic_label_created": False,
        "future_return_created": False,
    },
    "counts": {
        "decision_rows": int(len(assignments)),
        "decision_sessions": int(
            assignments["nyse_session_date"].nunique()
        ),
        "outer_train_rows": int(outer_counts["TRAIN"]),
        "outer_validation_rows": int(outer_counts["VALIDATION"]),
        "final_test_rows": int(outer_counts["FINAL_TEST"]),
        "final_test_sessions": final_test_sessions,
        "purged_before_final_test_rows": int(pretest_purge_mask.sum()),
        "walk_forward_folds": int(len(folds)),
        "total_fold_purged_rows": int(
            folds["purged_train_rows"].sum()
        ) if not folds.empty else 0,
        "boundary_overlap_rows_after_purge": int(
            boundaries["overlap_rows_after_purge"].sum()
        ),
    },
    "outer_partitions": partition_summary,
    "walk_forward_folds": folds.to_dict(orient="records"),
    "purge_boundaries": boundaries.to_dict(orient="records"),
    "artifacts": {
        "split_assignments": str(CELL8_ASSIGNMENTS_PATH),
        "walk_forward_folds": str(CELL8_FOLDS_PATH),
        "purge_boundaries": str(CELL8_BOUNDARIES_PATH),
    },
    "sha256": artifact_hashes,
    "failures": failures,
}

with open(CELL8_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        cell8_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 10) Final hard gate and compact output
# ------------------------------------------------------------
if failures:
    print("\nCELL 8 FAILURES")
    print("-" * 72)
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(
        "\nCELL 8 PURGED CHRONOLOGICAL SPLITS: FAIL\n"
        f"{CELL8_AUDIT_PATH}"
    )

print("\n" + "=" * 72)
print("CELL 8 — PURGED CHRONOLOGICAL SPLITS")
print("=" * 72)

print("\n[1] UPSTREAM BINDING")
print("Cell 7 rows        :", f"{len(assignments):,}")
print("Cell 7 sessions    :", f"{assignments['nyse_session_date'].nunique():,}")
print("Cell 7 SHA256      :", actual_cell7_hash)

print("\n[2] OUTER PARTITIONS")
print(
    pd.DataFrame(partition_summary)[
        [
            "outer_partition",
            "rows",
            "sessions",
            "first_session",
            "last_session",
        ]
    ].to_string(index=False)
)

print("\n[3] WALK-FORWARD FOLDS")
print(
    folds[
        [
            "fold_id",
            "train_rows_after_purge",
            "validation_rows",
            "purged_train_rows",
            "overlap_rows_after_purge",
        ]
    ].to_string(index=False)
)

print("\n[4] PURGE / TEST SAFETY")
print(
    "Final-test rows                 :",
    f"{int(outer_counts['FINAL_TEST']):,}",
)
print(
    "Purged before final test        :",
    f"{int(pretest_purge_mask.sum()):,}",
)
print(
    "Boundary overlap after purging  :",
    int(boundaries["overlap_rows_after_purge"].sum()),
)
print("Embargo minutes                  :", EMBARGO_MINUTES)
print("Economic label created           : False")

print("\n[5] SAVED ARTIFACTS")
print("Split assignments :", CELL8_ASSIGNMENTS_PATH)
print("Fold manifest     :", CELL8_FOLDS_PATH)
print("Boundary audit    :", CELL8_BOUNDARIES_PATH)
print("Cell 8 audit      :", CELL8_AUDIT_PATH)
print("Assignments SHA256:", artifact_hashes["split_assignments_sha256"])

print("\n" + "=" * 72)
print("CELL 8 PURGED CHRONOLOGICAL SPLITS: PASS")
print("=" * 72)




CELL 8 — PURGED CHRONOLOGICAL SPLITS

[1] UPSTREAM BINDING
Cell 7 rows        : 39,847
Cell 7 sessions    : 1,820
Cell 7 SHA256      : f86024c7a36780e6a559cc0eec15a7a52a851b24cb453a50136b609c440f2ca7

[2] OUTER PARTITIONS
outer_partition  rows  sessions first_session last_session
          TRAIN 25685      1173    2019-05-06   2023-12-29
     VALIDATION  5508       252    2024-01-02   2024-12-31
     FINAL_TEST  8654       395    2025-01-02   2026-07-31

[3] WALK-FORWARD FOLDS
fold_id  train_rows_after_purge  validation_rows  purged_train_rows  overlap_rows_after_purge
WF_2022                   14699             5510                  0                         0
WF_2023                   20209             5476                  0                         0
WF_2024                   25685             5508                  0                         0

[4] PURGE / TEST SAFETY
Final-test rows                 : 8,654
Purged before final test        : 0
Boundary overlap after purging  : 0
Emba

In [46]:
# CELL 9 — RESEARCH COST MODEL CONTRACT
# ============================================================
#
# PURPOSE
# -------
# Freeze a transparent, versioned transaction-cost contract before
# any economic label, signal threshold, or model is finalized.
#
# This cell:
# 1) binds itself to the exact Cell 8 split-assignment hash,
# 2) records the official MES contract mechanics,
# 3) snapshots current low-volume IBKR/CME direct fees,
# 4) creates fees-only, base, conservative, and stress scenarios,
# 5) converts all-in round-trip cost to USD, ticks, and index points,
# 6) saves parameters, scenarios, hashes, sources, and audit gates.
#
# IMPORTANT
# ---------
# - No price return, direction label, P&L, or model is created here.
# - Final-test outcomes are not loaded or inspected.
# - Live fees depend on account entity, plan, volume, tax, and date.
#   Therefore fee and execution assumptions remain PROVISIONAL until
#   reconciled against the user's actual IBKR statement/fills.
# ============================================================

import warnings as _warnings

_warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
)

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1) Paths
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Quant_Lab")
CLEAN_DIR = PROJECT_DIR / "Data" / "MES_Clean_Pipeline_V1"

CELL8_ASSIGNMENTS_PATH = (
    CLEAN_DIR / "cell8_purged_split_assignments_v1.parquet"
)
CELL8_AUDIT_PATH = CLEAN_DIR / "cell8_purged_split_audit.json"

CELL9_PARAMETERS_PATH = CLEAN_DIR / "cell9_cost_parameters_v1.csv"
CELL9_SCENARIOS_PATH = CLEAN_DIR / "cell9_cost_scenarios_v1.csv"
CELL9_AUDIT_PATH = CLEAN_DIR / "cell9_cost_model_audit.json"


# ------------------------------------------------------------
# 2) Version and official source snapshot
# ------------------------------------------------------------
COST_POLICY_VERSION = "MES_V1_COST_MODEL_1.0"
EXPECTED_CELL8_POLICY = "MES_V1_PURGED_SPLIT_1.0"
SOURCE_SNAPSHOT_DATE = "2026-08-09"

CME_CONTRACT_SOURCE = (
    "https://www.cmegroup.com/markets/equities/sp/"
    "micro-e-mini-sandp-500.contractSpecs.html"
)
CME_FEE_SOURCE = (
    "https://www.cmegroup.com/company/files/"
    "cme-fee-schedule-2026-02-01.pdf"
)
IBKR_COMMISSION_SOURCE = (
    "https://www.interactivebrokers.com/en/pricing/"
    "commissions-futures.php"
)
IBKR_CME_FEE_SOURCE = (
    "https://www.interactivebrokers.com/en/accounts/fees/CME.php"
)


# ------------------------------------------------------------
# 3) Frozen MES mechanics — official contract specification
# ------------------------------------------------------------
SYMBOL = "MES"
CURRENCY = "USD"
CONTRACT_MULTIPLIER_USD_PER_POINT = 5.00
TICK_SIZE_POINTS = 0.25
TICK_VALUE_USD = (
    CONTRACT_MULTIPLIER_USD_PER_POINT
    * TICK_SIZE_POINTS
)


# ------------------------------------------------------------
# 4) Dated direct-fee snapshot — PROVISIONAL
#
# Account/pricing assumptions:
# - non-member client,
# - IBKR Spot-Quoted/E-micro futures group containing MES,
# - <= 1,000 monthly contracts,
# - one contract, each side,
# - no give-up surcharge,
# - tax/entity adjustments not yet supplied by the user.
# ------------------------------------------------------------
IBKR_EXECUTION_FEE_PER_SIDE_USD = 0.25
CME_EXCHANGE_FEE_PER_SIDE_USD = 0.35
REGULATORY_FEE_PER_SIDE_USD = 0.01
CLEARING_FEE_PER_SIDE_USD = 0.00
TAX_OR_ENTITY_ADJUSTMENT_PER_SIDE_USD = 0.00

DIRECT_FEE_PER_SIDE_USD = (
    IBKR_EXECUTION_FEE_PER_SIDE_USD
    + CME_EXCHANGE_FEE_PER_SIDE_USD
    + REGULATORY_FEE_PER_SIDE_USD
    + CLEARING_FEE_PER_SIDE_USD
    + TAX_OR_ENTITY_ADJUSTMENT_PER_SIDE_USD
)
DIRECT_FEE_ROUND_TRIP_USD = 2.0 * DIRECT_FEE_PER_SIDE_USD

# V1 exits within +60 minutes and before the NYSE-policy close buffer,
# so no overnight-position fee is included in this intraday contract.
OVERNIGHT_FEE_ROUND_TRIP_USD = 0.00


# ------------------------------------------------------------
# 5) Scenario contract
#
# All tick inputs are per side. Spread is represented as half-spread
# paid on each marketable fill. Slippage, latency, and impact are kept
# separate so later real fills can replace each assumption cleanly.
# ------------------------------------------------------------
PRIMARY_ECONOMIC_GATE_SCENARIO = "CONSERVATIVE"

SCENARIO_INPUTS = [
    {
        "scenario": "FEES_ONLY",
        "purpose": "hard lower bound; not a tradable expectation",
        "spread_half_ticks_per_side": 0.00,
        "slippage_ticks_per_side": 0.00,
        "latency_ticks_per_side": 0.00,
        "market_impact_ticks_per_side": 0.00,
        "status": "PROVISIONAL",
    },
    {
        "scenario": "BASE",
        "purpose": "one-tick quoted spread plus modest adverse fill",
        "spread_half_ticks_per_side": 0.50,
        "slippage_ticks_per_side": 0.25,
        "latency_ticks_per_side": 0.00,
        "market_impact_ticks_per_side": 0.00,
        "status": "PROVISIONAL",
    },
    {
        "scenario": "CONSERVATIVE",
        "purpose": "primary economic gate before real fill calibration",
        "spread_half_ticks_per_side": 0.50,
        "slippage_ticks_per_side": 0.50,
        "latency_ticks_per_side": 0.25,
        "market_impact_ticks_per_side": 0.25,
        "status": "PROVISIONAL",
    },
    {
        "scenario": "STRESS",
        "purpose": "wider spread and adverse execution stress test",
        "spread_half_ticks_per_side": 1.00,
        "slippage_ticks_per_side": 1.00,
        "latency_ticks_per_side": 0.50,
        "market_impact_ticks_per_side": 0.50,
        "status": "PROVISIONAL",
    },
]


# ------------------------------------------------------------
# 6) Helpers
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ------------------------------------------------------------
# 7) Bind to Cell 8 without inspecting outcomes
# ------------------------------------------------------------
for required_path in [CELL8_ASSIGNMENTS_PATH, CELL8_AUDIT_PATH]:
    if not required_path.exists():
        raise RuntimeError(
            "CELL 9 STOPPED — missing Cell 8 artifact:\n"
            f"{required_path}\n\n"
            "Run Cell 0 → Cell 8 first."
        )

cell8_audit = load_json(CELL8_AUDIT_PATH)

if cell8_audit.get("status") != "PASS":
    raise RuntimeError(
        "CELL 9 STOPPED — Cell 8 audit status is not PASS."
    )
if cell8_audit.get("failures", []):
    raise RuntimeError(
        "CELL 9 STOPPED — Cell 8 audit contains failures."
    )
if cell8_audit.get("policy_version") != EXPECTED_CELL8_POLICY:
    raise RuntimeError(
        "CELL 9 STOPPED — unexpected Cell 8 policy version:\n"
        f"{cell8_audit.get('policy_version')}"
    )

expected_cell8_hash = (
    cell8_audit
    .get("sha256", {})
    .get("split_assignments_sha256")
)
actual_cell8_hash = sha256_file(CELL8_ASSIGNMENTS_PATH)

if not expected_cell8_hash:
    raise RuntimeError(
        "CELL 9 STOPPED — Cell 8 audit has no assignments SHA-256."
    )
if actual_cell8_hash != expected_cell8_hash:
    raise RuntimeError(
        "CELL 9 STOPPED — Cell 8 assignments hash mismatch.\n"
        f"Audit : {expected_cell8_hash}\n"
        f"Actual: {actual_cell8_hash}"
    )

# Load assignment identifiers only. No market price, future return,
# label, prediction, or final-test performance is inspected.
assignment_columns = [
    "decision_id",
    "outer_partition",
]
assignments = pd.read_parquet(
    CELL8_ASSIGNMENTS_PATH,
    columns=assignment_columns,
)

expected_decision_rows = int(
    cell8_audit
    .get("counts", {})
    .get("decision_rows", -1)
)
expected_final_test_rows = int(
    cell8_audit
    .get("counts", {})
    .get("final_test_rows", -1)
)
observed_final_test_rows = int(
    assignments["outer_partition"]
    .eq("FINAL_TEST")
    .sum()
)

if len(assignments) != expected_decision_rows:
    raise RuntimeError(
        "CELL 9 STOPPED — Cell 8 assignment count mismatch."
    )
if assignments["decision_id"].duplicated().any():
    raise RuntimeError(
        "CELL 9 STOPPED — duplicate decision_id in Cell 8 assignments."
    )
if observed_final_test_rows != expected_final_test_rows:
    raise RuntimeError(
        "CELL 9 STOPPED — final-test row count changed."
    )


# ------------------------------------------------------------
# 8) Build the parameter registry
# ------------------------------------------------------------
parameter_rows = [
    {
        "parameter": "contract_multiplier_usd_per_point",
        "value": CONTRACT_MULTIPLIER_USD_PER_POINT,
        "unit": "USD/index_point/contract",
        "status": "LOCKED",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": CME_CONTRACT_SOURCE,
        "notes": "Official MES contract multiplier.",
    },
    {
        "parameter": "tick_size_points",
        "value": TICK_SIZE_POINTS,
        "unit": "index_points",
        "status": "LOCKED",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": CME_CONTRACT_SOURCE,
        "notes": "Official minimum outright price fluctuation.",
    },
    {
        "parameter": "tick_value_usd",
        "value": TICK_VALUE_USD,
        "unit": "USD/tick/contract",
        "status": "LOCKED",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": CME_CONTRACT_SOURCE,
        "notes": "Derived exactly as multiplier × tick size.",
    },
    {
        "parameter": "ibkr_execution_fee_per_side_usd",
        "value": IBKR_EXECUTION_FEE_PER_SIDE_USD,
        "unit": "USD/side/contract",
        "status": "PROVISIONAL",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": IBKR_COMMISSION_SOURCE,
        "notes": (
            "MES group; <=1,000 monthly contracts; verify account plan."
        ),
    },
    {
        "parameter": "cme_exchange_fee_per_side_usd",
        "value": CME_EXCHANGE_FEE_PER_SIDE_USD,
        "unit": "USD/side/contract",
        "status": "PROVISIONAL",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": IBKR_CME_FEE_SOURCE,
        "notes": "Non-member Micro E-mini futures recovery charge.",
    },
    {
        "parameter": "regulatory_fee_per_side_usd",
        "value": REGULATORY_FEE_PER_SIDE_USD,
        "unit": "USD/side/contract",
        "status": "PROVISIONAL",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": IBKR_CME_FEE_SOURCE,
        "notes": "NFA regulatory fee recovery charge shown by IBKR.",
    },
    {
        "parameter": "clearing_fee_per_side_usd",
        "value": CLEARING_FEE_PER_SIDE_USD,
        "unit": "USD/side/contract",
        "status": "PROVISIONAL",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": IBKR_COMMISSION_SOURCE,
        "notes": "Zero working assumption; reconcile to actual statement.",
    },
    {
        "parameter": "tax_or_entity_adjustment_per_side_usd",
        "value": TAX_OR_ENTITY_ADJUSTMENT_PER_SIDE_USD,
        "unit": "USD/side/contract",
        "status": "OPEN",
        "as_of": SOURCE_SNAPSHOT_DATE,
        "source": "USER_IBKR_STATEMENT_REQUIRED",
        "notes": "Depends on IBKR entity, residence, and applicable tax.",
    },
]

parameters = pd.DataFrame(parameter_rows)


# ------------------------------------------------------------
# 9) Calculate cost scenarios
# ------------------------------------------------------------
scenario_rows = []

for item in SCENARIO_INPUTS:
    spread_round_trip_usd = (
        2.0
        * item["spread_half_ticks_per_side"]
        * TICK_VALUE_USD
    )
    slippage_round_trip_usd = (
        2.0
        * item["slippage_ticks_per_side"]
        * TICK_VALUE_USD
    )
    latency_round_trip_usd = (
        2.0
        * item["latency_ticks_per_side"]
        * TICK_VALUE_USD
    )
    market_impact_round_trip_usd = (
        2.0
        * item["market_impact_ticks_per_side"]
        * TICK_VALUE_USD
    )

    total_round_trip_usd = (
        DIRECT_FEE_ROUND_TRIP_USD
        + OVERNIGHT_FEE_ROUND_TRIP_USD
        + spread_round_trip_usd
        + slippage_round_trip_usd
        + latency_round_trip_usd
        + market_impact_round_trip_usd
    )

    scenario_rows.append(
        {
            **item,
            "contracts": 1,
            "direct_fee_per_side_usd": DIRECT_FEE_PER_SIDE_USD,
            "direct_fee_round_trip_usd": DIRECT_FEE_ROUND_TRIP_USD,
            "spread_round_trip_usd": spread_round_trip_usd,
            "slippage_round_trip_usd": slippage_round_trip_usd,
            "latency_round_trip_usd": latency_round_trip_usd,
            "market_impact_round_trip_usd": (
                market_impact_round_trip_usd
            ),
            "overnight_fee_round_trip_usd": (
                OVERNIGHT_FEE_ROUND_TRIP_USD
            ),
            "total_round_trip_usd": total_round_trip_usd,
            "break_even_ticks": total_round_trip_usd / TICK_VALUE_USD,
            "break_even_index_points": (
                total_round_trip_usd
                / CONTRACT_MULTIPLIER_USD_PER_POINT
            ),
            "primary_economic_gate": (
                item["scenario"]
                == PRIMARY_ECONOMIC_GATE_SCENARIO
            ),
        }
    )

scenarios = pd.DataFrame(scenario_rows)

# Remove binary floating-point display noise from saved contracts
# (for example 3.0949999999999998 -> 3.095000).
scenario_round_columns = [
    "direct_fee_per_side_usd",
    "direct_fee_round_trip_usd",
    "spread_round_trip_usd",
    "slippage_round_trip_usd",
    "latency_round_trip_usd",
    "market_impact_round_trip_usd",
    "overnight_fee_round_trip_usd",
    "total_round_trip_usd",
    "break_even_ticks",
    "break_even_index_points",
]
scenarios[scenario_round_columns] = (
    scenarios[scenario_round_columns]
    .round(6)
)


# ------------------------------------------------------------
# 10) Hard cost-model gates
# ------------------------------------------------------------
failures = []

if not np.isclose(TICK_VALUE_USD, 1.25, atol=1e-12):
    failures.append("MES tick value is not exactly USD 1.25.")
if not np.isclose(DIRECT_FEE_PER_SIDE_USD, 0.61, atol=1e-12):
    failures.append("Direct fee snapshot does not sum to USD 0.61/side.")
if not np.isclose(DIRECT_FEE_ROUND_TRIP_USD, 1.22, atol=1e-12):
    failures.append("Direct round-trip fee is not USD 1.22.")

numeric_cost_columns = [
    "direct_fee_round_trip_usd",
    "spread_round_trip_usd",
    "slippage_round_trip_usd",
    "latency_round_trip_usd",
    "market_impact_round_trip_usd",
    "overnight_fee_round_trip_usd",
    "total_round_trip_usd",
    "break_even_ticks",
    "break_even_index_points",
]

if scenarios[numeric_cost_columns].isna().any().any():
    failures.append("Missing numeric value in cost scenarios.")
if (scenarios[numeric_cost_columns] < 0).any().any():
    failures.append("Negative cost component in scenario table.")
if scenarios["scenario"].duplicated().any():
    failures.append("Duplicate scenario name.")

expected_scenario_order = [
    "FEES_ONLY",
    "BASE",
    "CONSERVATIVE",
    "STRESS",
]
if scenarios["scenario"].tolist() != expected_scenario_order:
    failures.append("Scenario ordering/version contract changed.")

if not scenarios["total_round_trip_usd"].is_monotonic_increasing:
    failures.append("Scenario total costs are not monotonic.")
if int(scenarios["primary_economic_gate"].sum()) != 1:
    failures.append("Primary economic gate must select exactly one scenario.")
if PRIMARY_ECONOMIC_GATE_SCENARIO not in set(scenarios["scenario"]):
    failures.append("Primary economic gate scenario is missing.")

recomputed_total = (
    scenarios[
        [
            "direct_fee_round_trip_usd",
            "spread_round_trip_usd",
            "slippage_round_trip_usd",
            "latency_round_trip_usd",
            "market_impact_round_trip_usd",
            "overnight_fee_round_trip_usd",
        ]
    ].sum(axis=1)
)
if not np.allclose(
    recomputed_total.to_numpy(),
    scenarios["total_round_trip_usd"].to_numpy(),
    atol=1e-12,
):
    failures.append("Scenario components do not reconcile to total cost.")

if not np.allclose(
    (
        scenarios["break_even_index_points"]
        / TICK_SIZE_POINTS
    ).to_numpy(),
    scenarios["break_even_ticks"].to_numpy(),
    atol=1e-12,
):
    failures.append("Tick and index-point break-even conversions disagree.")

if observed_final_test_rows != 8654:
    failures.append(
        "Final-test protection count changed from the Cell 8 contract."
    )


# ------------------------------------------------------------
# 11) Save artifacts and hashes
# ------------------------------------------------------------
parameters.to_csv(
    CELL9_PARAMETERS_PATH,
    index=False,
)
scenarios.to_csv(
    CELL9_SCENARIOS_PATH,
    index=False,
)

artifact_hashes = {
    "input_cell8_assignments_sha256": actual_cell8_hash,
    "input_cell8_audit_sha256": sha256_file(CELL8_AUDIT_PATH),
    "cost_parameters_sha256": sha256_file(CELL9_PARAMETERS_PATH),
    "cost_scenarios_sha256": sha256_file(CELL9_SCENARIOS_PATH),
}

primary_row = scenarios.loc[
    scenarios["scenario"].eq(PRIMARY_ECONOMIC_GATE_SCENARIO)
].iloc[0]

cell9_audit = {
    "audit_written_utc": datetime.now(timezone.utc).isoformat(),
    "policy_version": COST_POLICY_VERSION,
    "status": "PASS" if not failures else "FAIL",
    "upstream_binding": {
        "cell8_policy_version": cell8_audit.get("policy_version"),
        "cell8_status": cell8_audit.get("status"),
        "cell8_decision_rows": expected_decision_rows,
        "cell8_final_test_rows": expected_final_test_rows,
        "cell8_assignments_sha256": actual_cell8_hash,
        "final_test_outcomes_inspected": False,
    },
    "contract_mechanics": {
        "symbol": SYMBOL,
        "currency": CURRENCY,
        "contract_multiplier_usd_per_point": (
            CONTRACT_MULTIPLIER_USD_PER_POINT
        ),
        "tick_size_points": TICK_SIZE_POINTS,
        "tick_value_usd": TICK_VALUE_USD,
        "status": "LOCKED",
    },
    "fee_snapshot": {
        "as_of": SOURCE_SNAPSHOT_DATE,
        "status": "PROVISIONAL",
        "account_assumption": (
            "IBKR non-member MES; <=1,000 monthly contracts; "
            "one contract per side"
        ),
        "ibkr_execution_fee_per_side_usd": (
            IBKR_EXECUTION_FEE_PER_SIDE_USD
        ),
        "cme_exchange_fee_per_side_usd": (
            CME_EXCHANGE_FEE_PER_SIDE_USD
        ),
        "regulatory_fee_per_side_usd": (
            REGULATORY_FEE_PER_SIDE_USD
        ),
        "clearing_fee_per_side_usd": CLEARING_FEE_PER_SIDE_USD,
        "tax_or_entity_adjustment_per_side_usd": (
            TAX_OR_ENTITY_ADJUSTMENT_PER_SIDE_USD
        ),
        "direct_fee_per_side_usd": DIRECT_FEE_PER_SIDE_USD,
        "direct_fee_round_trip_usd": DIRECT_FEE_ROUND_TRIP_USD,
        "historical_fee_vintage_handling": (
            "constant current snapshot for comparable research; "
            "historical fee reconstruction remains OPEN"
        ),
        "actual_statement_reconciliation": "OPEN",
    },
    "execution_cost_contract": {
        "order_style_reference": "marketable entry and exit",
        "quantity_contracts": 1,
        "spread_data_available": False,
        "spread_method": "scenario proxy; current OHLCV has no bid/ask",
        "slippage_method": "scenario proxy pending paper/live fills",
        "latency_method": "scenario proxy pending server telemetry",
        "market_impact_method": (
            "scenario proxy; not identified from 1-minute OHLCV"
        ),
        "primary_economic_gate_scenario": (
            PRIMARY_ECONOMIC_GATE_SCENARIO
        ),
        "primary_round_trip_cost_usd": float(
            primary_row["total_round_trip_usd"]
        ),
        "primary_break_even_ticks": float(
            primary_row["break_even_ticks"]
        ),
        "primary_break_even_index_points": float(
            primary_row["break_even_index_points"]
        ),
        "scenario_status": "PROVISIONAL",
    },
    "research_safety": {
        "economic_label_created": False,
        "future_return_created": False,
        "model_fitted": False,
        "final_test_outcomes_inspected": False,
        "cost_before_final_label_rule_satisfied": True,
    },
    "sources": {
        "cme_contract_specification": CME_CONTRACT_SOURCE,
        "cme_fee_schedule": CME_FEE_SOURCE,
        "ibkr_futures_commissions": IBKR_COMMISSION_SOURCE,
        "ibkr_cme_fee_recovery": IBKR_CME_FEE_SOURCE,
        "source_snapshot_date": SOURCE_SNAPSHOT_DATE,
    },
    "scenario_count": int(len(scenarios)),
    "scenarios": scenarios.to_dict(orient="records"),
    "artifacts": {
        "cost_parameters": str(CELL9_PARAMETERS_PATH),
        "cost_scenarios": str(CELL9_SCENARIOS_PATH),
    },
    "sha256": artifact_hashes,
    "open_items": [
        "Reconcile fees and tax with the user's actual IBKR entity/statement.",
        "Replace spread proxy when historical bid/ask data is available.",
        "Calibrate slippage, latency, and impact from paper/live fills.",
        "Revisit quantity scaling before trading more than one contract.",
    ],
    "failures": failures,
}

with open(CELL9_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        cell9_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 12) Final hard gate and compact output
# ------------------------------------------------------------
if failures:
    print("\nCELL 9 FAILURES")
    print("-" * 72)
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(
        "\nCELL 9 RESEARCH COST MODEL: FAIL\n"
        f"{CELL9_AUDIT_PATH}"
    )

print("\n" + "=" * 72)
print("CELL 9 — RESEARCH COST MODEL CONTRACT")
print("=" * 72)

print("\n[1] UPSTREAM / TEST PROTECTION")
print("Cell 8 rows                  :", f"{len(assignments):,}")
print("Final-test rows protected    :", f"{observed_final_test_rows:,}")
print("Final-test outcomes inspected: False")
print("Cell 8 assignments SHA256    :", actual_cell8_hash)

print("\n[2] MES CONTRACT MECHANICS — LOCKED")
print("Multiplier                    : USD 5.00 / index point")
print("Tick size                     : 0.25 index points")
print("Tick value                    : USD 1.25 / contract")

print("\n[3] DIRECT FEE SNAPSHOT — PROVISIONAL")
print("IBKR execution / side         : USD 0.25")
print("CME exchange / side           : USD 0.35")
print("Regulatory / side             : USD 0.01")
print("Direct fee / side             :", f"USD {DIRECT_FEE_PER_SIDE_USD:.2f}")
print("Direct fee / round trip       :", f"USD {DIRECT_FEE_ROUND_TRIP_USD:.2f}")
print("Snapshot date                 :", SOURCE_SNAPSHOT_DATE)

print("\n[4] ONE-CONTRACT ROUND-TRIP SCENARIOS")
print(
    scenarios[
        [
            "scenario",
            "total_round_trip_usd",
            "break_even_ticks",
            "break_even_index_points",
            "primary_economic_gate",
        ]
    ].to_string(index=False)
)

print("\n[5] PRIMARY ECONOMIC GATE")
print("Scenario                      :", PRIMARY_ECONOMIC_GATE_SCENARIO)
print(
    "Round-trip cost               :",
    f"USD {float(primary_row['total_round_trip_usd']):.3f}",
)
print(
    "Break-even                    :",
    f"{float(primary_row['break_even_ticks']):.3f} ticks / "
    f"{float(primary_row['break_even_index_points']):.3f} points",
)

print("\n[6] SAVED ARTIFACTS")
print("Cost parameters :", CELL9_PARAMETERS_PATH)
print("Cost scenarios  :", CELL9_SCENARIOS_PATH)
print("Cell 9 audit    :", CELL9_AUDIT_PATH)
print("Scenario SHA256 :", artifact_hashes["cost_scenarios_sha256"])

print("\n" + "=" * 72)
print("CELL 9 RESEARCH COST MODEL: PASS")
print("=" * 72)




CELL 9 — RESEARCH COST MODEL CONTRACT

[1] UPSTREAM / TEST PROTECTION
Cell 8 rows                  : 39,847
Final-test rows protected    : 8,654
Final-test outcomes inspected: False
Cell 8 assignments SHA256    : 2e13ee7d1e7de321411604c3500c73e68a080b02fa2983288d41d399aeb43035

[2] MES CONTRACT MECHANICS — LOCKED
Multiplier                    : USD 5.00 / index point
Tick size                     : 0.25 index points
Tick value                    : USD 1.25 / contract

[3] DIRECT FEE SNAPSHOT — PROVISIONAL
IBKR execution / side         : USD 0.25
CME exchange / side           : USD 0.35
Regulatory / side             : USD 0.01
Direct fee / side             : USD 0.61
Direct fee / round trip       : USD 1.22
Snapshot date                 : 2026-08-09

[4] ONE-CONTRACT ROUND-TRIP SCENARIOS
    scenario  total_round_trip_usd  break_even_ticks  break_even_index_points  primary_economic_gate
   FEES_ONLY                 1.220             0.976                    0.244                  False

In [47]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 10 — POINT-IN-TIME +60m ECONOMIC LABELS
# ============================================================
#
# PURPOSE
# -------
# Build auditable +60-minute development labels only after the
# Decision Universe, chronological splits, purging, and cost contract
# have been frozen.
#
# This cell:
# 1) binds to the exact Cell 7 market-data hash,
# 2) binds to the exact Cell 8 assignment hash,
# 3) binds to the exact Cell 9 cost-scenario hash,
# 4) computes close-to-close +60m outcomes for Train/Validation only,
# 5) rejects labels with missing/partial/roll-mixed future paths,
# 6) creates LONG / SHORT / NO_TRADE labels after transaction costs,
# 7) leaves every Final Test outcome SEALED and uncomputed.
#
# IMPORTANT
# ---------
# Future-derived columns created here are TARGETS ONLY. They must
# never enter the online feature matrix or decision-time eligibility.
# ============================================================

import warnings as _warnings

_warnings.filterwarnings(
    "ignore",
    message=r"datetime\.datetime\.utcnow\(\) is deprecated.*",
    category=DeprecationWarning,
)

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1) Paths and version contract
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Quant_Lab")
CLEAN_DIR = PROJECT_DIR / "Data" / "MES_Clean_Pipeline_V1"

MES_15M_PATH = CLEAN_DIR / "MES_2019_2026_15m_clean.parquet"
CELL7_AUDIT_PATH = CLEAN_DIR / "cell7_decision_universe_audit.json"

CELL8_ASSIGNMENTS_PATH = (
    CLEAN_DIR / "cell8_purged_split_assignments_v1.parquet"
)
CELL8_AUDIT_PATH = CLEAN_DIR / "cell8_purged_split_audit.json"

CELL9_SCENARIOS_PATH = CLEAN_DIR / "cell9_cost_scenarios_v1.csv"
CELL9_AUDIT_PATH = CLEAN_DIR / "cell9_cost_model_audit.json"

CELL10_LABELS_PATH = (
    CLEAN_DIR / "cell10_point_in_time_economic_labels_v1.parquet"
)
CELL10_SUMMARY_PATH = (
    CLEAN_DIR / "cell10_development_label_summary_v1.csv"
)
CELL10_UNUSABLE_PATH = (
    CLEAN_DIR / "cell10_unusable_label_events_v1.parquet"
)
CELL10_AUDIT_PATH = CLEAN_DIR / "cell10_economic_label_audit.json"

LABEL_POLICY_VERSION = "MES_V1_ECONOMIC_LABELS_1.0"
EXPECTED_CELL7_POLICY = "MES_V1_DECISION_UNIVERSE_1.0"
EXPECTED_CELL8_POLICY = "MES_V1_PURGED_SPLIT_1.0"
EXPECTED_CELL9_POLICY = "MES_V1_COST_MODEL_1.0"

BAR_MINUTES = 15
LABEL_HORIZON_MINUTES = 60
HORIZON_BAR_COUNT = LABEL_HORIZON_MINUTES // BAR_MINUTES
CONTRACT_MULTIPLIER_USD_PER_POINT = 5.00


# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_name(value):
    return (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


# ------------------------------------------------------------
# 3) Upstream artifact gates
# ------------------------------------------------------------
required_paths = [
    MES_15M_PATH,
    CELL7_AUDIT_PATH,
    CELL8_ASSIGNMENTS_PATH,
    CELL8_AUDIT_PATH,
    CELL9_SCENARIOS_PATH,
    CELL9_AUDIT_PATH,
]
for required_path in required_paths:
    if not required_path.exists():
        raise RuntimeError(
            "CELL 10 STOPPED — missing upstream artifact:\n"
            f"{required_path}\n\n"
            "Run Cell 0 → Cell 9 first."
        )

cell7_audit = load_json(CELL7_AUDIT_PATH)
cell8_audit = load_json(CELL8_AUDIT_PATH)
cell9_audit = load_json(CELL9_AUDIT_PATH)

upstream_specs = [
    (
        "Cell 7",
        cell7_audit,
        EXPECTED_CELL7_POLICY,
    ),
    (
        "Cell 8",
        cell8_audit,
        EXPECTED_CELL8_POLICY,
    ),
    (
        "Cell 9",
        cell9_audit,
        EXPECTED_CELL9_POLICY,
    ),
]

for cell_name, audit, expected_policy in upstream_specs:
    if audit.get("status") != "PASS":
        raise RuntimeError(
            f"CELL 10 STOPPED — {cell_name} status is not PASS."
        )
    if audit.get("failures", []):
        raise RuntimeError(
            f"CELL 10 STOPPED — {cell_name} contains failures."
        )
    if audit.get("policy_version") != expected_policy:
        raise RuntimeError(
            f"CELL 10 STOPPED — unexpected {cell_name} policy:\n"
            f"{audit.get('policy_version')}"
        )

expected_mes_15m_hash = (
    cell7_audit
    .get("sha256", {})
    .get("input_mes_15m_sha256")
)
expected_cell8_hash = (
    cell8_audit
    .get("sha256", {})
    .get("split_assignments_sha256")
)
expected_cell9_hash = (
    cell9_audit
    .get("sha256", {})
    .get("cost_scenarios_sha256")
)

actual_mes_15m_hash = sha256_file(MES_15M_PATH)
actual_cell8_hash = sha256_file(CELL8_ASSIGNMENTS_PATH)
actual_cell9_hash = sha256_file(CELL9_SCENARIOS_PATH)

hash_gates = [
    (
        "MES 15m",
        expected_mes_15m_hash,
        actual_mes_15m_hash,
    ),
    (
        "Cell 8 assignments",
        expected_cell8_hash,
        actual_cell8_hash,
    ),
    (
        "Cell 9 scenarios",
        expected_cell9_hash,
        actual_cell9_hash,
    ),
]

for artifact_name, expected_hash, actual_hash in hash_gates:
    if not expected_hash:
        raise RuntimeError(
            f"CELL 10 STOPPED — no expected hash for {artifact_name}."
        )
    if expected_hash != actual_hash:
        raise RuntimeError(
            f"CELL 10 STOPPED — {artifact_name} hash mismatch.\n"
            f"Audit : {expected_hash}\n"
            f"Actual: {actual_hash}"
        )


# ------------------------------------------------------------
# 4) Load the frozen assignments and cost scenarios
# ------------------------------------------------------------
assignments = pd.read_parquet(CELL8_ASSIGNMENTS_PATH)

required_assignment_columns = {
    "decision_id",
    "decision_time",
    "label_end_time",
    "nyse_session_date",
    "instrument_id",
    "dataset_degraded_utc",
    "outer_partition",
    "purged_before_final_test",
    "outer_modeling_eligible",
}
missing_assignment_columns = (
    required_assignment_columns
    - set(assignments.columns)
)
if missing_assignment_columns:
    raise RuntimeError(
        "CELL 10 STOPPED — missing Cell 8 columns:\n"
        + ", ".join(sorted(missing_assignment_columns))
    )

assignments = assignments.copy()
assignments["decision_time"] = pd.to_datetime(
    assignments["decision_time"],
    utc=True,
    errors="raise",
)
assignments["label_end_time"] = pd.to_datetime(
    assignments["label_end_time"],
    utc=True,
    errors="raise",
)
assignments = (
    assignments
    .sort_values("decision_time")
    .reset_index(drop=True)
)

expected_label_end_time = (
    assignments["decision_time"]
    + pd.Timedelta(minutes=LABEL_HORIZON_MINUTES)
)
if not np.array_equal(
    assignments["label_end_time"].array.asi8,
    expected_label_end_time.array.asi8,
):
    raise RuntimeError(
        "CELL 10 STOPPED — Cell 8 label_end_time is not +60m."
    )

expected_rows = int(
    cell8_audit
    .get("counts", {})
    .get("decision_rows", -1)
)
expected_final_test_rows = int(
    cell8_audit
    .get("counts", {})
    .get("final_test_rows", -1)
)

if len(assignments) != expected_rows:
    raise RuntimeError(
        "CELL 10 STOPPED — assignment row count mismatch."
    )
if assignments["decision_id"].duplicated().any():
    raise RuntimeError(
        "CELL 10 STOPPED — duplicate decision_id."
    )

cost_scenarios = pd.read_csv(CELL9_SCENARIOS_PATH)
required_cost_columns = {
    "scenario",
    "total_round_trip_usd",
    "break_even_ticks",
    "break_even_index_points",
    "primary_economic_gate",
}
missing_cost_columns = required_cost_columns - set(cost_scenarios.columns)
if missing_cost_columns:
    raise RuntimeError(
        "CELL 10 STOPPED — missing Cell 9 cost columns:\n"
        + ", ".join(sorted(missing_cost_columns))
    )

cost_scenarios = cost_scenarios.copy()
cost_scenarios["scenario"] = (
    cost_scenarios["scenario"].astype(str)
)
if cost_scenarios["scenario"].duplicated().any():
    raise RuntimeError(
        "CELL 10 STOPPED — duplicate cost scenario."
    )
if int(cost_scenarios["primary_economic_gate"].sum()) != 1:
    raise RuntimeError(
        "CELL 10 STOPPED — expected exactly one primary cost scenario."
    )

primary_scenario = str(
    cell9_audit
    .get("execution_cost_contract", {})
    .get("primary_economic_gate_scenario")
)
primary_cost_row = cost_scenarios.loc[
    cost_scenarios["scenario"].eq(primary_scenario)
]
if len(primary_cost_row) != 1:
    raise RuntimeError(
        "CELL 10 STOPPED — primary cost scenario is missing."
    )
primary_cost_row = primary_cost_row.iloc[0]


# ------------------------------------------------------------
# 5) Load the clean 15-minute reference prices
# ------------------------------------------------------------
price_columns = [
    "close",
    "decision_time",
    "instrument_id",
    "active_1m_count",
    "bar_complete_15m",
    "crosses_roll",
    "dataset_degraded_utc",
]
prices = pd.read_parquet(
    MES_15M_PATH,
    columns=price_columns,
)
prices = prices.copy()
prices["decision_time"] = pd.to_datetime(
    prices["decision_time"],
    utc=True,
    errors="raise",
)

if prices["decision_time"].duplicated().any():
    raise RuntimeError(
        "CELL 10 STOPPED — duplicate decision_time in 15m prices."
    )
if prices["close"].isna().any():
    raise RuntimeError(
        "CELL 10 STOPPED — missing close in 15m prices."
    )
if (prices["close"] <= 0).any():
    raise RuntimeError(
        "CELL 10 STOPPED — non-positive close in 15m prices."
    )

price_by_time = (
    prices
    .sort_values("decision_time")
    .set_index("decision_time")
)


# ------------------------------------------------------------
# 6) Map entry and future path for development rows only
#
# Final Test rows are deliberately excluded from every price lookup.
# ------------------------------------------------------------
labels = assignments.copy()

final_test_mask = labels["outer_partition"].eq("FINAL_TEST")
development_mask = labels["outer_partition"].isin(
    ["TRAIN", "VALIDATION"]
)

observed_final_test_rows = int(final_test_mask.sum())
if observed_final_test_rows != expected_final_test_rows:
    raise RuntimeError(
        "CELL 10 STOPPED — final-test count changed."
    )
if int((~development_mask & ~final_test_mask).sum()) != 0:
    raise RuntimeError(
        "CELL 10 STOPPED — unexpected outer partition."
    )

n_rows = len(labels)
dev_positions = np.flatnonzero(development_mask.to_numpy())
dev_times = labels.loc[
    development_mask,
    "decision_time",
]
entry_instrument_expected = (
    pd.to_numeric(
        labels.loc[development_mask, "instrument_id"],
        errors="raise",
    )
    .to_numpy(dtype=float)
)

entry_close = np.full(n_rows, np.nan, dtype=float)
exit_close = np.full(n_rows, np.nan, dtype=float)
entry_present = np.zeros(n_rows, dtype=bool)
entry_instrument_match = np.zeros(n_rows, dtype=bool)
target_present = np.zeros(n_rows, dtype=bool)
horizon_present_count = np.zeros(n_rows, dtype=np.int16)
horizon_complete_count = np.zeros(n_rows, dtype=np.int16)
horizon_crosses_roll = np.zeros(n_rows, dtype=bool)
horizon_instrument_changed = np.zeros(n_rows, dtype=bool)
horizon_degraded_context = np.zeros(n_rows, dtype=bool)

for step in range(HORIZON_BAR_COUNT + 1):
    lookup_times = pd.DatetimeIndex(
        dev_times
        + pd.Timedelta(minutes=step * BAR_MINUTES)
    )
    observed = price_by_time.reindex(lookup_times)

    observed_instrument = pd.to_numeric(
        observed["instrument_id"],
        errors="coerce",
    ).to_numpy(dtype=float)
    observed_close = pd.to_numeric(
        observed["close"],
        errors="coerce",
    ).to_numpy(dtype=float)

    present = (
        np.isfinite(observed_instrument)
        & np.isfinite(observed_close)
    )
    complete = (
        present
        & observed["bar_complete_15m"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
        & observed["active_1m_count"]
        .fillna(0)
        .eq(BAR_MINUTES)
        .to_numpy()
    )
    crosses_roll = (
        observed["crosses_roll"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )
    degraded = (
        observed["dataset_degraded_utc"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )

    if step == 0:
        entry_close[dev_positions] = observed_close
        entry_present[dev_positions] = present
        entry_instrument_match[dev_positions] = (
            present
            & np.equal(
                observed_instrument,
                entry_instrument_expected,
            )
        )
    else:
        horizon_present_count[dev_positions] += present.astype(np.int16)
        horizon_complete_count[dev_positions] += complete.astype(np.int16)
        horizon_crosses_roll[dev_positions] |= crosses_roll
        horizon_instrument_changed[dev_positions] |= (
            present
            & ~np.equal(
                observed_instrument,
                entry_instrument_expected,
            )
        )
        horizon_degraded_context[dev_positions] |= degraded

        if step == HORIZON_BAR_COUNT:
            exit_close[dev_positions] = observed_close
            target_present[dev_positions] = present

labels["entry_reference_close"] = entry_close
labels["exit_reference_close_60m"] = exit_close
labels["horizon_bars_expected"] = HORIZON_BAR_COUNT
labels["horizon_bars_present"] = horizon_present_count
labels["horizon_bars_complete"] = horizon_complete_count
labels["horizon_crosses_roll"] = horizon_crosses_roll
labels["horizon_instrument_changed"] = horizon_instrument_changed
labels["horizon_degraded_context"] = horizon_degraded_context


# ------------------------------------------------------------
# 7) One primary label-status reason per row
# ------------------------------------------------------------
label_status = np.full(n_rows, "USABLE", dtype=object)
label_status[final_test_mask.to_numpy()] = "SEALED_FINAL_TEST"


def mark_status(mask, reason):
    active = mask & np.equal(label_status, "USABLE")
    label_status[active] = reason


dev = development_mask.to_numpy()
mark_status(
    dev & ~entry_present,
    "ENTRY_BAR_MISSING",
)
mark_status(
    dev & entry_present & ~entry_instrument_match,
    "ENTRY_INSTRUMENT_MISMATCH",
)
mark_status(
    dev & ~target_present,
    "TARGET_BAR_MISSING",
)
mark_status(
    dev & (horizon_present_count != HORIZON_BAR_COUNT),
    "HORIZON_BAR_MISSING",
)
mark_status(
    dev & (horizon_complete_count != HORIZON_BAR_COUNT),
    "HORIZON_BAR_PARTIAL",
)
mark_status(
    dev & horizon_crosses_roll,
    "HORIZON_CROSSES_ROLL",
)
mark_status(
    dev & horizon_instrument_changed,
    "HORIZON_INSTRUMENT_CHANGED",
)
mark_status(
    dev
    & (
        ~np.isfinite(entry_close)
        | ~np.isfinite(exit_close)
        | (entry_close <= 0)
        | (exit_close <= 0)
    ),
    "INVALID_REFERENCE_PRICE",
)

labels["label_status"] = label_status
labels["label_usable"] = labels["label_status"].eq("USABLE")

usable_mask = labels["label_usable"].to_numpy()
unusable_development_mask = dev & ~usable_mask


# ------------------------------------------------------------
# 8) Gross +60m outcome — usable development rows only
# ------------------------------------------------------------
gross_move_points = np.full(n_rows, np.nan, dtype=float)
gross_move_usd = np.full(n_rows, np.nan, dtype=float)

gross_move_points[usable_mask] = (
    exit_close[usable_mask]
    - entry_close[usable_mask]
)
gross_move_usd[usable_mask] = (
    gross_move_points[usable_mask]
    * CONTRACT_MULTIPLIER_USD_PER_POINT
)

labels["gross_move_points_60m"] = gross_move_points
labels["gross_move_usd_60m"] = gross_move_usd
labels["gross_direction_60m"] = "UNAVAILABLE"
labels.loc[final_test_mask, "gross_direction_60m"] = "SEALED"
labels.loc[
    labels["label_usable"]
    & labels["gross_move_points_60m"].gt(0),
    "gross_direction_60m",
] = "UP"
labels.loc[
    labels["label_usable"]
    & labels["gross_move_points_60m"].lt(0),
    "gross_direction_60m",
] = "DOWN"
labels.loc[
    labels["label_usable"]
    & labels["gross_move_points_60m"].eq(0),
    "gross_direction_60m",
] = "FLAT"


# ------------------------------------------------------------
# 9) Scenario-specific economic labels
#
# LONG     when gross move > +round-trip break-even points
# SHORT    when gross move < -round-trip break-even points
# NO_TRADE otherwise
# ------------------------------------------------------------
scenario_label_columns = []
scenario_long_net_columns = []
scenario_short_net_columns = []

for scenario in cost_scenarios.itertuples(index=False):
    scenario_name = str(scenario.scenario)
    suffix = safe_name(scenario_name)
    cost_usd = float(scenario.total_round_trip_usd)
    cost_points = float(scenario.break_even_index_points)

    long_net_column = f"long_net_usd_{suffix}"
    short_net_column = f"short_net_usd_{suffix}"
    label_column = f"economic_label_{suffix}"

    labels[long_net_column] = np.nan
    labels[short_net_column] = np.nan
    labels[label_column] = "UNAVAILABLE"
    labels.loc[final_test_mask, label_column] = "SEALED"

    labels.loc[
        labels["label_usable"],
        long_net_column,
    ] = (
        labels.loc[
            labels["label_usable"],
            "gross_move_usd_60m",
        ]
        - cost_usd
    )
    labels.loc[
        labels["label_usable"],
        short_net_column,
    ] = (
        -labels.loc[
            labels["label_usable"],
            "gross_move_usd_60m",
        ]
        - cost_usd
    )

    labels.loc[
        labels["label_usable"],
        label_column,
    ] = "NO_TRADE"
    labels.loc[
        labels["label_usable"]
        & labels["gross_move_points_60m"].gt(cost_points),
        label_column,
    ] = "LONG"
    labels.loc[
        labels["label_usable"]
        & labels["gross_move_points_60m"].lt(-cost_points),
        label_column,
    ] = "SHORT"

    scenario_label_columns.append(label_column)
    scenario_long_net_columns.append(long_net_column)
    scenario_short_net_columns.append(short_net_column)

primary_suffix = safe_name(primary_scenario)
primary_label_column = f"economic_label_{primary_suffix}"
primary_long_net_column = f"long_net_usd_{primary_suffix}"
primary_short_net_column = f"short_net_usd_{primary_suffix}"

labels["economic_label_primary"] = labels[primary_label_column]
labels["primary_net_if_traded_usd"] = np.nan
labels.loc[
    labels["economic_label_primary"].eq("LONG"),
    "primary_net_if_traded_usd",
] = labels.loc[
    labels["economic_label_primary"].eq("LONG"),
    primary_long_net_column,
]
labels.loc[
    labels["economic_label_primary"].eq("SHORT"),
    "primary_net_if_traded_usd",
] = labels.loc[
    labels["economic_label_primary"].eq("SHORT"),
    primary_short_net_column,
]
labels.loc[
    labels["economic_label_primary"].eq("NO_TRADE"),
    "primary_net_if_traded_usd",
] = 0.0


# ------------------------------------------------------------
# 10) Summaries — development only; test distribution stays sealed
# ------------------------------------------------------------
development_summary = (
    labels.loc[
        development_mask,
        [
            "outer_partition",
            "label_status",
            "economic_label_primary",
        ],
    ]
    .groupby(
        [
            "outer_partition",
            "label_status",
            "economic_label_primary",
        ],
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(
        [
            "outer_partition",
            "label_status",
            "economic_label_primary",
        ]
    )
    .reset_index(drop=True)
)

unusable_columns = [
    "decision_id",
    "decision_time",
    "label_end_time",
    "outer_partition",
    "instrument_id",
    "label_status",
    "horizon_bars_expected",
    "horizon_bars_present",
    "horizon_bars_complete",
    "horizon_crosses_roll",
    "horizon_instrument_changed",
    "horizon_degraded_context",
]
unusable_events = labels.loc[
    unusable_development_mask,
    unusable_columns,
].copy()


# ------------------------------------------------------------
# 11) Final hard gates
# ------------------------------------------------------------
failures = []

development_rows = int(development_mask.sum())
usable_development_rows = int(
    (development_mask & labels["label_usable"]).sum()
)
unusable_development_rows = int(
    development_rows - usable_development_rows
)

if len(labels) != expected_rows:
    failures.append("Cell 10 changed the assignment row count.")
if labels["decision_id"].duplicated().any():
    failures.append("Duplicate decision_id in Cell 10 labels.")
if not labels["decision_time"].is_monotonic_increasing:
    failures.append("Cell 10 labels are not time-sorted.")
if usable_development_rows + unusable_development_rows != development_rows:
    failures.append("Development label accounting does not reconcile.")
if len(unusable_events) != unusable_development_rows:
    failures.append("Unusable-event artifact count does not reconcile.")

if labels.loc[final_test_mask, "label_usable"].any():
    failures.append("Final Test row became label-usable.")
if not labels.loc[
    final_test_mask,
    "label_status",
].eq("SEALED_FINAL_TEST").all():
    failures.append("Final Test label status is not SEALED_FINAL_TEST.")
if not labels.loc[
    final_test_mask,
    "economic_label_primary",
].eq("SEALED").all():
    failures.append("Final Test primary labels are not SEALED.")

sensitive_numeric_columns = [
    "entry_reference_close",
    "exit_reference_close_60m",
    "gross_move_points_60m",
    "gross_move_usd_60m",
    "primary_net_if_traded_usd",
] + scenario_long_net_columns + scenario_short_net_columns

if labels.loc[
    final_test_mask,
    sensitive_numeric_columns,
].notna().any().any():
    failures.append("A future-derived numeric value entered Final Test rows.")

if labels.loc[
    labels["label_usable"],
    [
        "entry_reference_close",
        "exit_reference_close_60m",
        "gross_move_points_60m",
        "gross_move_usd_60m",
    ],
].isna().any().any():
    failures.append("Usable label has a missing outcome value.")

if not labels.loc[
    labels["label_usable"],
    "horizon_bars_present",
].eq(HORIZON_BAR_COUNT).all():
    failures.append("Usable label has a missing horizon bar.")
if not labels.loc[
    labels["label_usable"],
    "horizon_bars_complete",
].eq(HORIZON_BAR_COUNT).all():
    failures.append("Usable label has a partial horizon bar.")
if labels.loc[
    labels["label_usable"],
    "horizon_crosses_roll",
].any():
    failures.append("Usable label crosses a roll bar.")
if labels.loc[
    labels["label_usable"],
    "horizon_instrument_changed",
].any():
    failures.append("Usable label changes instrument inside +60m.")

allowed_development_labels = {
    "LONG",
    "SHORT",
    "NO_TRADE",
    "UNAVAILABLE",
}
observed_development_labels = set(
    labels.loc[
        development_mask,
        "economic_label_primary",
    ].unique()
)
if not observed_development_labels.issubset(allowed_development_labels):
    failures.append("Unexpected primary economic label.")

for scenario in cost_scenarios.itertuples(index=False):
    suffix = safe_name(scenario.scenario)
    cost_points = float(scenario.break_even_index_points)
    label_column = f"economic_label_{suffix}"
    long_net_column = f"long_net_usd_{suffix}"
    short_net_column = f"short_net_usd_{suffix}"

    usable = labels["label_usable"]
    long_mask = usable & labels[label_column].eq("LONG")
    short_mask = usable & labels[label_column].eq("SHORT")
    no_trade_mask = usable & labels[label_column].eq("NO_TRADE")

    if not labels.loc[
        long_mask,
        "gross_move_points_60m",
    ].gt(cost_points).all():
        failures.append(f"{scenario.scenario}: invalid LONG threshold.")
    if not labels.loc[
        short_mask,
        "gross_move_points_60m",
    ].lt(-cost_points).all():
        failures.append(f"{scenario.scenario}: invalid SHORT threshold.")
    if not labels.loc[
        long_mask,
        long_net_column,
    ].gt(0).all():
        failures.append(f"{scenario.scenario}: LONG net is not positive.")
    if not labels.loc[
        short_mask,
        short_net_column,
    ].gt(0).all():
        failures.append(f"{scenario.scenario}: SHORT net is not positive.")

    no_trade_gross = labels.loc[
        no_trade_mask,
        "gross_move_points_60m",
    ]
    if not (
        no_trade_gross.ge(-cost_points)
        & no_trade_gross.le(cost_points)
    ).all():
        failures.append(f"{scenario.scenario}: invalid NO_TRADE band.")

trade_mask = labels["economic_label_primary"].isin(
    ["LONG", "SHORT"]
)
if not labels.loc[
    trade_mask,
    "primary_net_if_traded_usd",
].gt(0).all():
    failures.append("Primary traded labels do not have positive net value.")
if not labels.loc[
    labels["economic_label_primary"].eq("NO_TRADE"),
    "primary_net_if_traded_usd",
].eq(0).all():
    failures.append("Primary NO_TRADE net value is not zero.")


# ------------------------------------------------------------
# 12) Save versioned artifacts and hashes
# ------------------------------------------------------------
labels.to_parquet(
    CELL10_LABELS_PATH,
    index=False,
)
development_summary.to_csv(
    CELL10_SUMMARY_PATH,
    index=False,
)
unusable_events.to_parquet(
    CELL10_UNUSABLE_PATH,
    index=False,
)

artifact_hashes = {
    "input_mes_15m_sha256": actual_mes_15m_hash,
    "input_cell8_assignments_sha256": actual_cell8_hash,
    "input_cell8_audit_sha256": sha256_file(CELL8_AUDIT_PATH),
    "input_cell9_scenarios_sha256": actual_cell9_hash,
    "input_cell9_audit_sha256": sha256_file(CELL9_AUDIT_PATH),
    "economic_labels_sha256": sha256_file(CELL10_LABELS_PATH),
    "development_summary_sha256": sha256_file(CELL10_SUMMARY_PATH),
    "unusable_events_sha256": sha256_file(CELL10_UNUSABLE_PATH),
}

primary_label_counts = (
    labels.loc[
        labels["label_usable"],
        "economic_label_primary",
    ]
    .value_counts()
    .reindex(
        ["LONG", "SHORT", "NO_TRADE"],
        fill_value=0,
    )
    .astype(int)
    .to_dict()
)
unusable_reason_counts = (
    labels.loc[
        unusable_development_mask,
        "label_status",
    ]
    .value_counts()
    .sort_index()
    .astype(int)
    .to_dict()
)

cell10_audit = {
    "audit_written_utc": datetime.now(timezone.utc).isoformat(),
    "policy_version": LABEL_POLICY_VERSION,
    "status": "PASS" if not failures else "FAIL",
    "upstream_binding": {
        "cell7_policy_version": cell7_audit.get("policy_version"),
        "cell8_policy_version": cell8_audit.get("policy_version"),
        "cell9_policy_version": cell9_audit.get("policy_version"),
        "mes_15m_sha256": actual_mes_15m_hash,
        "cell8_assignments_sha256": actual_cell8_hash,
        "cell9_cost_scenarios_sha256": actual_cell9_hash,
    },
    "label_contract": {
        "instrument": "MES continuous active contract",
        "decision_price": "close of completed 15m bar at decision_time",
        "exit_price": "close of completed 15m bar at decision_time +60m",
        "horizon_minutes": LABEL_HORIZON_MINUTES,
        "future_15m_bars_required": HORIZON_BAR_COUNT,
        "future_bar_completeness_required": True,
        "same_instrument_through_horizon_required": True,
        "roll_crossing_allowed": False,
        "degraded_date_auto_exclusion": False,
        "degraded_date_retained_as_context": True,
        "economic_label_rule": (
            "LONG if gross_points > cost_points; SHORT if gross_points "
            "< -cost_points; otherwise NO_TRADE"
        ),
        "threshold_strict": True,
        "contract_multiplier_usd_per_point": (
            CONTRACT_MULTIPLIER_USD_PER_POINT
        ),
        "primary_cost_scenario": primary_scenario,
        "primary_round_trip_cost_usd": float(
            primary_cost_row["total_round_trip_usd"]
        ),
        "primary_break_even_ticks": float(
            primary_cost_row["break_even_ticks"]
        ),
        "primary_break_even_index_points": float(
            primary_cost_row["break_even_index_points"]
        ),
    },
    "counts": {
        "decision_rows": int(len(labels)),
        "development_rows": development_rows,
        "train_rows": int(
            labels["outer_partition"].eq("TRAIN").sum()
        ),
        "validation_rows": int(
            labels["outer_partition"].eq("VALIDATION").sum()
        ),
        "final_test_rows_sealed": observed_final_test_rows,
        "usable_development_labels": usable_development_rows,
        "unusable_development_labels": unusable_development_rows,
        "primary_label_counts_development_only": primary_label_counts,
        "unusable_reason_counts_development_only": (
            unusable_reason_counts
        ),
        "usable_degraded_context_rows": int(
            labels.loc[
                labels["label_usable"],
                "horizon_degraded_context",
            ].sum()
        ),
    },
    "point_in_time_safety": {
        "future_fields_are_targets_only": True,
        "future_fields_allowed_as_features": False,
        "future_quality_used_for_decision_eligibility": False,
        "future_quality_used_for_label_usability_only": True,
        "final_test_price_lookup_performed": False,
        "final_test_outcomes_computed": False,
        "final_test_label_distribution_inspected": False,
        "model_fitted": False,
    },
    "cost_scenarios": cost_scenarios[
        [
            "scenario",
            "total_round_trip_usd",
            "break_even_ticks",
            "break_even_index_points",
            "primary_economic_gate",
        ]
    ].to_dict(orient="records"),
    "artifacts": {
        "economic_labels": str(CELL10_LABELS_PATH),
        "development_label_summary": str(CELL10_SUMMARY_PATH),
        "unusable_label_events": str(CELL10_UNUSABLE_PATH),
    },
    "sha256": artifact_hashes,
    "failures": failures,
}

with open(CELL10_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        cell10_audit,
        f,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 13) Final hard gate and compact output
# ------------------------------------------------------------
if failures:
    print("\nCELL 10 FAILURES")
    print("-" * 72)
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(
        "\nCELL 10 POINT-IN-TIME ECONOMIC LABELS: FAIL\n"
        f"{CELL10_AUDIT_PATH}"
    )

print("\n" + "=" * 72)
print("CELL 10 — POINT-IN-TIME +60m ECONOMIC LABELS")
print("=" * 72)

print("\n[1] UPSTREAM BINDING")
print("Decision rows                 :", f"{len(labels):,}")
print("Cell 8 assignments SHA256     :", actual_cell8_hash)
print("Cell 9 cost scenarios SHA256  :", actual_cell9_hash)

print("\n[2] LABEL CONTRACT")
print("Reference                     : completed 15m close → +60m close")
print("Required future bars          :", HORIZON_BAR_COUNT)
print("Same instrument required      : True")
print("Roll crossing allowed         : False")
print("Primary cost scenario         :", primary_scenario)
print(
    "Primary break-even           :",
    f"{float(primary_cost_row['break_even_ticks']):.3f} ticks / "
    f"{float(primary_cost_row['break_even_index_points']):.3f} points",
)

print("\n[3] DEVELOPMENT LABEL AVAILABILITY")
print("Train + Validation rows       :", f"{development_rows:,}")
print("Usable labels                 :", f"{usable_development_rows:,}")
print("Unusable labels               :", f"{unusable_development_rows:,}")
print("Unusable reasons              :", unusable_reason_counts)

print("\n[4] PRIMARY LABEL COUNTS — DEVELOPMENT ONLY")
for label_name in ["LONG", "SHORT", "NO_TRADE"]:
    print(
        f"{label_name:30s}:",
        f"{primary_label_counts[label_name]:,}",
    )

print("\n[5] FINAL TEST SAFETY")
print("Final-test rows               :", f"{observed_final_test_rows:,}")
print("Price lookup performed        : False")
print("Outcomes computed             : False")
print("Labels                        : SEALED")

print("\n[6] SAVED ARTIFACTS")
print("Economic labels :", CELL10_LABELS_PATH)
print("Label summary   :", CELL10_SUMMARY_PATH)
print("Unusable events :", CELL10_UNUSABLE_PATH)
print("Cell 10 audit   :", CELL10_AUDIT_PATH)
print("Labels SHA256   :", artifact_hashes["economic_labels_sha256"])

print("\n" + "=" * 72)
print("CELL 10 POINT-IN-TIME ECONOMIC LABELS: PASS")
print("=" * 72)




CELL 10 — POINT-IN-TIME +60m ECONOMIC LABELS

[1] UPSTREAM BINDING
Decision rows                 : 39,847
Cell 8 assignments SHA256     : 2e13ee7d1e7de321411604c3500c73e68a080b02fa2983288d41d399aeb43035
Cell 9 cost scenarios SHA256  : 2248d59ff32361dff9c5df94bfdf8d7ad6942ee50ef3d6e1c6a3731779aeff4f

[2] LABEL CONTRACT
Reference                     : completed 15m close → +60m close
Required future bars          : 4
Same instrument required      : True
Roll crossing allowed         : False
Primary cost scenario         : CONSERVATIVE
Primary break-even           : 3.976 ticks / 0.994 points

[3] DEVELOPMENT LABEL AVAILABILITY
Train + Validation rows       : 31,193
Usable labels                 : 31,165
Unusable labels               : 28
Unusable reasons              : {'HORIZON_BAR_PARTIAL': 28}

[4] PRIMARY LABEL COUNTS — DEVELOPMENT ONLY
LONG                          : 15,188
SHORT                         : 13,147
NO_TRADE                      : 2,830

[5] FINAL TEST SAFETY
Final-tes

In [48]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 11 — COST TEMPORALITY REGISTRY
# ============================================================
#
# PURPOSE
# -------
# Make the time meaning of Cell 9 explicit without inventing historical
# fees. Cell 9 remains a dated current-deployment cost snapshot. This
# cell creates a semantic sidecar that distinguishes:
#
# - CURRENT_DEPLOYMENT_COUNTERFACTUAL
# - STRESS_COUNTERFACTUAL
# - HISTORICAL_VINTAGE (OPEN; unavailable until sourced)
#
# No price, return, label distribution, prediction, or final-test
# outcome is read in this cell.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1) Paths and policy
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Quant_Lab")
CLEAN_DIR = PROJECT_DIR / "Data" / "MES_Clean_Pipeline_V1"

CELL9_PARAMETERS_PATH = CLEAN_DIR / "cell9_cost_parameters_v1.csv"
CELL9_SCENARIOS_PATH = CLEAN_DIR / "cell9_cost_scenarios_v1.csv"
CELL9_AUDIT_PATH = CLEAN_DIR / "cell9_cost_model_audit.json"

CELL11_SEMANTIC_SCENARIOS_PATH = (
    CLEAN_DIR / "cell11_cost_scenarios_semantic_v1.csv"
)
CELL11_VINTAGE_REGISTRY_PATH = (
    CLEAN_DIR / "cell11_fee_vintage_registry_v1.csv"
)
CELL11_AUDIT_PATH = CLEAN_DIR / "cell11_cost_temporality_audit.json"

COST_TEMPORALITY_POLICY_VERSION = "MES_V1_COST_TEMPORALITY_1.0"
EXPECTED_CELL9_POLICY = "MES_V1_COST_MODEL_1.0"


# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ------------------------------------------------------------
# 3) Upstream gates
# ------------------------------------------------------------
required_paths = [
    CELL9_PARAMETERS_PATH,
    CELL9_SCENARIOS_PATH,
    CELL9_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(
        "CELL 11 STOPPED — missing Cell 9 artifacts:\n"
        + "\n".join(missing_paths)
        + "\n\nRun the corrected Cell 9 first."
    )

cell9_audit = load_json(CELL9_AUDIT_PATH)
if cell9_audit.get("status") != "PASS":
    raise RuntimeError("CELL 11 STOPPED — Cell 9 audit is not PASS.")
if cell9_audit.get("failures", []):
    raise RuntimeError("CELL 11 STOPPED — Cell 9 audit has failures.")
if cell9_audit.get("policy_version") != EXPECTED_CELL9_POLICY:
    raise RuntimeError(
        "CELL 11 STOPPED — unexpected Cell 9 policy version: "
        f"{cell9_audit.get('policy_version')}"
    )

expected_scenario_hash = (
    cell9_audit.get("sha256", {}).get("cost_scenarios_sha256")
)
actual_scenario_hash = sha256_file(CELL9_SCENARIOS_PATH)
if not expected_scenario_hash or expected_scenario_hash != actual_scenario_hash:
    raise RuntimeError(
        "CELL 11 STOPPED — Cell 9 scenario hash mismatch.\n"
        f"Audit : {expected_scenario_hash}\n"
        f"Actual: {actual_scenario_hash}"
    )

expected_parameter_hash = (
    cell9_audit.get("sha256", {}).get("cost_parameters_sha256")
)
actual_parameter_hash = sha256_file(CELL9_PARAMETERS_PATH)
if not expected_parameter_hash or expected_parameter_hash != actual_parameter_hash:
    raise RuntimeError(
        "CELL 11 STOPPED — Cell 9 parameter hash mismatch.\n"
        f"Audit : {expected_parameter_hash}\n"
        f"Actual: {actual_parameter_hash}"
    )


# ------------------------------------------------------------
# 4) Build explicit scenario semantics
# ------------------------------------------------------------
scenarios = pd.read_csv(CELL9_SCENARIOS_PATH)
parameters = pd.read_csv(CELL9_PARAMETERS_PATH)

required_scenario_columns = {
    "scenario",
    "status",
    "total_round_trip_usd",
    "direct_fee_round_trip_usd",
    "break_even_ticks",
    "break_even_index_points",
    "primary_economic_gate",
}
missing_columns = required_scenario_columns - set(scenarios.columns)
if missing_columns:
    raise RuntimeError(
        "CELL 11 STOPPED — missing cost-scenario columns:\n"
        + ", ".join(sorted(missing_columns))
    )

source_snapshot_date = str(
    cell9_audit.get("sources", {}).get("source_snapshot_date")
)
if source_snapshot_date in {"None", "", "nan"}:
    raise RuntimeError("CELL 11 STOPPED — Cell 9 source date is missing.")

semantic = scenarios.copy()
semantic["cost_view"] = np.where(
    semantic["scenario"].astype(str).eq("STRESS"),
    "STRESS",
    "CURRENT_DEPLOYMENT",
)
semantic["application_mode"] = "COUNTERFACTUAL_ALL_DEVELOPMENT_DATES"
semantic["fee_vintage_id"] = (
    "CURRENT_SNAPSHOT_" + source_snapshot_date.replace("-", "_")
)
semantic["source_observed_as_of"] = source_snapshot_date
semantic["historical_actual_claim"] = False
semantic["historical_matching_allowed"] = False
semantic["labels_allowed"] = True
semantic["semantic_status"] = "PROVISIONAL"
semantic["label_semantics"] = np.where(
    semantic["cost_view"].eq("STRESS"),
    "STRESS_COUNTERFACTUAL",
    "CURRENT_DEPLOYMENT_COUNTERFACTUAL",
)

semantic = semantic.sort_values(
    ["primary_economic_gate", "scenario"],
    ascending=[False, True],
).reset_index(drop=True)


# ------------------------------------------------------------
# 5) Effective-dated vintage registry
#
# The current snapshot is evidence of what was observed on one date.
# It is not assigned an invented historical effective interval.
# ------------------------------------------------------------
fee_sources = cell9_audit.get("sources", {})
fee_snapshot = cell9_audit.get("fee_snapshot", {})

vintage_rows = [
    {
        "vintage_id": "CURRENT_SNAPSHOT_" + source_snapshot_date.replace("-", "_"),
        "cost_view": "CURRENT_DEPLOYMENT",
        "observed_as_of": source_snapshot_date,
        "effective_from": pd.NA,
        "effective_to": pd.NA,
        "broker_entity": "IBKR_ENTITY_UNCONFIRMED",
        "pricing_plan": "MES_NON_MEMBER_LE_1000_MONTHLY_CONTRACTS",
        "volume_tier": "LE_1000_MONTHLY_CONTRACTS",
        "ibkr_execution_fee_per_side_usd": fee_snapshot.get(
            "ibkr_execution_fee_per_side_usd"
        ),
        "cme_exchange_fee_per_side_usd": fee_snapshot.get(
            "cme_exchange_fee_per_side_usd"
        ),
        "regulatory_fee_per_side_usd": fee_snapshot.get(
            "regulatory_fee_per_side_usd"
        ),
        "clearing_fee_per_side_usd": fee_snapshot.get(
            "clearing_fee_per_side_usd"
        ),
        "tax_or_entity_adjustment_per_side_usd": fee_snapshot.get(
            "tax_or_entity_adjustment_per_side_usd"
        ),
        "source_cme_fee_schedule": fee_sources.get("cme_fee_schedule"),
        "source_ibkr_commissions": fee_sources.get("ibkr_futures_commissions"),
        "source_ibkr_cme_recovery": fee_sources.get("ibkr_cme_fee_recovery"),
        "source_document_sha256": pd.NA,
        "eligible_for_historical_matching": False,
        "status": "PROVISIONAL_CURRENT_SNAPSHOT_ONLY",
        "notes": (
            "Observed current snapshot only; effective start/end and actual "
            "IBKR entity remain unverified."
        ),
    },
    {
        "vintage_id": "HISTORICAL_VINTAGE_UNAVAILABLE",
        "cost_view": "HISTORICAL_VINTAGE",
        "observed_as_of": pd.NA,
        "effective_from": pd.NA,
        "effective_to": pd.NA,
        "broker_entity": "OPEN",
        "pricing_plan": "OPEN",
        "volume_tier": "CAUSAL_MONTH_TO_DATE_VOLUME_REQUIRED",
        "ibkr_execution_fee_per_side_usd": np.nan,
        "cme_exchange_fee_per_side_usd": np.nan,
        "regulatory_fee_per_side_usd": np.nan,
        "clearing_fee_per_side_usd": np.nan,
        "tax_or_entity_adjustment_per_side_usd": np.nan,
        "source_cme_fee_schedule": "HISTORICAL_ARCHIVE_REQUIRED",
        "source_ibkr_commissions": "HISTORICAL_ARCHIVE_REQUIRED",
        "source_ibkr_cme_recovery": "HISTORICAL_ARCHIVE_REQUIRED",
        "source_document_sha256": pd.NA,
        "eligible_for_historical_matching": False,
        "status": "OPEN",
        "notes": (
            "No historical numeric value is invented. Historical labels stay "
            "disabled until sourced, non-overlapping vintages cover every "
            "development decision date."
        ),
    },
]
vintages = pd.DataFrame(vintage_rows)


# ------------------------------------------------------------
# 6) Hard semantic gates
# ------------------------------------------------------------
failures = []

allowed_views = {"CURRENT_DEPLOYMENT", "STRESS"}
if not set(semantic["cost_view"]).issubset(allowed_views):
    failures.append("Unexpected cost_view in semantic scenarios.")
if int(semantic["primary_economic_gate"].sum()) != 1:
    failures.append("Expected exactly one primary economic gate.")

primary = semantic.loc[semantic["primary_economic_gate"]]
if len(primary) != 1 or not primary["cost_view"].eq("CURRENT_DEPLOYMENT").all():
    failures.append("Primary gate is not exactly one CURRENT_DEPLOYMENT row.")
if semantic["historical_actual_claim"].astype(bool).any():
    failures.append("A current/stress scenario claims historical actuality.")
if semantic["historical_matching_allowed"].astype(bool).any():
    failures.append("Current snapshot was enabled for historical matching.")

numeric_cost_columns = [
    "total_round_trip_usd",
    "break_even_ticks",
    "break_even_index_points",
]
numeric_costs = semantic[numeric_cost_columns].apply(
    pd.to_numeric,
    errors="coerce",
)
if not np.isfinite(numeric_costs.to_numpy(dtype=float)).all():
    failures.append("Non-finite scenario cost detected.")
if numeric_costs.lt(0).any().any():
    failures.append("Negative scenario cost detected.")

stress_total = semantic.loc[
    semantic["cost_view"].eq("STRESS"),
    "total_round_trip_usd",
]
primary_total = float(primary.iloc[0]["total_round_trip_usd"])
if stress_total.empty or not stress_total.ge(primary_total).all():
    failures.append("Stress cost is below the primary current cost.")

historical_rows = vintages["cost_view"].eq("HISTORICAL_VINTAGE")
historical_numeric = vintages.loc[
    historical_rows,
    [
        "ibkr_execution_fee_per_side_usd",
        "cme_exchange_fee_per_side_usd",
        "regulatory_fee_per_side_usd",
        "clearing_fee_per_side_usd",
        "tax_or_entity_adjustment_per_side_usd",
    ],
]
if historical_numeric.notna().any().any():
    failures.append("An unsourced historical numeric fee was created.")
if vintages.loc[historical_rows, "eligible_for_historical_matching"].any():
    failures.append("Unavailable historical vintage was enabled.")

current_rows = vintages["cost_view"].eq("CURRENT_DEPLOYMENT")
current_component_columns = [
    "ibkr_execution_fee_per_side_usd",
    "cme_exchange_fee_per_side_usd",
    "regulatory_fee_per_side_usd",
    "clearing_fee_per_side_usd",
    "tax_or_entity_adjustment_per_side_usd",
]
current_components = vintages.loc[
    current_rows, current_component_columns
].apply(pd.to_numeric, errors="coerce")
if len(current_components) != 1:
    failures.append("Expected exactly one current fee-snapshot row.")
elif not np.isfinite(current_components.to_numpy(dtype=float)).all():
    failures.append("Current fee snapshot has a missing/non-finite component.")
elif current_components.lt(0).any().any():
    failures.append("Current fee snapshot has a negative component.")
else:
    component_direct_round_trip = float(
        2.0 * current_components.iloc[0].sum()
    )
    audited_direct_round_trip = float(
        fee_snapshot.get("direct_fee_round_trip_usd", np.nan)
    )
    fees_only_rows = semantic.loc[
        semantic["scenario"].astype(str).eq("FEES_ONLY")
    ]
    if len(fees_only_rows) != 1:
        failures.append("Expected exactly one FEES_ONLY scenario.")
        fees_only_direct_round_trip = np.nan
    else:
        fees_only_direct_round_trip = float(
            fees_only_rows.iloc[0]["direct_fee_round_trip_usd"]
        )
    if not np.isclose(
        component_direct_round_trip, audited_direct_round_trip, atol=1e-12
    ):
        failures.append("Current fee components do not reconcile to Cell 9 audit.")
    if np.isfinite(fees_only_direct_round_trip) and not np.isclose(
        component_direct_round_trip, fees_only_direct_round_trip, atol=1e-12
    ):
        failures.append("Current fee components do not reconcile to FEES_ONLY.")

if str(fee_snapshot.get("as_of")) != source_snapshot_date:
    failures.append("Fee snapshot date differs from the source snapshot date.")


# ------------------------------------------------------------
# 7) Save artifacts and audit
# ------------------------------------------------------------
semantic.to_csv(CELL11_SEMANTIC_SCENARIOS_PATH, index=False)
vintages.to_csv(CELL11_VINTAGE_REGISTRY_PATH, index=False)

artifact_hashes = {
    "input_cell9_parameters_sha256": actual_parameter_hash,
    "input_cell9_scenarios_sha256": actual_scenario_hash,
    "input_cell9_audit_sha256": sha256_file(CELL9_AUDIT_PATH),
    "semantic_scenarios_sha256": sha256_file(CELL11_SEMANTIC_SCENARIOS_PATH),
    "fee_vintage_registry_sha256": sha256_file(CELL11_VINTAGE_REGISTRY_PATH),
}

audit = {
    "audit_written_utc": datetime.now(timezone.utc).isoformat(),
    "policy_version": COST_TEMPORALITY_POLICY_VERSION,
    "status": "PASS" if not failures else "FAIL",
    "upstream_binding": {
        "cell9_policy_version": cell9_audit.get("policy_version"),
        "cell9_scenarios_sha256": actual_scenario_hash,
    },
    "semantic_contract": {
        "current_deployment": "PROVISIONAL_COUNTERFACTUAL",
        "stress": "PROVISIONAL_COUNTERFACTUAL",
        "historical_vintage": "OPEN",
        "historical_actual_labels_available": False,
        "historical_coverage_ratio": 0.0,
        "historical_labels_allowed": False,
        "current_snapshot_historical_matching_allowed": False,
        "volume_tier_rule": (
            "future historical implementation must use causal cumulative "
            "month-to-date contracts only"
        ),
    },
    "cell10_interpretation": {
        "existing_primary_label_status": "PROVISIONAL",
        "existing_primary_label_semantics": (
            "CURRENT_DEPLOYMENT_COUNTERFACTUAL; NOT HISTORICAL_ACTUAL_PNL"
        ),
    },
    "scenario_count": int(len(semantic)),
    "vintage_registry_rows": int(len(vintages)),
    "artifacts": {
        "semantic_scenarios": str(CELL11_SEMANTIC_SCENARIOS_PATH),
        "fee_vintage_registry": str(CELL11_VINTAGE_REGISTRY_PATH),
    },
    "sha256": artifact_hashes,
    "open_items": [
        "Collect effective-dated CME and IBKR historical fee sources.",
        "Confirm the user's IBKR legal entity, pricing plan, and taxes.",
        "Add historical bid/ask or a sourced regime proxy for execution costs.",
        "Rebuild historical-vintage labels only after 100% causal coverage.",
    ],
    "failures": failures,
}

with open(CELL11_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

if failures:
    print("\nCELL 11 FAILURES")
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(
        "\nCELL 11 COST TEMPORALITY REGISTRY: FAIL\n"
        f"{CELL11_AUDIT_PATH}"
    )

print("\n" + "=" * 72)
print("CELL 11 — COST TEMPORALITY REGISTRY")
print("=" * 72)
print("Cell 9 scenario SHA256      :", actual_scenario_hash)
print("Current deployment semantics: PROVISIONAL COUNTERFACTUAL")
print("Stress semantics            : PROVISIONAL COUNTERFACTUAL")
print("Historical vintage          : OPEN")
print("Historical numeric invented : False")
print("Historical labels allowed   : False")
print("Semantic scenarios          :", CELL11_SEMANTIC_SCENARIOS_PATH)
print("Vintage registry            :", CELL11_VINTAGE_REGISTRY_PATH)
print("Cell 11 audit               :", CELL11_AUDIT_PATH)
print("\nCELL 11 COST TEMPORALITY REGISTRY: PASS")
print("=" * 72)



CELL 11 — COST TEMPORALITY REGISTRY
Cell 9 scenario SHA256      : 2248d59ff32361dff9c5df94bfdf8d7ad6942ee50ef3d6e1c6a3731779aeff4f
Current deployment semantics: PROVISIONAL COUNTERFACTUAL
Stress semantics            : PROVISIONAL COUNTERFACTUAL
Historical vintage          : OPEN
Historical numeric invented : False
Historical labels allowed   : False
Semantic scenarios          : /content/drive/MyDrive/Quant_Lab/Data/MES_Clean_Pipeline_V1/cell11_cost_scenarios_semantic_v1.csv
Vintage registry            : /content/drive/MyDrive/Quant_Lab/Data/MES_Clean_Pipeline_V1/cell11_fee_vintage_registry_v1.csv
Cell 11 audit               : /content/drive/MyDrive/Quant_Lab/Data/MES_Clean_Pipeline_V1/cell11_cost_temporality_audit.json

CELL 11 COST TEMPORALITY REGISTRY: PASS


In [49]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 12 — DEVELOPMENT-ONLY 1m PATH OUTCOMES
# ============================================================
#
# PURPOSE
# -------
# Preserve the locked +60m endpoint outcome from Cell 10 and add the
# path information that a risk/execution layer needs: MFE, MAE, and
# close-path drawdown. Triple-barrier labels are deliberately NOT
# created because stop/target parameters and same-bar ordering remain
# OPEN policy decisions.
#
# Final Test is never looked up in mes_1m. All final-test path fields
# remain NaN/NaT and categorical fields remain SEALED.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1) Paths and policy
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Quant_Lab")
CLEAN_DIR = PROJECT_DIR / "Data" / "MES_Clean_Pipeline_V1"

CELL2_AUDIT_PATH = CLEAN_DIR / "cell2_raw_integrity_audit.json"
CELL10_LABELS_PATH = (
    CLEAN_DIR / "cell10_point_in_time_economic_labels_v1.parquet"
)
CELL10_AUDIT_PATH = CLEAN_DIR / "cell10_economic_label_audit.json"

CELL12_PATH_OUTCOMES_PATH = (
    CLEAN_DIR / "cell12_development_path_outcomes_v1.parquet"
)
CELL12_STATUS_SUMMARY_PATH = (
    CLEAN_DIR / "cell12_path_status_summary_v1.csv"
)
CELL12_AUDIT_PATH = CLEAN_DIR / "cell12_path_outcomes_audit.json"

PATH_POLICY_VERSION = "MES_V1_DEVELOPMENT_PATH_OUTCOMES_1.0"
EXPECTED_CELL2_POLICY = "MES_V1_RAW_INTEGRITY_1.1"
EXPECTED_CELL10_POLICY = "MES_V1_ECONOMIC_LABELS_1.0"
PATH_MINUTES = 60
CONTRACT_MULTIPLIER_USD_PER_POINT = 5.0


# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def strict_boolean(series, column_name):
    if pd.api.types.is_bool_dtype(series.dtype):
        return series.astype(bool)
    tokens = series.astype("string").str.strip().str.lower()
    parsed = tokens.map({"true": True, "false": False})
    if parsed.isna().any():
        bad = sorted(tokens.loc[parsed.isna()].dropna().unique().tolist())
        raise RuntimeError(
            f"CELL 12 STOPPED — invalid boolean token in {column_name}: {bad}"
        )
    return parsed.astype(bool)


# ------------------------------------------------------------
# 3) Upstream and same-runtime gates
# ------------------------------------------------------------
required_paths = [CELL2_AUDIT_PATH, CELL10_LABELS_PATH, CELL10_AUDIT_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(
        "CELL 12 STOPPED — missing upstream artifacts:\n"
        + "\n".join(missing_paths)
        + "\n\nRun Cell 0 → Cell 10 first."
    )

if "mes_1m" not in globals():
    raise RuntimeError(
        "CELL 12 STOPPED — canonical mes_1m is not in memory.\n\n"
        "Restart the session and Run All so Cell 2 decodes the canonical "
        "DBN in this same runtime."
    )
if not isinstance(mes_1m, pd.DataFrame):
    raise RuntimeError("CELL 12 STOPPED — mes_1m is not a DataFrame.")

cell2_audit = load_json(CELL2_AUDIT_PATH)
cell10_audit = load_json(CELL10_AUDIT_PATH)

if cell2_audit.get("status") != "PASS" or cell2_audit.get("failures", []):
    raise RuntimeError("CELL 12 STOPPED — Cell 2 audit is not clean PASS.")
if cell2_audit.get("policy_version") != EXPECTED_CELL2_POLICY:
    raise RuntimeError(
        "CELL 12 STOPPED — Cell 2 provenance addendum is missing."
    )
if cell10_audit.get("status") != "PASS":
    raise RuntimeError("CELL 12 STOPPED — Cell 10 audit is not PASS.")
if cell10_audit.get("failures", []):
    raise RuntimeError("CELL 12 STOPPED — Cell 10 audit has failures.")
if cell10_audit.get("policy_version") != EXPECTED_CELL10_POLICY:
    raise RuntimeError(
        "CELL 12 STOPPED — unexpected Cell 10 policy version: "
        f"{cell10_audit.get('policy_version')}"
    )

expected_label_hash = (
    cell10_audit.get("sha256", {}).get("economic_labels_sha256")
)
actual_label_hash = sha256_file(CELL10_LABELS_PATH)
if not expected_label_hash or expected_label_hash != actual_label_hash:
    raise RuntimeError(
        "CELL 12 STOPPED — Cell 10 label hash mismatch.\n"
        f"Audit : {expected_label_hash}\n"
        f"Actual: {actual_label_hash}"
    )


# ------------------------------------------------------------
# 4) Bind in-memory mes_1m to the Cell 2 audit
# ------------------------------------------------------------
required_1m_columns = {"open", "high", "low", "close", "instrument_id"}
missing_1m_columns = required_1m_columns - set(mes_1m.columns)
if missing_1m_columns:
    raise RuntimeError(
        "CELL 12 STOPPED — missing mes_1m columns:\n"
        + ", ".join(sorted(missing_1m_columns))
    )
if not isinstance(mes_1m.index, pd.DatetimeIndex):
    raise RuntimeError("CELL 12 STOPPED — mes_1m index is not DatetimeIndex.")
if mes_1m.index.tz is None:
    raise RuntimeError("CELL 12 STOPPED — mes_1m index is timezone-naive.")
if not mes_1m.index.is_monotonic_increasing:
    raise RuntimeError("CELL 12 STOPPED — mes_1m index is not sorted.")
if mes_1m.index.duplicated().any():
    raise RuntimeError("CELL 12 STOPPED — duplicate mes_1m timestamps.")

decoded_contract = cell2_audit.get("decoded", {})
expected_1m_rows = int(decoded_contract.get("rows", -1))
if len(mes_1m) != expected_1m_rows:
    raise RuntimeError(
        "CELL 12 STOPPED — mes_1m row count differs from Cell 2 audit."
    )

expected_first = pd.Timestamp(decoded_contract.get("first_timestamp"))
expected_last = pd.Timestamp(decoded_contract.get("last_timestamp"))
if mes_1m.index[0] != expected_first or mes_1m.index[-1] != expected_last:
    raise RuntimeError(
        "CELL 12 STOPPED — mes_1m endpoints differ from Cell 2 audit."
    )

raw_1m = mes_1m

# Strongly fingerprint the exact in-memory decoded frame consumed here.
# The raw DBN SHA in Cell 2 identifies the source file; this fingerprint
# identifies the decoded bytes actually used by the path calculation.
memory_hash_columns = ["open", "high", "low", "close", "instrument_id"]
memory_row_hashes = pd.util.hash_pandas_object(
    raw_1m[memory_hash_columns],
    index=True,
    categorize=False,
).to_numpy(dtype="uint64", copy=False)
mes_1m_memory_sha256 = hashlib.sha256(memory_row_hashes.tobytes()).hexdigest()
del memory_row_hashes

expected_memory_sha256 = decoded_contract.get("content_sha256")
runtime_memory_sha256 = globals().get("CELL2_MES_1M_CONTENT_SHA256")
if not expected_memory_sha256:
    raise RuntimeError("CELL 12 STOPPED — Cell 2 audit has no content SHA256.")
if runtime_memory_sha256 != expected_memory_sha256:
    raise RuntimeError(
        "CELL 12 STOPPED — the Cell 2 same-runtime fingerprint is missing/mismatched."
    )
if mes_1m_memory_sha256 != expected_memory_sha256:
    raise RuntimeError(
        "CELL 12 STOPPED — in-memory mes_1m content changed after Cell 2."
    )


# ------------------------------------------------------------
# 5) Load Cell 10 labels and prove the Final Test seal
# ------------------------------------------------------------
labels = pd.read_parquet(CELL10_LABELS_PATH)
required_label_columns = {
    "decision_id",
    "decision_time",
    "label_end_time",
    "nyse_session_date",
    "instrument_id",
    "outer_partition",
    "label_status",
    "label_usable",
    "entry_reference_close",
    "exit_reference_close_60m",
    "gross_move_points_60m",
    "gross_move_usd_60m",
    "economic_label_primary",
}
missing_label_columns = required_label_columns - set(labels.columns)
if missing_label_columns:
    raise RuntimeError(
        "CELL 12 STOPPED — missing Cell 10 columns:\n"
        + ", ".join(sorted(missing_label_columns))
    )

labels = labels.copy()
labels["decision_time"] = pd.to_datetime(
    labels["decision_time"], utc=True, errors="raise"
)
labels["label_end_time"] = pd.to_datetime(
    labels["label_end_time"], utc=True, errors="raise"
)
labels["label_usable"] = strict_boolean(labels["label_usable"], "label_usable")
labels = labels.sort_values("decision_time").reset_index(drop=True)

final_test_mask = labels["outer_partition"].eq("FINAL_TEST")
development_mask = labels["outer_partition"].isin(["TRAIN", "VALIDATION"])
usable_dev_mask = development_mask & labels["label_usable"]

expected_final_rows = int(
    cell10_audit.get("counts", {}).get("final_test_rows_sealed", -1)
)
observed_final_rows = int(final_test_mask.sum())
if observed_final_rows != expected_final_rows or observed_final_rows != 8654:
    raise RuntimeError("CELL 12 STOPPED — final-test row count changed.")
if not labels.loc[final_test_mask, "label_status"].eq(
    "SEALED_FINAL_TEST"
).all():
    raise RuntimeError("CELL 12 STOPPED — final-test status is not SEALED.")
if not labels.loc[final_test_mask, "economic_label_primary"].eq("SEALED").all():
    raise RuntimeError("CELL 12 STOPPED — final-test labels are not SEALED.")

sealed_numeric_columns = [
    "entry_reference_close",
    "exit_reference_close_60m",
    "gross_move_points_60m",
    "gross_move_usd_60m",
]
if labels.loc[final_test_mask, sealed_numeric_columns].notna().any().any():
    raise RuntimeError(
        "CELL 12 STOPPED — a future-derived numeric value exists in Final Test."
    )


# ------------------------------------------------------------
# 6) Allocate outputs; build lookup requests from Development only
# ------------------------------------------------------------
n_rows = len(labels)
dev_positions = np.flatnonzero(usable_dev_mask.to_numpy())
final_positions = np.flatnonzero(final_test_mask.to_numpy())
final_test_price_lookup_count = int(
    np.intersect1d(dev_positions, final_positions, assume_unique=True).size
    * PATH_MINUTES
)
if final_test_price_lookup_count != 0:
    raise RuntimeError("CELL 12 STOPPED — a Final Test row entered the price lookup.")
dev_times = labels.loc[usable_dev_mask, "decision_time"].reset_index(drop=True)
dev_instrument = pd.to_numeric(
    labels.loc[usable_dev_mask, "instrument_id"], errors="raise"
).to_numpy(dtype=np.int64)
dev_entry = pd.to_numeric(
    labels.loc[usable_dev_mask, "entry_reference_close"], errors="raise"
).to_numpy(dtype=float)
dev_endpoint = pd.to_numeric(
    labels.loc[usable_dev_mask, "exit_reference_close_60m"], errors="raise"
).to_numpy(dtype=float)

n_dev = len(dev_positions)
path_present = np.zeros(n_dev, dtype=np.int16)
path_instrument_changed = np.zeros(n_dev, dtype=bool)
path_high = np.full(n_dev, -np.inf, dtype=float)
path_low = np.full(n_dev, np.inf, dtype=float)
path_high_first_offset = np.full(n_dev, -1, dtype=np.int16)
path_low_first_offset = np.full(n_dev, -1, dtype=np.int16)
last_close = np.full(n_dev, np.nan, dtype=float)

running_peak_close = dev_entry.copy()
running_trough_close = dev_entry.copy()
long_close_max_drawdown = np.zeros(n_dev, dtype=float)
short_close_max_drawup = np.zeros(n_dev, dtype=float)
minutes_above_entry = np.zeros(n_dev, dtype=np.int16)
minutes_below_entry = np.zeros(n_dev, dtype=np.int16)
minutes_at_entry = np.zeros(n_dev, dtype=np.int16)


# ------------------------------------------------------------
# 7) Exact 1m path: bar starts t, t+1m, ..., t+59m
# ------------------------------------------------------------
for minute_offset in range(PATH_MINUTES):
    lookup_times = pd.DatetimeIndex(
        dev_times + pd.Timedelta(minutes=minute_offset)
    )
    observed = raw_1m.reindex(lookup_times)

    observed_instrument = pd.to_numeric(
        observed["instrument_id"], errors="coerce"
    ).to_numpy(dtype=float)
    observed_high = pd.to_numeric(
        observed["high"], errors="coerce"
    ).to_numpy(dtype=float)
    observed_low = pd.to_numeric(
        observed["low"], errors="coerce"
    ).to_numpy(dtype=float)
    observed_close = pd.to_numeric(
        observed["close"], errors="coerce"
    ).to_numpy(dtype=float)

    present = (
        np.isfinite(observed_instrument)
        & np.isfinite(observed_high)
        & np.isfinite(observed_low)
        & np.isfinite(observed_close)
    )
    instrument_match = present & np.equal(
        observed_instrument, dev_instrument.astype(float)
    )

    path_present += present.astype(np.int16)
    path_instrument_changed |= present & ~instrument_match

    new_high = instrument_match & (observed_high > path_high)
    new_low = instrument_match & (observed_low < path_low)
    path_high[new_high] = observed_high[new_high]
    path_low[new_low] = observed_low[new_low]
    path_high_first_offset[new_high] = minute_offset
    path_low_first_offset[new_low] = minute_offset

    valid_close = instrument_match & np.isfinite(observed_close)
    running_peak_close[valid_close] = np.maximum(
        running_peak_close[valid_close], observed_close[valid_close]
    )
    running_trough_close[valid_close] = np.minimum(
        running_trough_close[valid_close], observed_close[valid_close]
    )
    long_close_max_drawdown[valid_close] = np.maximum(
        long_close_max_drawdown[valid_close],
        running_peak_close[valid_close] - observed_close[valid_close],
    )
    short_close_max_drawup[valid_close] = np.maximum(
        short_close_max_drawup[valid_close],
        observed_close[valid_close] - running_trough_close[valid_close],
    )

    minutes_above_entry += (
        valid_close & (observed_close > dev_entry)
    ).astype(np.int16)
    minutes_below_entry += (
        valid_close & (observed_close < dev_entry)
    ).astype(np.int16)
    minutes_at_entry += (
        valid_close & np.isclose(observed_close, dev_entry, atol=1e-12)
    ).astype(np.int16)

    if minute_offset == PATH_MINUTES - 1:
        last_close[:] = observed_close


# ------------------------------------------------------------
# 8) Reconciliation and path metrics
# ------------------------------------------------------------
path_complete = path_present == PATH_MINUTES
endpoint_match = np.isclose(last_close, dev_endpoint, atol=1e-12, rtol=0.0)
path_usable_dev = (
    path_complete
    & ~path_instrument_changed
    & endpoint_match
    & np.isfinite(path_high)
    & np.isfinite(path_low)
)

failures = []
if not path_usable_dev.all():
    failures.append(
        "A Cell 10 usable development label lacks an exact 60-row 1m path."
    )
if not endpoint_match.all():
    failures.append("1m endpoint close does not match Cell 10 +60m close.")
if path_instrument_changed.any():
    failures.append("Instrument changed inside a usable 1m path.")

path_status = np.full(n_rows, "LABEL_UNUSABLE", dtype=object)
path_status[final_test_mask.to_numpy()] = "SEALED_FINAL_TEST"
path_status[dev_positions] = np.where(
    path_usable_dev, "USABLE", "PATH_INTEGRITY_FAILURE"
)

labels["path_status"] = path_status
labels["path_usable"] = False
labels.loc[dev_positions, "path_usable"] = path_usable_dev
labels["path_1m_expected"] = PATH_MINUTES
labels["path_1m_present"] = np.nan
labels.loc[dev_positions, "path_1m_present"] = path_present.astype(float)
labels["path_instrument_changed"] = pd.NA
labels.loc[dev_positions, "path_instrument_changed"] = path_instrument_changed

numeric_path_columns = [
    "path_high_60m",
    "path_low_60m",
    "long_mfe_points_60m",
    "long_mae_points_60m",
    "short_mfe_points_60m",
    "short_mae_points_60m",
    "long_mfe_usd_60m",
    "long_mae_usd_60m",
    "short_mfe_usd_60m",
    "short_mae_usd_60m",
    "long_close_path_max_drawdown_points_60m",
    "short_close_path_max_drawup_points_60m",
    "minutes_above_entry",
    "minutes_below_entry",
    "minutes_at_entry",
]
for column in numeric_path_columns:
    labels[column] = np.nan

labels.loc[dev_positions, "path_high_60m"] = path_high
labels.loc[dev_positions, "path_low_60m"] = path_low

long_mfe = np.maximum(path_high - dev_entry, 0.0)
long_mae = np.maximum(dev_entry - path_low, 0.0)
short_mfe = long_mae.copy()
short_mae = long_mfe.copy()

labels.loc[dev_positions, "long_mfe_points_60m"] = long_mfe
labels.loc[dev_positions, "long_mae_points_60m"] = long_mae
labels.loc[dev_positions, "short_mfe_points_60m"] = short_mfe
labels.loc[dev_positions, "short_mae_points_60m"] = short_mae
labels.loc[dev_positions, "long_mfe_usd_60m"] = (
    long_mfe * CONTRACT_MULTIPLIER_USD_PER_POINT
)
labels.loc[dev_positions, "long_mae_usd_60m"] = (
    long_mae * CONTRACT_MULTIPLIER_USD_PER_POINT
)
labels.loc[dev_positions, "short_mfe_usd_60m"] = (
    short_mfe * CONTRACT_MULTIPLIER_USD_PER_POINT
)
labels.loc[dev_positions, "short_mae_usd_60m"] = (
    short_mae * CONTRACT_MULTIPLIER_USD_PER_POINT
)
labels.loc[
    dev_positions, "long_close_path_max_drawdown_points_60m"
] = long_close_max_drawdown
labels.loc[
    dev_positions, "short_close_path_max_drawup_points_60m"
] = short_close_max_drawup
labels.loc[dev_positions, "minutes_above_entry"] = minutes_above_entry
labels.loc[dev_positions, "minutes_below_entry"] = minutes_below_entry
labels.loc[dev_positions, "minutes_at_entry"] = minutes_at_entry

labels["path_high_first_bar_start"] = pd.Series(
    pd.NaT, index=labels.index, dtype="datetime64[ns, UTC]"
)
labels["path_low_first_bar_start"] = pd.Series(
    pd.NaT, index=labels.index, dtype="datetime64[ns, UTC]"
)
high_offset_valid = path_high_first_offset >= 0
low_offset_valid = path_low_first_offset >= 0
high_times = pd.Series(pd.NaT, index=np.arange(n_dev), dtype="datetime64[ns, UTC]")
low_times = pd.Series(pd.NaT, index=np.arange(n_dev), dtype="datetime64[ns, UTC]")
high_times.loc[high_offset_valid] = (
    dev_times.loc[high_offset_valid].reset_index(drop=True)
    + pd.to_timedelta(path_high_first_offset[high_offset_valid], unit="m")
).to_numpy()
low_times.loc[low_offset_valid] = (
    dev_times.loc[low_offset_valid].reset_index(drop=True)
    + pd.to_timedelta(path_low_first_offset[low_offset_valid], unit="m")
).to_numpy()
labels.loc[dev_positions, "path_high_first_bar_start"] = high_times.to_numpy()
labels.loc[dev_positions, "path_low_first_bar_start"] = low_times.to_numpy()

labels["triple_barrier_status"] = "OPEN_POLICY_NOT_LOCKED"
labels.loc[final_test_mask, "triple_barrier_status"] = "SEALED"


# ------------------------------------------------------------
# 9) Final hard gates
# ------------------------------------------------------------
if labels.loc[final_test_mask, numeric_path_columns].notna().any().any():
    failures.append("A path-derived numeric value entered Final Test.")
if labels.loc[
    final_test_mask,
    ["path_high_first_bar_start", "path_low_first_bar_start"],
].notna().any().any():
    failures.append("A path-derived timestamp entered Final Test.")
if labels.loc[final_test_mask, "path_usable"].any():
    failures.append("A Final Test row became path-usable.")
if not labels.loc[final_test_mask, "path_status"].eq(
    "SEALED_FINAL_TEST"
).all():
    failures.append("Final Test path status is not SEALED.")
if not labels.loc[usable_dev_mask, "path_usable"].all():
    failures.append("Not every usable development label has a usable path.")
if not labels.loc[usable_dev_mask, "path_1m_present"].eq(PATH_MINUTES).all():
    failures.append("A usable path does not contain exactly 60 one-minute bars.")
if not np.allclose(
    labels.loc[usable_dev_mask, "long_mfe_points_60m"],
    labels.loc[usable_dev_mask, "short_mae_points_60m"],
    atol=1e-12,
):
    failures.append("Long MFE and Short MAE symmetry failed.")
if not np.allclose(
    labels.loc[usable_dev_mask, "long_mae_points_60m"],
    labels.loc[usable_dev_mask, "short_mfe_points_60m"],
    atol=1e-12,
):
    failures.append("Long MAE and Short MFE symmetry failed.")


# ------------------------------------------------------------
# 10) Save artifacts and audit
# ------------------------------------------------------------
status_summary = (
    labels.groupby(["outer_partition", "path_status"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
)

labels.to_parquet(CELL12_PATH_OUTCOMES_PATH, index=False)
status_summary.to_csv(CELL12_STATUS_SUMMARY_PATH, index=False)

artifact_hashes = {
    "input_cell2_audit_sha256": sha256_file(CELL2_AUDIT_PATH),
    "input_cell10_labels_sha256": actual_label_hash,
    "input_cell10_audit_sha256": sha256_file(CELL10_AUDIT_PATH),
    "path_outcomes_sha256": sha256_file(CELL12_PATH_OUTCOMES_PATH),
    "path_status_summary_sha256": sha256_file(CELL12_STATUS_SUMMARY_PATH),
}

audit = {
    "audit_written_utc": datetime.now(timezone.utc).isoformat(),
    "policy_version": PATH_POLICY_VERSION,
    "status": "PASS" if not failures else "FAIL",
    "upstream_binding": {
        "cell10_policy_version": cell10_audit.get("policy_version"),
        "cell10_labels_sha256": actual_label_hash,
        "cell2_raw_file_sha256": cell2_audit.get("raw_file", {}).get("sha256"),
        "mes_1m_memory_sha256": mes_1m_memory_sha256,
        "mes_1m_rows_in_memory": int(len(raw_1m)),
        "mes_1m_first_timestamp": raw_1m.index[0].isoformat(),
        "mes_1m_last_timestamp": raw_1m.index[-1].isoformat(),
    },
    "path_contract": {
        "entry": "completed 15m close at decision_time",
        "one_minute_bar_start_offsets": "0..59 minutes after decision_time",
        "one_minute_rows_required": PATH_MINUTES,
        "endpoint_reconciliation": (
            "close of 1m bar starting t+59m equals Cell 10 +60m close"
        ),
        "same_instrument_required": True,
        "mfe_mae_before_cost": True,
        "triple_barrier_created": False,
        "triple_barrier_status": "OPEN",
        "same_1m_bar_barrier_order": "UNOBSERVABLE_WITH_OHLC; NEVER_GUESSED",
        "target_fields_allowed_as_features": False,
    },
    "counts": {
        "rows": int(len(labels)),
        "usable_development_paths": int(
            labels.loc[development_mask, "path_usable"].sum()
        ),
        "final_test_rows_sealed": observed_final_rows,
        "final_test_price_lookup_count": final_test_price_lookup_count,
    },
    "artifacts": {
        "path_outcomes": str(CELL12_PATH_OUTCOMES_PATH),
        "path_status_summary": str(CELL12_STATUS_SUMMARY_PATH),
    },
    "sha256": artifact_hashes,
    "open_items": [
        "Lock LONG/FLAT execution policy before converting outcomes to actions.",
        "Lock stop/target distances before any triple-barrier auxiliary label.",
        "Treat a 1m bar touching both barriers as AMBIGUOUS, never ordered.",
    ],
    "failures": failures,
}

with open(CELL12_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

if failures:
    print("\nCELL 12 FAILURES")
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(
        "\nCELL 12 DEVELOPMENT PATH OUTCOMES: FAIL\n"
        f"{CELL12_AUDIT_PATH}"
    )

print("\n" + "=" * 72)
print("CELL 12 — DEVELOPMENT-ONLY 1m PATH OUTCOMES")
print("=" * 72)
print("Cell 10 labels SHA256       :", actual_label_hash)
print("Development paths usable   :", f"{int(usable_dev_mask.sum()):,}")
print("Required 1m bars per path  :", PATH_MINUTES)
print("Endpoint mismatches         :", int((~endpoint_match).sum()))
print("Final-test lookup count     : 0")
print("Final-test path fields      : SEALED")
print("Triple barrier              : OPEN / NOT CREATED")
print("Path outcomes               :", CELL12_PATH_OUTCOMES_PATH)
print("Cell 12 audit               :", CELL12_AUDIT_PATH)
print("\nCELL 12 DEVELOPMENT PATH OUTCOMES: PASS")
print("=" * 72)



CELL 12 — DEVELOPMENT-ONLY 1m PATH OUTCOMES
Cell 10 labels SHA256       : 1f73f06d92bc54ccceff637503ef9cbece0c2b0c6b2018802923ef51d7352bd0
Development paths usable   : 31,165
Required 1m bars per path  : 60
Endpoint mismatches         : 0
Final-test lookup count     : 0
Final-test path fields      : SEALED
Triple barrier              : OPEN / NOT CREATED
Path outcomes               : /content/drive/MyDrive/Quant_Lab/Data/MES_Clean_Pipeline_V1/cell12_development_path_outcomes_v1.parquet
Cell 12 audit               : /content/drive/MyDrive/Quant_Lab/Data/MES_Clean_Pipeline_V1/cell12_path_outcomes_audit.json

CELL 12 DEVELOPMENT PATH OUTCOMES: PASS


In [50]:
# ============================================================
# MES QUANT PIPELINE V1 CLEAN
# CELL 13 — DEVELOPMENT-ONLY DEPENDENCE AUDIT & NAIVE BASELINES
# ============================================================
#
# PURPOSE
# -------
# Establish honest reference lines before any predictive model:
#
# - V1 action space is LONG / FLAT (SHORT remains diagnostic only).
# - A position is held for 60 minutes and positions do not overlap.
# - Every reported result is out-of-fold validation for 2022–2024.
# - Confidence intervals resample consecutive NYSE sessions, never rows.
# - Final Test stays sealed and is removed before aggregation.
#
# P&L uses the CURRENT_DEPLOYMENT_COUNTERFACTUAL cost view. It is not
# represented as historically realized P&L.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1) Paths and locked evaluation policy
# ------------------------------------------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/Quant_Lab")
CLEAN_DIR = PROJECT_DIR / "Data" / "MES_Clean_Pipeline_V1"

CELL8_AUDIT_PATH = CLEAN_DIR / "cell8_purged_split_audit.json"
CELL10_AUDIT_PATH = CLEAN_DIR / "cell10_economic_label_audit.json"
CELL11_SEMANTIC_SCENARIOS_PATH = (
    CLEAN_DIR / "cell11_cost_scenarios_semantic_v1.csv"
)
CELL11_AUDIT_PATH = CLEAN_DIR / "cell11_cost_temporality_audit.json"
CELL12_PATH_OUTCOMES_PATH = (
    CLEAN_DIR / "cell12_development_path_outcomes_v1.parquet"
)
CELL12_AUDIT_PATH = CLEAN_DIR / "cell12_path_outcomes_audit.json"

CELL13_EVENTS_PATH = (
    CLEAN_DIR / "cell13_development_oof_baseline_events_v1.parquet"
)
CELL13_DEPENDENCE_PATH = (
    CLEAN_DIR / "cell13_dependence_ess_audit_v1.csv"
)
CELL13_METRICS_PATH = (
    CLEAN_DIR / "cell13_naive_baseline_metrics_v1.csv"
)
CELL13_BOOTSTRAP_PATH = (
    CLEAN_DIR / "cell13_block_bootstrap_ci_v1.csv"
)
CELL13_AUDIT_PATH = CLEAN_DIR / "cell13_development_baseline_audit.json"

BASELINE_POLICY_VERSION = "MES_V1_DEPENDENCE_BASELINES_1.0"
EXPECTED_CELL8_POLICY = "MES_V1_PURGED_SPLIT_1.0"
EXPECTED_CELL10_POLICY = "MES_V1_ECONOMIC_LABELS_1.0"
EXPECTED_CELL11_POLICY = "MES_V1_COST_TEMPORALITY_1.0"
EXPECTED_CELL12_POLICY = "MES_V1_DEVELOPMENT_PATH_OUTCOMES_1.0"

ACTION_SPACE = "LONG_FLAT"
POSITION_POLICY = "NON_OVERLAPPING_60M"
HOLD_MINUTES = 60
EXIT_THEN_ENTRY_AT_SAME_TIMESTAMP = True
MASTER_SEED = 20260809
RANDOM_SENSITIVITY_SEEDS = 1000
BOOTSTRAP_REPETITIONS = 2000
BOOTSTRAP_BLOCK_LENGTHS = [1, 5, 20]
PRIMARY_BOOTSTRAP_BLOCK_SESSIONS = 5

FOLD_SPECS = [
    ("WF_2022", "role_wf_2022", 2022),
    ("WF_2023", "role_wf_2023", 2023),
    ("WF_2024", "role_wf_2024", 2024),
]


# ------------------------------------------------------------
# 2) Helpers
# ------------------------------------------------------------
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def strict_boolean(series, column_name):
    """Parse booleans without treating the string 'False' as truthy."""
    if pd.api.types.is_bool_dtype(series.dtype):
        return series.astype(bool)

    tokens = series.astype("string").str.strip().str.lower()
    parsed = tokens.map({"true": True, "false": False})
    if parsed.isna().any():
        bad = sorted(tokens.loc[parsed.isna()].dropna().unique().tolist())
        raise RuntimeError(
            f"CELL 13 STOPPED — invalid boolean token in {column_name}: {bad}"
        )
    return parsed.astype(bool)


def finite_distribution_summary(values):
    """Summarize only finite replications; never warn on all-NaN Sharpe."""
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return {
            "mean": np.nan,
            "p025": np.nan,
            "p975": np.nan,
            "valid_repetitions": 0,
            "status": "UNDEFINED_ZERO_VARIANCE",
        }
    return {
        "mean": float(finite.mean()),
        "p025": float(np.quantile(finite, 0.025)),
        "p975": float(np.quantile(finite, 0.975)),
        "valid_repetitions": int(finite.size),
        "status": "DEFINED",
    }


def classification_metrics(y_true, y_pred, prob_long):
    y_true = np.asarray(y_true, dtype=np.int8)
    y_pred = np.asarray(y_pred, dtype=np.int8)
    prob_long = np.asarray(prob_long, dtype=float)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    # Explicit zero_division=0 policy: an unpredicted class contributes
    # F1=0 to macro-F1; it is never silently dropped by nanmean.
    recall_long = tp / (tp + fn) if (tp + fn) else 0.0
    recall_flat = tn / (tn + fp) if (tn + fp) else 0.0
    precision_long = tp / (tp + fp) if (tp + fp) else 0.0
    precision_flat = tn / (tn + fn) if (tn + fn) else 0.0

    f1_long = (
        2.0 * precision_long * recall_long / (precision_long + recall_long)
        if (precision_long + recall_long)
        else 0.0
    )
    f1_flat = (
        2.0 * precision_flat * recall_flat / (precision_flat + recall_flat)
        if (precision_flat + recall_flat)
        else 0.0
    )

    clipped = np.clip(prob_long, 1e-12, 1.0 - 1e-12)
    log_loss = -np.mean(
        y_true * np.log(clipped) + (1 - y_true) * np.log(1 - clipped)
    )

    return {
        "rows": int(len(y_true)),
        "accuracy": float(np.mean(y_true == y_pred)),
        "balanced_accuracy": float((recall_long + recall_flat) / 2.0),
        "macro_f1": float((f1_long + f1_flat) / 2.0),
        "recall_enter_long": recall_long,
        "recall_flat": recall_flat,
        "predicted_long_coverage": float(np.mean(y_pred == 1)),
        "brier_score": float(np.mean((prob_long - y_true) ** 2)),
        "log_loss": float(log_loss),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def simulate_non_overlapping_long(valid_rows, predicted_long, cost_usd):
    rows = valid_rows.reset_index(drop=True).copy()
    predicted_long = np.asarray(predicted_long, dtype=bool)
    if len(predicted_long) != len(rows):
        raise RuntimeError("CELL 13 STOPPED — prediction length mismatch.")
    canonical_order = rows.sort_values(
        ["decision_time", "decision_id"], kind="stable"
    ).index.to_numpy()
    if not np.array_equal(canonical_order, np.arange(len(rows))):
        raise RuntimeError(
            "CELL 13 STOPPED — simulator input is not in canonical time/ID order."
        )

    executed = np.zeros(len(rows), dtype=bool)
    ignored = np.zeros(len(rows), dtype=bool)
    gross_pnl = np.zeros(len(rows), dtype=float)
    charged_cost = np.zeros(len(rows), dtype=float)
    net_pnl = np.zeros(len(rows), dtype=float)
    exit_times = pd.Series(pd.NaT, index=rows.index, dtype="datetime64[ns, UTC]")

    next_available = None
    for i, row in rows.iterrows():
        decision_time = row["decision_time"]
        if not predicted_long[i]:
            continue
        if next_available is not None and decision_time < next_available:
            ignored[i] = True
            continue

        exit_time = decision_time + pd.Timedelta(minutes=HOLD_MINUTES)
        executed[i] = True
        exit_times.iloc[i] = exit_time
        gross_pnl[i] = float(row["gross_move_usd_60m"])
        charged_cost[i] = float(cost_usd)
        net_pnl[i] = gross_pnl[i] - charged_cost[i]
        next_available = exit_time

    return {
        "rows": rows,
        "executed": executed,
        "ignored": ignored,
        "gross_pnl_usd": gross_pnl,
        "charged_cost_usd": charged_cost,
        "net_pnl_usd": net_pnl,
        "exit_time": exit_times,
    }


def economic_metrics(event_rows):
    sessions = (
        event_rows.groupby("nyse_session_date", sort=True)["net_pnl_usd"]
        .sum()
        .astype(float)
    )
    executed = event_rows["executed"].astype(bool)
    trade_net = event_rows.loc[executed, "net_pnl_usd"].astype(float)

    session_std = float(sessions.std(ddof=1)) if len(sessions) > 1 else np.nan
    annualized_sharpe = (
        float(np.sqrt(252.0) * sessions.mean() / session_std)
        if np.isfinite(session_std) and session_std > 0
        else np.nan
    )

    equity_values = sessions.cumsum().to_numpy(dtype=float)
    equity_with_origin = np.concatenate(([0.0], equity_values))
    running_peak = np.maximum.accumulate(equity_with_origin)
    drawdown = equity_with_origin - running_peak
    maximum_drawdown = float(drawdown.min()) if len(drawdown) else 0.0

    if len(sessions):
        tail_count = max(1, int(math.ceil(0.05 * len(sessions))))
        expected_shortfall_95 = float(np.sort(sessions.to_numpy())[:tail_count].mean())
    else:
        expected_shortfall_95 = np.nan

    wins = trade_net[trade_net > 0]
    losses = trade_net[trade_net < 0]
    profit_factor = (
        float(wins.sum() / abs(losses.sum()))
        if len(losses) and abs(losses.sum()) > 0
        else np.nan
    )

    return {
        "executed_trades": int(executed.sum()),
        "ignored_long_signals": int(event_rows["ignored_signal"].sum()),
        "gross_pnl_usd": float(event_rows["gross_pnl_usd"].sum()),
        "total_cost_usd": float(event_rows["charged_cost_usd"].sum()),
        "net_pnl_usd": float(event_rows["net_pnl_usd"].sum()),
        "mean_net_usd_per_trade": float(trade_net.mean()) if len(trade_net) else np.nan,
        "median_net_usd_per_trade": float(trade_net.median()) if len(trade_net) else np.nan,
        "trade_hit_rate": float((trade_net > 0).mean()) if len(trade_net) else np.nan,
        "profit_factor": profit_factor,
        "trades_per_session": safe_divide(int(executed.sum()), len(sessions)),
        "mean_session_net_usd": float(sessions.mean()) if len(sessions) else np.nan,
        "annualized_session_sharpe": annualized_sharpe,
        "maximum_drawdown_usd": maximum_drawdown,
        # This is signed lower-tail P&L (normally negative), not a positive loss.
        "session_lower_tail_mean_pnl_5pct_usd": expected_shortfall_95,
        "maximum_concurrent_positions": 1 if executed.any() else 0,
    }


def random_economic_sensitivity(valid_rows, random_draws, cost_usd):
    rows = valid_rows.sort_values("decision_time").reset_index(drop=True)
    times_ns = rows["decision_time"].astype("int64").to_numpy()
    gross = rows["gross_move_usd_60m"].to_numpy(dtype=float)
    session_codes, session_values = pd.factorize(
        rows["nyse_session_date"].astype(str), sort=True
    )
    hold_ns = int(pd.Timedelta(minutes=HOLD_MINUTES).value)

    total_net = np.zeros(len(random_draws), dtype=float)
    sharpe = np.full(len(random_draws), np.nan, dtype=float)
    max_drawdown = np.zeros(len(random_draws), dtype=float)

    for rep, prediction in enumerate(random_draws):
        session_net = np.zeros(len(session_values), dtype=float)
        next_available_ns = np.iinfo(np.int64).min
        for i in range(len(rows)):
            if prediction[i] and times_ns[i] >= next_available_ns:
                session_net[session_codes[i]] += gross[i] - cost_usd
                next_available_ns = times_ns[i] + hold_ns

        total_net[rep] = session_net.sum()
        session_std = session_net.std(ddof=1) if len(session_net) > 1 else np.nan
        if np.isfinite(session_std) and session_std > 0:
            sharpe[rep] = np.sqrt(252.0) * session_net.mean() / session_std

        equity = np.concatenate(([0.0], np.cumsum(session_net)))
        max_drawdown[rep] = np.min(equity - np.maximum.accumulate(equity))

    return {
        "net_pnl_usd": total_net,
        "annualized_session_sharpe": sharpe,
        "maximum_drawdown_usd": max_drawdown,
    }


def empirical_overlap_diagnostics(frame, fold_id):
    ordered = frame.sort_values(["nyse_session_date", "decision_time"]).copy()
    rows = []
    positive_rhos = []

    for lag in [1, 2, 3]:
        grouped = ordered.groupby("nyse_session_date", sort=False)
        prior_return = grouped["gross_move_points_60m"].shift(lag)
        prior_time = grouped["decision_time"].shift(lag)
        exact_pair = (
            ordered["decision_time"] - prior_time
        ).eq(pd.Timedelta(minutes=15 * lag))
        valid = exact_pair & prior_return.notna()

        if int(valid.sum()) >= 2:
            rho = float(
                np.corrcoef(
                    ordered.loc[valid, "gross_move_points_60m"].astype(float),
                    prior_return.loc[valid].astype(float),
                )[0, 1]
            )
        else:
            rho = np.nan
        if np.isfinite(rho):
            positive_rhos.append(max(rho, 0.0))
        rows.append(
            {
                "fold_id": fold_id,
                "diagnostic": f"return_autocorrelation_lag_{lag}",
                "value": rho,
                "pairs": int(valid.sum()),
                "status": "DESCRIPTIVE",
            }
        )

    n = int(len(ordered))
    design_effect = max(1.0, 1.0 + 2.0 * sum(positive_rhos))
    empirical_ess = n / design_effect

    concurrency_values = []
    for _, session in ordered.groupby("nyse_session_date", sort=False):
        times = session["decision_time"].sort_values().tolist()
        active_exits = []
        for timestamp in times:
            active_exits = [exit_time for exit_time in active_exits if exit_time > timestamp]
            active_exits.append(timestamp + pd.Timedelta(minutes=HOLD_MINUTES))
            concurrency_values.append(len(active_exits))

    rows.extend(
        [
            {
                "fold_id": fold_id,
                "diagnostic": "observed_design_effect_positive_rho_lags_1_3",
                "value": float(design_effect),
                "pairs": n,
                "status": "DESCRIPTIVE_NOT_FOR_CI",
            },
            {
                "fold_id": fold_id,
                "diagnostic": "observed_effective_sample_size",
                "value": float(empirical_ess),
                "pairs": n,
                "status": "DESCRIPTIVE_NOT_FOR_CI",
            },
            {
                "fold_id": fold_id,
                "diagnostic": "theoretical_n_div_4_reference",
                "value": float(n / 4.0),
                "pairs": n,
                "status": "REFERENCE_NOT_OBSERVED_FACT",
            },
            {
                "fold_id": fold_id,
                "diagnostic": "average_available_label_concurrency",
                "value": float(np.mean(concurrency_values)),
                "pairs": n,
                "status": "DESCRIPTIVE",
            },
            {
                "fold_id": fold_id,
                "diagnostic": "maximum_available_label_concurrency",
                "value": float(np.max(concurrency_values)),
                "pairs": n,
                "status": "DESCRIPTIVE",
            },
        ]
    )
    return rows


def moving_block_indices(n_sessions, block_length, repetitions, rng):
    if n_sessions <= 0:
        raise ValueError("No sessions for bootstrap.")
    effective_block = min(block_length, n_sessions)
    max_start = n_sessions - effective_block
    blocks_needed = int(math.ceil(n_sessions / effective_block))
    draws = np.empty((repetitions, n_sessions), dtype=np.int32)

    for rep in range(repetitions):
        starts = rng.integers(0, max_start + 1, size=blocks_needed)
        indices = np.concatenate(
            [
                np.arange(start, start + effective_block, dtype=np.int32)
                for start in starts
            ]
        )[:n_sessions]
        draws[rep] = indices
    return draws


# ------------------------------------------------------------
# 3) Upstream artifact and hash gates
# ------------------------------------------------------------
required_paths = [
    CELL8_AUDIT_PATH,
    CELL10_AUDIT_PATH,
    CELL11_SEMANTIC_SCENARIOS_PATH,
    CELL11_AUDIT_PATH,
    CELL12_PATH_OUTCOMES_PATH,
    CELL12_AUDIT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise RuntimeError(
        "CELL 13 STOPPED — missing upstream artifacts:\n"
        + "\n".join(missing_paths)
        + "\n\nRun Cell 0 → Cell 12 first."
    )

cell8_audit = load_json(CELL8_AUDIT_PATH)
cell10_audit = load_json(CELL10_AUDIT_PATH)
cell11_audit = load_json(CELL11_AUDIT_PATH)
cell12_audit = load_json(CELL12_AUDIT_PATH)

expected_versions = [
    (cell8_audit, EXPECTED_CELL8_POLICY, "Cell 8"),
    (cell10_audit, EXPECTED_CELL10_POLICY, "Cell 10"),
    (cell11_audit, EXPECTED_CELL11_POLICY, "Cell 11"),
    (cell12_audit, EXPECTED_CELL12_POLICY, "Cell 12"),
]
for audit, expected_version, name in expected_versions:
    if audit.get("status") != "PASS" or audit.get("failures", []):
        raise RuntimeError(f"CELL 13 STOPPED — {name} audit is not clean PASS.")
    if audit.get("policy_version") != expected_version:
        raise RuntimeError(
            f"CELL 13 STOPPED — unexpected {name} policy version: "
            f"{audit.get('policy_version')}"
        )

expected_path_hash = (
    cell12_audit.get("sha256", {}).get("path_outcomes_sha256")
)
actual_path_hash = sha256_file(CELL12_PATH_OUTCOMES_PATH)
if not expected_path_hash or expected_path_hash != actual_path_hash:
    raise RuntimeError("CELL 13 STOPPED — Cell 12 path hash mismatch.")

expected_semantic_hash = (
    cell11_audit.get("sha256", {}).get("semantic_scenarios_sha256")
)
actual_semantic_hash = sha256_file(CELL11_SEMANTIC_SCENARIOS_PATH)
if not expected_semantic_hash or expected_semantic_hash != actual_semantic_hash:
    raise RuntimeError("CELL 13 STOPPED — Cell 11 cost hash mismatch.")

cell10_labels_hash = (
    cell10_audit.get("sha256", {}).get("economic_labels_sha256")
)
cell12_cell10_labels_hash = (
    cell12_audit.get("upstream_binding", {}).get("cell10_labels_sha256")
)
if not cell10_labels_hash or cell12_cell10_labels_hash != cell10_labels_hash:
    raise RuntimeError(
        "CELL 13 STOPPED — Cell 12 path outcomes do not descend from the "
        "current Cell 10 labels."
    )

actual_cell10_audit_hash = sha256_file(CELL10_AUDIT_PATH)
cell12_cell10_audit_hash = (
    cell12_audit.get("sha256", {}).get("input_cell10_audit_sha256")
)
if cell12_cell10_audit_hash != actual_cell10_audit_hash:
    raise RuntimeError(
        "CELL 13 STOPPED — Cell 12 was not built from the current Cell 10 audit."
    )

cell10_cell9_hash = (
    cell10_audit.get("upstream_binding", {}).get(
        "cell9_cost_scenarios_sha256"
    )
)
cell11_cell9_hash = (
    cell11_audit.get("upstream_binding", {}).get("cell9_scenarios_sha256")
)
if not cell10_cell9_hash or cell10_cell9_hash != cell11_cell9_hash:
    raise RuntimeError(
        "CELL 13 STOPPED — Cell 10 labels and Cell 11 costs descend from "
        "different Cell 9 scenarios."
    )

cell10_cell8_hash = (
    cell10_audit.get("upstream_binding", {}).get(
        "cell8_assignments_sha256"
    )
)
cell8_assignment_hash = (
    cell8_audit.get("sha256", {}).get("split_assignments_sha256")
)
if not cell10_cell8_hash or cell10_cell8_hash != cell8_assignment_hash:
    raise RuntimeError(
        "CELL 13 STOPPED — Cell 10 fold assignments do not match Cell 8."
    )


# ------------------------------------------------------------
# 4) Select the explicitly counterfactual primary cost
# ------------------------------------------------------------
semantic_costs = pd.read_csv(CELL11_SEMANTIC_SCENARIOS_PATH)
required_cost_columns = {
    "scenario",
    "primary_economic_gate",
    "cost_view",
    "historical_actual_claim",
    "total_round_trip_usd",
    "break_even_index_points",
}
missing_cost_columns = required_cost_columns - set(semantic_costs.columns)
if missing_cost_columns:
    raise RuntimeError(
        "CELL 13 STOPPED — semantic cost schema is incomplete: "
        + ", ".join(sorted(missing_cost_columns))
    )
primary_gate_mask = strict_boolean(
    semantic_costs["primary_economic_gate"], "primary_economic_gate"
)
historical_claim_mask = strict_boolean(
    semantic_costs["historical_actual_claim"], "historical_actual_claim"
)
primary_cost = semantic_costs.loc[primary_gate_mask]
if len(primary_cost) != 1:
    raise RuntimeError("CELL 13 STOPPED — expected one primary cost row.")
primary_cost = primary_cost.iloc[0]

if primary_cost["cost_view"] != "CURRENT_DEPLOYMENT":
    raise RuntimeError("CELL 13 STOPPED — primary cost is not CURRENT_DEPLOYMENT.")
if bool(historical_claim_mask.loc[primary_cost.name]):
    raise RuntimeError("CELL 13 STOPPED — primary cost claims historical actuality.")

round_trip_cost_usd = float(primary_cost["total_round_trip_usd"])
primary_cost_scenario = str(primary_cost["scenario"])
primary_break_even_points = float(primary_cost["break_even_index_points"])
if not np.isfinite(round_trip_cost_usd) or round_trip_cost_usd < 0:
    raise RuntimeError("CELL 13 STOPPED — invalid primary round-trip cost.")
if not np.isfinite(primary_break_even_points) or primary_break_even_points < 0:
    raise RuntimeError("CELL 13 STOPPED — invalid primary break-even points.")

cell10_contract = cell10_audit.get("label_contract", {})
if str(cell10_contract.get("primary_cost_scenario")) != primary_cost_scenario:
    raise RuntimeError("CELL 13 STOPPED — Cell 10 primary scenario mismatch.")
if not np.isclose(
    float(cell10_contract.get("primary_round_trip_cost_usd", np.nan)),
    round_trip_cost_usd,
    atol=1e-12,
):
    raise RuntimeError("CELL 13 STOPPED — Cell 10 round-trip cost mismatch.")
if not np.isclose(
    float(cell10_contract.get("primary_break_even_index_points", np.nan)),
    primary_break_even_points,
    atol=1e-12,
):
    raise RuntimeError("CELL 13 STOPPED — Cell 10 break-even mismatch.")


# ------------------------------------------------------------
# 5) Load path outcomes; prove seal before dropping Final Test
# ------------------------------------------------------------
data = pd.read_parquet(CELL12_PATH_OUTCOMES_PATH)
required_columns = {
    "decision_id",
    "decision_time",
    "label_end_time",
    "nyse_session_date",
    "outer_partition",
    "label_status",
    "label_usable",
    "path_status",
    "path_usable",
    "long_mfe_points_60m",
    "long_mae_points_60m",
    "gross_move_points_60m",
    "gross_move_usd_60m",
    "economic_label_primary",
    *[role_column for _, role_column, _ in FOLD_SPECS],
}
missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise RuntimeError(
        "CELL 13 STOPPED — missing required columns:\n"
        + ", ".join(sorted(missing_columns))
    )

data = data.copy()
data["decision_time"] = pd.to_datetime(data["decision_time"], utc=True)
data["label_end_time"] = pd.to_datetime(data["label_end_time"], utc=True)
data["label_usable"] = strict_boolean(data["label_usable"], "label_usable")
data["path_usable"] = strict_boolean(data["path_usable"], "path_usable")

decision_ids = data["decision_id"].astype("string").str.strip()
invalid_id_tokens = {"", "nan", "none", "null", "<na>"}
if decision_ids.isna().any() or decision_ids.str.lower().isin(invalid_id_tokens).any():
    raise RuntimeError("CELL 13 STOPPED — missing/blank validation decision ID.")
if decision_ids.duplicated().any():
    raise RuntimeError("CELL 13 STOPPED — decision IDs are not globally unique.")
data["decision_id"] = decision_ids
if data["decision_time"].isna().any() or data["decision_time"].duplicated().any():
    raise RuntimeError("CELL 13 STOPPED — decision timestamps are missing/duplicated.")

final_mask = data["outer_partition"].eq("FINAL_TEST")
expected_final_rows = int(
    cell12_audit.get("counts", {}).get("final_test_rows_sealed", -1)
)
if expected_final_rows != 8654 or int(final_mask.sum()) != expected_final_rows:
    raise RuntimeError("CELL 13 STOPPED — final-test row count changed.")
if not data.loc[final_mask, "label_status"].eq("SEALED_FINAL_TEST").all():
    raise RuntimeError("CELL 13 STOPPED — final-test label status is not SEALED.")
if not data.loc[final_mask, "path_status"].eq("SEALED_FINAL_TEST").all():
    raise RuntimeError("CELL 13 STOPPED — final-test path status is not SEALED.")
if not data.loc[final_mask, "economic_label_primary"].eq("SEALED").all():
    raise RuntimeError("CELL 13 STOPPED — final-test economic label is not SEALED.")

sealed_numeric = [
    "gross_move_points_60m",
    "gross_move_usd_60m",
    "long_mfe_points_60m",
    "long_mae_points_60m",
]
if data.loc[final_mask, sealed_numeric].notna().any().any():
    raise RuntimeError("CELL 13 STOPPED — Final Test contains outcome numerics.")

# Final Test is discarded before any class count, prior, metric, or P&L.
development = data.loc[~final_mask].copy()
del data

usable_development_labels = set(
    development.loc[
        development["label_usable"], "economic_label_primary"
    ].dropna().astype(str)
)
allowed_development_labels = {"LONG", "SHORT", "NO_TRADE"}
if not usable_development_labels.issubset(allowed_development_labels):
    raise RuntimeError(
        "CELL 13 STOPPED — unexpected usable economic label: "
        + ", ".join(sorted(usable_development_labels - allowed_development_labels))
    )

development_session_year = pd.to_datetime(
    development["nyse_session_date"], errors="raise"
).dt.year
if development_session_year.ge(2025).any():
    raise RuntimeError("CELL 13 STOPPED — a 2025+ timestamp entered Development.")


# ------------------------------------------------------------
# 6) OOF baselines fold by fold
# ------------------------------------------------------------
event_frames = []
metric_rows = []
dependence_rows = []
validation_ids = []
boundary_failures = []
canonical_validation_sessions = {}

for fold_index, (fold_id, role_column, validation_year) in enumerate(FOLD_SPECS):
    canonical_validation_rows = development.loc[
        development[role_column].eq("VALIDATION")
    ].copy()
    if canonical_validation_rows.empty:
        raise RuntimeError(
            f"CELL 13 STOPPED — no canonical validation rows in {fold_id}."
        )
    canonical_year = pd.to_datetime(
        canonical_validation_rows["nyse_session_date"], errors="raise"
    ).dt.year
    if not canonical_year.eq(validation_year).all():
        raise RuntimeError(
            f"CELL 13 STOPPED — canonical validation year mismatch in {fold_id}."
        )
    canonical_sessions = sorted(
        canonical_validation_rows["nyse_session_date"].astype(str).unique().tolist()
    )
    canonical_validation_sessions[fold_id] = canonical_sessions

    train_mask = development[role_column].eq("TRAIN") & development["label_usable"]
    validation_mask = (
        development[role_column].eq("VALIDATION")
        & development["label_usable"]
        & development["path_usable"]
    )

    train = development.loc[train_mask].sort_values(
        ["decision_time", "decision_id"], kind="stable"
    ).copy()
    valid = development.loc[validation_mask].sort_values(
        ["decision_time", "decision_id"], kind="stable"
    ).copy()

    if train.empty or valid.empty:
        raise RuntimeError(f"CELL 13 STOPPED — empty train/validation in {fold_id}.")
    valid_session_year = pd.to_datetime(
        valid["nyse_session_date"], errors="raise"
    ).dt.year
    if not valid_session_year.eq(validation_year).all():
        raise RuntimeError(f"CELL 13 STOPPED — wrong validation year in {fold_id}.")
    observed_sessions = sorted(
        valid["nyse_session_date"].astype(str).unique().tolist()
    )
    if observed_sessions != canonical_sessions:
        missing_sessions = sorted(set(canonical_sessions) - set(observed_sessions))
        raise RuntimeError(
            f"CELL 13 STOPPED — usable outcomes do not cover every canonical "
            f"validation session in {fold_id}: {missing_sessions[:10]}"
        )
    if train["label_end_time"].max() >= valid["decision_time"].min():
        boundary_failures.append(f"{fold_id}: train label overlaps validation.")

    validation_ids.extend(valid["decision_id"].tolist())

    y_train = train["economic_label_primary"].eq("LONG").to_numpy(dtype=np.int8)
    y_valid = valid["economic_label_primary"].eq("LONG").to_numpy(dtype=np.int8)
    train_prior_long = float(y_train.mean())

    fold_rng = np.random.default_rng(MASTER_SEED + fold_index)
    random_prediction = fold_rng.random(len(valid)) < train_prior_long

    baseline_specs = [
        ("ALWAYS_FLAT", np.zeros(len(valid), dtype=np.int8), np.zeros(len(valid))),
        ("ALWAYS_LONG", np.ones(len(valid), dtype=np.int8), np.ones(len(valid))),
        (
            "TRAIN_PRIOR_PROBABILITY",
            np.full(len(valid), int(train_prior_long >= 0.5), dtype=np.int8),
            np.full(len(valid), train_prior_long, dtype=float),
        ),
        (
            "STRATIFIED_RANDOM_ACTION",
            random_prediction.astype(np.int8),
            np.full(len(valid), train_prior_long, dtype=float),
        ),
    ]

    random_sensitivity_rng = np.random.default_rng(
        MASTER_SEED + 100 + fold_index
    )
    random_draws = (
        random_sensitivity_rng.random(
            (RANDOM_SENSITIVITY_SEEDS, len(valid))
        )
        < train_prior_long
    )
    random_accuracy_distribution = np.mean(
        random_draws == y_valid[np.newaxis, :], axis=1
    )
    random_economic_distribution = random_economic_sensitivity(
        valid, random_draws, round_trip_cost_usd
    )
    random_accuracy_summary = finite_distribution_summary(
        random_accuracy_distribution
    )
    random_net_summary = finite_distribution_summary(
        random_economic_distribution["net_pnl_usd"]
    )
    random_sharpe_summary = finite_distribution_summary(
        random_economic_distribution["annualized_session_sharpe"]
    )
    random_drawdown_summary = finite_distribution_summary(
        random_economic_distribution["maximum_drawdown_usd"]
    )

    for strategy, prediction, probability in baseline_specs:
        class_result = classification_metrics(y_valid, prediction, probability)
        simulation = simulate_non_overlapping_long(
            valid, prediction == 1, round_trip_cost_usd
        )
        rows = simulation["rows"]

        event = pd.DataFrame(
            {
                "fold_id": fold_id,
                "validation_year": validation_year,
                "strategy": strategy,
                "decision_id": rows["decision_id"].astype(str).to_numpy(),
                "decision_time": rows["decision_time"].to_numpy(),
                "nyse_session_date": rows["nyse_session_date"].astype(str).to_numpy(),
                "target_action": np.where(y_valid == 1, "ENTER_LONG", "FLAT"),
                "predicted_action": np.where(prediction == 1, "ENTER_LONG", "FLAT"),
                "probability_enter_long": probability,
                "correct": (prediction == y_valid),
                "executed": simulation["executed"],
                "ignored_signal": simulation["ignored"],
                "exit_time": simulation["exit_time"].to_numpy(),
                "gross_pnl_usd": simulation["gross_pnl_usd"],
                "charged_cost_usd": simulation["charged_cost_usd"],
                "net_pnl_usd": simulation["net_pnl_usd"],
                "cost_semantics": "CURRENT_DEPLOYMENT_COUNTERFACTUAL",
            }
        )
        event_frames.append(event)

        economics = economic_metrics(event)
        metrics = {
            "fold_id": fold_id,
            "validation_year": validation_year,
            "strategy": strategy,
            "action_space": ACTION_SPACE,
            "position_policy": POSITION_POLICY,
            "train_prior_enter_long": train_prior_long,
            "round_trip_cost_usd": round_trip_cost_usd,
            "cost_semantics": "CURRENT_DEPLOYMENT_COUNTERFACTUAL",
            "classification_scope": "OPPORTUNITY_ROWS_BEFORE_POSITION_FILTER",
            **class_result,
            **economics,
        }
        if strategy == "STRATIFIED_RANDOM_ACTION":
            metrics.update(
                {
                    "random_1000_seed_accuracy_mean": random_accuracy_summary["mean"],
                    "random_1000_seed_accuracy_p025": random_accuracy_summary["p025"],
                    "random_1000_seed_accuracy_p975": random_accuracy_summary["p975"],
                    "random_1000_seed_net_pnl_mean": random_net_summary["mean"],
                    "random_1000_seed_net_pnl_p025": random_net_summary["p025"],
                    "random_1000_seed_net_pnl_p975": random_net_summary["p975"],
                    "random_1000_seed_sharpe_mean": random_sharpe_summary["mean"],
                    "random_1000_seed_sharpe_p025": random_sharpe_summary["p025"],
                    "random_1000_seed_sharpe_p975": random_sharpe_summary["p975"],
                    "random_1000_seed_sharpe_valid_repetitions": (
                        random_sharpe_summary["valid_repetitions"]
                    ),
                    "random_1000_seed_sharpe_status": random_sharpe_summary["status"],
                    "random_1000_seed_max_drawdown_mean": random_drawdown_summary["mean"],
                    "random_1000_seed_max_drawdown_p025": random_drawdown_summary["p025"],
                    "random_1000_seed_max_drawdown_p975": random_drawdown_summary["p975"],
                }
            )
        metric_rows.append(metrics)

    dependence_rows.extend(empirical_overlap_diagnostics(valid, fold_id))

if boundary_failures:
    raise RuntimeError("CELL 13 STOPPED — " + " | ".join(boundary_failures))
if len(validation_ids) != len(set(validation_ids)):
    raise RuntimeError("CELL 13 STOPPED — validation decision IDs repeat across folds.")

events = pd.concat(event_frames, ignore_index=True)
metrics = pd.DataFrame(metric_rows)
dependence = pd.DataFrame(dependence_rows)

# Pooled chronological OOF metrics are recomputed from concatenated
# validation events; fold Sharpe values are never averaged.
pooled_metric_rows = []
for strategy, pooled_events in events.groupby("strategy", sort=True):
    pooled_events = pooled_events.sort_values("decision_time").copy()
    pooled_y = pooled_events["target_action"].eq("ENTER_LONG").to_numpy(dtype=np.int8)
    pooled_prediction = pooled_events["predicted_action"].eq(
        "ENTER_LONG"
    ).to_numpy(dtype=np.int8)
    pooled_probability = pooled_events["probability_enter_long"].to_numpy(dtype=float)
    pooled_metric_rows.append(
        {
            "fold_id": "OOF_POOLED",
            "validation_year": "2022-2024",
            "strategy": strategy,
            "action_space": ACTION_SPACE,
            "position_policy": POSITION_POLICY,
            "train_prior_enter_long": np.nan,
            "round_trip_cost_usd": round_trip_cost_usd,
            "cost_semantics": "CURRENT_DEPLOYMENT_COUNTERFACTUAL",
            "classification_scope": "OPPORTUNITY_ROWS_BEFORE_POSITION_FILTER",
            **classification_metrics(
                pooled_y, pooled_prediction, pooled_probability
            ),
            **economic_metrics(pooled_events),
        }
    )
metrics = pd.concat(
    [metrics, pd.DataFrame(pooled_metric_rows)], ignore_index=True
)


# ------------------------------------------------------------
# 7) Session moving-block bootstrap
# ------------------------------------------------------------
bootstrap_rows = []


def append_bootstrap_summary(
    fold_id,
    validation_year,
    strategy,
    metric_name,
    block_length,
    seed,
    values,
):
    summary = finite_distribution_summary(values)
    bootstrap_rows.append(
        {
            "fold_id": fold_id,
            "validation_year": validation_year,
            "strategy": strategy,
            "metric": metric_name,
            "block_length_sessions": block_length,
            "repetitions": BOOTSTRAP_REPETITIONS,
            "valid_repetitions": summary["valid_repetitions"],
            "interval_status": summary["status"],
            "seed": seed,
            "estimate_bootstrap_mean": summary["mean"],
            "ci_95_lower": summary["p025"],
            "ci_95_upper": summary["p975"],
            "primary_ci": block_length == PRIMARY_BOOTSTRAP_BLOCK_SESSIONS,
        }
    )


strategy_names = sorted(events["strategy"].unique().tolist())
for fold_index, (fold_id, _, validation_year) in enumerate(FOLD_SPECS):
    fold_events = events.loc[events["fold_id"].eq(fold_id)].copy()
    sessions = canonical_validation_sessions[fold_id]

    session_metrics = (
        fold_events.groupby(["strategy", "nyse_session_date"], sort=True)
        .agg(
            correct_count=("correct", "sum"),
            row_count=("correct", "size"),
            net_pnl_usd=("net_pnl_usd", "sum"),
        )
        .reset_index()
    )

    for strategy in strategy_names:
        observed_strategy_sessions = sorted(
            session_metrics.loc[
                session_metrics["strategy"].eq(strategy), "nyse_session_date"
            ].astype(str).tolist()
        )
        if observed_strategy_sessions != sessions:
            raise RuntimeError(
                f"CELL 13 STOPPED — incomplete session table for "
                f"{fold_id}/{strategy}."
            )

    for block_length in BOOTSTRAP_BLOCK_LENGTHS:
        bootstrap_seed = MASTER_SEED + 1000 * (fold_index + 1) + block_length
        bootstrap_rng = np.random.default_rng(bootstrap_seed)
        draws = moving_block_indices(
            len(sessions),
            block_length,
            BOOTSTRAP_REPETITIONS,
            bootstrap_rng,
        )

        for strategy in strategy_names:
            table = (
                session_metrics.loc[session_metrics["strategy"].eq(strategy)]
                .set_index("nyse_session_date")
                .loc[sessions]
            )
            correct = table["correct_count"].to_numpy(dtype=float)
            row_count = table["row_count"].to_numpy(dtype=float)
            session_net = table["net_pnl_usd"].to_numpy(dtype=float)

            sampled_correct = correct[draws].sum(axis=1)
            sampled_rows = row_count[draws].sum(axis=1)
            sampled_accuracy = sampled_correct / sampled_rows
            sampled_session_net = session_net[draws]
            sampled_mean_session_net = sampled_session_net.mean(axis=1)
            sampled_std_session_net = sampled_session_net.std(axis=1, ddof=1)
            sampled_sharpe = np.full(
                BOOTSTRAP_REPETITIONS, np.nan, dtype=float
            )
            valid_std = sampled_std_session_net > 0
            sampled_sharpe[valid_std] = (
                np.sqrt(252.0)
                * sampled_mean_session_net[valid_std]
                / sampled_std_session_net[valid_std]
            )

            for metric_name, values in [
                ("accuracy", sampled_accuracy),
                ("mean_session_net_usd", sampled_mean_session_net),
                ("annualized_session_sharpe", sampled_sharpe),
            ]:
                append_bootstrap_summary(
                    fold_id,
                    validation_year,
                    strategy,
                    metric_name,
                    block_length,
                    bootstrap_seed,
                    values,
                )


# Headline pooled OOF intervals are assembled from independent, fold-bounded
# moving blocks. No sampled block is allowed to cross a walk-forward boundary.
for block_length in BOOTSTRAP_BLOCK_LENGTHS:
    pooled_seed = MASTER_SEED + 90000 + block_length
    fold_draws = {}
    fold_session_tables = {}
    for fold_index, (fold_id, _, _) in enumerate(FOLD_SPECS):
        fold_sessions = canonical_validation_sessions[fold_id]
        fold_rng = np.random.default_rng(
            pooled_seed + 1000 * (fold_index + 1)
        )
        fold_draws[fold_id] = moving_block_indices(
            len(fold_sessions),
            block_length,
            BOOTSTRAP_REPETITIONS,
            fold_rng,
        )
        fold_session_tables[fold_id] = (
            events.loc[events["fold_id"].eq(fold_id)]
            .groupby(["strategy", "nyse_session_date"], sort=True)
            .agg(
                correct_count=("correct", "sum"),
                row_count=("correct", "size"),
                net_pnl_usd=("net_pnl_usd", "sum"),
            )
            .reset_index()
        )

    for strategy in strategy_names:
        sampled_correct_parts = []
        sampled_row_parts = []
        sampled_session_net_parts = []

        for fold_id, _, _ in FOLD_SPECS:
            fold_sessions = canonical_validation_sessions[fold_id]
            table = (
                fold_session_tables[fold_id]
                .loc[lambda x: x["strategy"].eq(strategy)]
                .set_index("nyse_session_date")
                .loc[fold_sessions]
            )
            draws = fold_draws[fold_id]
            sampled_correct_parts.append(
                table["correct_count"].to_numpy(dtype=float)[draws].sum(axis=1)
            )
            sampled_row_parts.append(
                table["row_count"].to_numpy(dtype=float)[draws].sum(axis=1)
            )
            sampled_session_net_parts.append(
                table["net_pnl_usd"].to_numpy(dtype=float)[draws]
            )

        sampled_correct = np.sum(sampled_correct_parts, axis=0)
        sampled_rows = np.sum(sampled_row_parts, axis=0)
        sampled_accuracy = sampled_correct / sampled_rows
        sampled_session_net = np.concatenate(sampled_session_net_parts, axis=1)
        sampled_mean_session_net = sampled_session_net.mean(axis=1)
        sampled_std_session_net = sampled_session_net.std(axis=1, ddof=1)
        sampled_sharpe = np.full(BOOTSTRAP_REPETITIONS, np.nan, dtype=float)
        valid_std = sampled_std_session_net > 0
        sampled_sharpe[valid_std] = (
            np.sqrt(252.0)
            * sampled_mean_session_net[valid_std]
            / sampled_std_session_net[valid_std]
        )

        for metric_name, values in [
            ("accuracy", sampled_accuracy),
            ("mean_session_net_usd", sampled_mean_session_net),
            ("annualized_session_sharpe", sampled_sharpe),
        ]:
            append_bootstrap_summary(
                "OOF_POOLED",
                "2022-2024",
                strategy,
                metric_name,
                block_length,
                pooled_seed,
                values,
            )

bootstrap = pd.DataFrame(bootstrap_rows)


# ------------------------------------------------------------
# 8) Final integrity gates
# ------------------------------------------------------------
failures = []

if ACTION_SPACE != "LONG_FLAT":
    failures.append("V1 action space is not LONG_FLAT.")
if POSITION_POLICY != "NON_OVERLAPPING_60M":
    failures.append("Position policy is not NON_OVERLAPPING_60M.")
event_session_year = pd.to_datetime(
    events["nyse_session_date"], errors="raise"
).dt.year
if event_session_year.ge(2025).any():
    failures.append("A 2025+ timestamp entered baseline events.")
if events["decision_id"].isna().any():
    failures.append("Missing validation decision ID.")
if events["decision_id"].astype(str).str.strip().eq("").any():
    failures.append("Blank validation decision ID.")
if not events["cost_semantics"].eq(
    "CURRENT_DEPLOYMENT_COUNTERFACTUAL"
).all():
    failures.append("Baseline event has ambiguous cost semantics.")
if not events.loc[events["executed"], "exit_time"].eq(
    events.loc[events["executed"], "decision_time"]
    + pd.Timedelta(minutes=HOLD_MINUTES)
).all():
    failures.append("An executed exit is not exactly +60 minutes.")
if not np.allclose(
    events.loc[events["executed"], "net_pnl_usd"],
    events.loc[events["executed"], "gross_pnl_usd"]
    - events.loc[events["executed"], "charged_cost_usd"],
    atol=1e-12,
):
    failures.append("Executed action P&L does not reconcile.")
if not events.loc[events["executed"], "charged_cost_usd"].eq(
    round_trip_cost_usd
).all():
    failures.append("Round-trip cost was not charged once per executed trade.")
if events.loc[~events["executed"], "charged_cost_usd"].ne(0).any():
    failures.append("A non-executed decision was charged a cost.")
if metrics["maximum_concurrent_positions"].gt(1).any():
    failures.append("Non-overlap mode exceeded one concurrent position.")
if not metrics["classification_scope"].eq(
    "OPPORTUNITY_ROWS_BEFORE_POSITION_FILTER"
).all():
    failures.append("Classification metric scope is ambiguous.")

for (fold_id, strategy), group in events.groupby(
    ["fold_id", "strategy"], sort=True
):
    group = group.sort_values(["decision_time", "decision_id"], kind="stable")
    last_exit = None
    for row in group.itertuples(index=False):
        predicted_long = row.predicted_action == "ENTER_LONG"
        signal_accounted_for = bool(row.executed) ^ bool(row.ignored_signal)
        if predicted_long != signal_accounted_for:
            failures.append(
                f"{fold_id}/{strategy}: LONG signal is not exactly executed or ignored."
            )
            break
        if row.executed and not predicted_long:
            failures.append(f"{fold_id}/{strategy}: executed without LONG signal.")
            break
        if row.ignored_signal and (not predicted_long or row.executed):
            failures.append(f"{fold_id}/{strategy}: invalid ignored signal.")
            break
        if row.executed:
            if last_exit is not None and row.decision_time < last_exit:
                failures.append(f"{fold_id}/{strategy}: overlapping execution.")
                break
            last_exit = row.exit_time
        elif row.ignored_signal:
            if last_exit is None or row.decision_time >= last_exit:
                failures.append(
                    f"{fold_id}/{strategy}: ignored signal without open position."
                )
                break
expected_bootstrap_rows = (
    (len(FOLD_SPECS) + 1)
    * len(strategy_names)
    * len(BOOTSTRAP_BLOCK_LENGTHS)
    * 3
)
if len(bootstrap) != expected_bootstrap_rows:
    failures.append(
        f"Expected {expected_bootstrap_rows} bootstrap rows; observed {len(bootstrap)}."
    )
expected_primary_rows = (len(FOLD_SPECS) + 1) * len(strategy_names) * 3
if int(bootstrap["primary_ci"].sum()) != expected_primary_rows:
    failures.append("Primary five-session bootstrap intervals are incomplete.")

non_sharpe_bootstrap = bootstrap.loc[
    ~bootstrap["metric"].eq("annualized_session_sharpe")
]
if not non_sharpe_bootstrap["valid_repetitions"].eq(
    BOOTSTRAP_REPETITIONS
).all():
    failures.append("A non-Sharpe bootstrap interval lost finite replications.")
if not np.isfinite(
    non_sharpe_bootstrap[
        ["estimate_bootstrap_mean", "ci_95_lower", "ci_95_upper"]
    ].to_numpy(dtype=float)
).all():
    failures.append("A non-Sharpe bootstrap interval is non-finite.")

sharpe_bootstrap = bootstrap.loc[
    bootstrap["metric"].eq("annualized_session_sharpe")
]
undefined_sharpe = sharpe_bootstrap["valid_repetitions"].eq(0)
if not sharpe_bootstrap.loc[undefined_sharpe, "interval_status"].eq(
    "UNDEFINED_ZERO_VARIANCE"
).all():
    failures.append("Undefined Sharpe interval lacks an explicit status.")
defined_sharpe = ~undefined_sharpe
if not np.isfinite(
    sharpe_bootstrap.loc[
        defined_sharpe,
        ["estimate_bootstrap_mean", "ci_95_lower", "ci_95_upper"],
    ].to_numpy(dtype=float)
).all():
    failures.append("A defined Sharpe bootstrap interval is non-finite.")


# ------------------------------------------------------------
# 9) Save deterministic artifacts and audit
# ------------------------------------------------------------
numeric_metric_columns = metrics.select_dtypes(include=[np.number]).columns
metrics[numeric_metric_columns] = metrics[numeric_metric_columns].round(10)
dependence["value"] = dependence["value"].round(10)
numeric_bootstrap_columns = bootstrap.select_dtypes(include=[np.number]).columns
bootstrap[numeric_bootstrap_columns] = bootstrap[numeric_bootstrap_columns].round(10)

events.to_parquet(CELL13_EVENTS_PATH, index=False)
dependence.to_csv(CELL13_DEPENDENCE_PATH, index=False)
metrics.to_csv(CELL13_METRICS_PATH, index=False)
bootstrap.to_csv(CELL13_BOOTSTRAP_PATH, index=False)

artifact_hashes = {
    "input_cell8_audit_sha256": sha256_file(CELL8_AUDIT_PATH),
    "input_cell10_audit_sha256": sha256_file(CELL10_AUDIT_PATH),
    "input_cell11_semantic_costs_sha256": actual_semantic_hash,
    "input_cell11_audit_sha256": sha256_file(CELL11_AUDIT_PATH),
    "input_cell12_path_outcomes_sha256": actual_path_hash,
    "input_cell12_audit_sha256": sha256_file(CELL12_AUDIT_PATH),
    "baseline_events_sha256": sha256_file(CELL13_EVENTS_PATH),
    "dependence_audit_sha256": sha256_file(CELL13_DEPENDENCE_PATH),
    "baseline_metrics_sha256": sha256_file(CELL13_METRICS_PATH),
    "block_bootstrap_ci_sha256": sha256_file(CELL13_BOOTSTRAP_PATH),
}

audit = {
    "audit_written_utc": datetime.now(timezone.utc).isoformat(),
    "policy_version": BASELINE_POLICY_VERSION,
    "status": "PASS" if not failures else "FAIL",
    "upstream_binding": {
        "cell8_policy_version": cell8_audit.get("policy_version"),
        "cell10_policy_version": cell10_audit.get("policy_version"),
        "cell11_policy_version": cell11_audit.get("policy_version"),
        "cell12_policy_version": cell12_audit.get("policy_version"),
        "cell8_assignments_sha256": cell8_assignment_hash,
        "cell10_labels_sha256": cell10_labels_hash,
        "cell10_audit_sha256": actual_cell10_audit_hash,
        "cell11_semantic_costs_sha256": actual_semantic_hash,
        "cell12_path_outcomes_sha256": actual_path_hash,
    },
    "evaluation_contract": {
        "action_space": ACTION_SPACE,
        "short_action_allowed": False,
        "short_outcome_retained_as_diagnostic": True,
        "position_policy": POSITION_POLICY,
        "hold_minutes": HOLD_MINUTES,
        "exit_then_entry_at_same_timestamp": EXIT_THEN_ENTRY_AT_SAME_TIMESTAMP,
        "maximum_concurrent_positions": 1,
        "oof_validation_years": [2022, 2023, 2024],
        "final_test_used": False,
        "cost_scenario": primary_cost_scenario,
        "round_trip_cost_usd": round_trip_cost_usd,
        "cost_semantics": "CURRENT_DEPLOYMENT_COUNTERFACTUAL",
        "historical_actual_pnl_claim": False,
        "classification_scope": "OPPORTUNITY_ROWS_BEFORE_POSITION_FILTER",
        "session_universe": "ALL_CANONICAL_VALIDATION_SESSIONS",
        "missing_canonical_sessions_allowed": False,
    },
    "dependence_contract": {
        "decision_frequency_minutes": 15,
        "label_horizon_minutes": 60,
        "empirical_acf_lags": [1, 2, 3],
        "ess_status": "DESCRIPTIVE_ONLY",
        "n_div_4_status": "THEORETICAL_REFERENCE_ONLY",
        "confidence_interval_method": "CONSECUTIVE_SESSION_MOVING_BLOCK_BOOTSTRAP",
        "primary_block_length_sessions": PRIMARY_BOOTSTRAP_BLOCK_SESSIONS,
        "sensitivity_block_lengths_sessions": BOOTSTRAP_BLOCK_LENGTHS,
        "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "master_seed": MASTER_SEED,
        "pooled_oof_blocks_cross_fold_boundaries": False,
    },
    "counts": {
        "unique_oof_validation_decisions": int(len(set(validation_ids))),
        "baseline_event_rows": int(len(events)),
        "strategies": int(events["strategy"].nunique()),
        "canonical_validation_sessions_by_fold": {
            fold_id: len(sessions)
            for fold_id, sessions in canonical_validation_sessions.items()
        },
        "final_test_rows_used": 0,
    },
    "artifacts": {
        "baseline_events": str(CELL13_EVENTS_PATH),
        "dependence_ess_audit": str(CELL13_DEPENDENCE_PATH),
        "naive_baseline_metrics": str(CELL13_METRICS_PATH),
        "block_bootstrap_ci": str(CELL13_BOOTSTRAP_PATH),
    },
    "sha256": artifact_hashes,
    "modeling_gate": {
        "naive_baselines_established": not failures,
        "model_fitted": False,
        "required_future_comparison": (
            "paired five-session block-bootstrap differences versus "
            "ALWAYS_FLAT, ALWAYS_LONG, and TRAIN_PRIOR_PROBABILITY"
        ),
    },
    "failures": failures,
}

with open(CELL13_AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2, ensure_ascii=False)

if failures:
    print("\nCELL 13 FAILURES")
    for failure in failures:
        print(" -", failure)
    raise RuntimeError(
        "\nCELL 13 DEPENDENCE AUDIT & NAIVE BASELINES: FAIL\n"
        f"{CELL13_AUDIT_PATH}"
    )

display_columns = [
    "fold_id",
    "strategy",
    "accuracy",
    "balanced_accuracy",
    "predicted_long_coverage",
    "executed_trades",
    "ignored_long_signals",
    "net_pnl_usd",
    "annualized_session_sharpe",
    "maximum_drawdown_usd",
]

print("\n" + "=" * 72)
print("CELL 13 — DEVELOPMENT-ONLY DEPENDENCE AUDIT & NAIVE BASELINES")
print("=" * 72)
print("Action space              : LONG / FLAT")
print("Position policy           : NON-OVERLAPPING 60m")
print("Cost semantics            : CURRENT DEPLOYMENT COUNTERFACTUAL")
print("OOF validation years      : 2022, 2023, 2024")
print("Final-test rows used      : 0")
print("Bootstrap                 : 2,000 reps; 5-session primary")
print("\n[OOF BASELINES]")
print(metrics[display_columns].to_string(index=False))
print("\nBaseline events           :", CELL13_EVENTS_PATH)
print("Dependence audit          :", CELL13_DEPENDENCE_PATH)
print("Baseline metrics          :", CELL13_METRICS_PATH)
print("Block-bootstrap intervals :", CELL13_BOOTSTRAP_PATH)
print("Cell 13 audit             :", CELL13_AUDIT_PATH)
print("\nCELL 13 DEPENDENCE AUDIT & NAIVE BASELINES: PASS")
print("=" * 72)



CELL 13 — DEVELOPMENT-ONLY DEPENDENCE AUDIT & NAIVE BASELINES
Action space              : LONG / FLAT
Position policy           : NON-OVERLAPPING 60m
Cost semantics            : CURRENT DEPLOYMENT COUNTERFACTUAL
OOF validation years      : 2022, 2023, 2024
Final-test rows used      : 0
Bootstrap                 : 2,000 reps; 5-session primary

[OOF BASELINES]
   fold_id                 strategy  accuracy  balanced_accuracy  predicted_long_coverage  executed_trades  ignored_long_signals  net_pnl_usd  annualized_session_sharpe  maximum_drawdown_usd
   WF_2022              ALWAYS_FLAT  0.511434           0.500000                 0.000000                0                     0         0.00                        NaN                  0.00
   WF_2022              ALWAYS_LONG  0.488566           0.500000                 1.000000             1503                  4007     -8304.91                  -2.236690              -8387.77
   WF_2022  TRAIN_PRIOR_PROBABILITY  0.511434           0.500000